In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from pathlib import Path
import hashlib
import json
import torch

# Kaggle input and writable output locations
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/JFAMP")
OUT = WORK_ROOT / "retrain_to150"

OUT.mkdir(parents=True, exist_ok=True)

# Confirm GPU
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory: {:.2f} GiB".format(
            torch.cuda.get_device_properties(0).total_memory / 1024**3
        )
    )
else:
    raise RuntimeError(
        "No GPU detected. Enable a GPU in Kaggle Settings before proceeding."
    )

# Find the required files inside attached datasets
def find_file(filename):
    matches = list(INPUT_ROOT.rglob(filename))
    if not matches:
        raise FileNotFoundError(
            f"{filename} not found under {INPUT_ROOT}. "
            "Check that the private dataset is attached."
        )
    if len(matches) > 1:
        print(f"Multiple copies of {filename} found:")
        for m in matches:
            print(" ", m)
        raise RuntimeError(
            f"Ambiguous file: {filename}. Keep one matching dataset attached."
        )
    return matches[0]

LAST_CKPT = find_file("resnet18_cifar100_last.pth")
BEST_CKPT = find_file("resnet18_cifar100_best.pth")
SPLIT_PATH = find_file("data_split_indices.json")

print("\n========== FILES FOUND ==========")
for p in [LAST_CKPT, BEST_CKPT, SPLIT_PATH]:
    print(p)
    print(f"  Size: {p.stat().st_size / (1024**2):.2f} MiB")

# Verify the original split hash
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

split_hash = sha256_file(SPLIT_PATH)
expected_hash = (
    "b632b03ccc8166092cbf5c9e02a41c754248533ef1f60292c0404e3a8d80d0e7"
)

print("\nOriginal split SHA256:", split_hash)
assert split_hash == expected_hash, "Original validation split hash mismatch!"

# Inspect the checkpoint without changing it
checkpoint = torch.load(
    LAST_CKPT,
    map_location="cpu",
    weights_only=True
)

print("\n========== CHECKPOINT METADATA ==========")
for key in [
    "epoch", "best_epoch", "train_accuracy",
    "val_acc", "best_val_acc"
]:
    print(f"{key}: {checkpoint.get(key)}")

assert checkpoint["epoch"] == 108, (
    f"Expected epoch 108, found epoch {checkpoint['epoch']}"
)

for key in [
    "model_state_dict",
    "optimizer_state_dict",
    "scheduler_state_dict",
    "scaler_state_dict",
]:
    assert key in checkpoint, f"Checkpoint missing {key}"

print("\nSUCCESS: GPU, checkpoint and original split verified.")
print("New output directory:", OUT)

In [ ]:

# ================================================================
# J-FAMP: ResNet-18 CIFAR-100
# Resume training from epoch 108 to epoch 150 on Kaggle T4
# ================================================================

import os
import csv
import json
import time
import random
import shutil
import hashlib
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

# ------------------------------------------------
# 1. Paths
# ------------------------------------------------

OUT = Path("/kaggle/working/JFAMP/retrain_to150")
CKPT_OUT = OUT / "checkpoints"
DATA_DIR = Path("/kaggle/working/cifar100_data")

OUT.mkdir(parents=True, exist_ok=True)
CKPT_OUT.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

ORIGINAL_LAST = Path(
    "/kaggle/input/datasets/crahulnaidu/"
    "jfamp-resnet18-cifar100-checkpoint/"
    "resnet18_cifar100_last.pth"
)

ORIGINAL_BEST = ORIGINAL_LAST.with_name(
    "resnet18_cifar100_best.pth"
)

SPLIT_PATH = ORIGINAL_LAST.with_name(
    "data_split_indices.json"
)

LATEST_CKPT = CKPT_OUT / "resnet18_cifar100_last_150.pth"
BEST_CKPT = CKPT_OUT / "resnet18_cifar100_best_150.pth"

HISTORY_CSV = OUT / "training_history.csv"
SUMMARY_JSON = OUT / "training_summary.json"
CONFIG_JSON = OUT / "training_config.json"

assert torch.cuda.is_available(), "CUDA GPU is required."

DEVICE = torch.device("cuda")
TARGET_EPOCH = 150
BATCH_SIZE = 128
NUM_WORKERS = 2

# ------------------------------------------------
# 2. Reproducibility settings
# ------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

# ------------------------------------------------
# 3. Configuration
# ------------------------------------------------

CONFIG = {
    "seed": SEED,
    "target_epochs": TARGET_EPOCH,
    "batch_size": BATCH_SIZE,
    "learning_rate": 0.1,
    "momentum": 0.9,
    "weight_decay": 0.0005,
    "num_classes": 100,
    "architecture": "CIFAR-style ResNet-18",
    "normalization_mean": [0.5071, 0.4867, 0.4408],
    "normalization_std": [0.2675, 0.2565, 0.2761],
    "amp": True,
    "device": torch.cuda.get_device_name(0),
}

with open(CONFIG_JSON, "w") as f:
    json.dump(CONFIG, f, indent=4)

# ------------------------------------------------
# 4. Dataset and original split
# ------------------------------------------------

mean = (0.5071, 0.4867, 0.4408)
std = (0.2675, 0.2565, 0.2761)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

# Download if not already present
train_aug_dataset = datasets.CIFAR100(
    root=str(DATA_DIR),
    train=True,
    download=True,
    transform=train_transform,
)

train_eval_dataset = datasets.CIFAR100(
    root=str(DATA_DIR),
    train=True,
    download=False,
    transform=val_transform,
)

# Load the original split -- do not regenerate it
with open(SPLIT_PATH, "r") as f:
    split_data = json.load(f)

print("Split JSON keys:", list(split_data.keys()))

# Support common key names used for saved index files
def get_indices(data, candidates):
    for key in candidates:
        if key in data:
            return data[key]
    raise KeyError(
        f"Could not find any of these keys: {candidates}. "
        f"Available keys: {list(data.keys())}"
    )

train_indices = get_indices(
    split_data,
    ["train_indices", "train_idx", "train"]
)

val_indices = get_indices(
    split_data,
    ["val_indices", "validation_indices", "val_idx", "val"]
)

assert len(train_indices) == 45000
assert len(val_indices) == 5000

train_dataset = Subset(train_aug_dataset, train_indices)
val_dataset = Subset(train_eval_dataset, val_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

# ------------------------------------------------
# 5. Model
# ------------------------------------------------

model = models.resnet18(weights=None)

model.conv1 = nn.Conv2d(
    3, 64,
    kernel_size=3,
    stride=1,
    padding=1,
    bias=False,
)

model.maxpool = nn.Identity()
model.fc = nn.Linear(model.fc.in_features, 100)

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    model.parameters(),
    lr=0.1,
    momentum=0.9,
    weight_decay=0.0005,
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=150,
    eta_min=0.0,
)

scaler = torch.amp.GradScaler("cuda")

# ------------------------------------------------
# 6. Load latest checkpoint if available
# ------------------------------------------------

if LATEST_CKPT.exists():
    print("\nResuming from Kaggle's latest saved checkpoint.")
    resume_path = LATEST_CKPT
else:
    print("\nStarting from original epoch-108 checkpoint.")
    resume_path = ORIGINAL_LAST

checkpoint = torch.load(
    resume_path,
    map_location="cpu",
    weights_only=True,
)

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
scaler.load_state_dict(checkpoint["scaler_state_dict"])

START_EPOCH = int(checkpoint["epoch"]) + 1

best_val_acc = float(checkpoint["best_val_acc"])
best_epoch = int(checkpoint["best_epoch"])

assert START_EPOCH <= TARGET_EPOCH + 1

print("Loaded checkpoint:", resume_path)
print("Checkpoint epoch:", checkpoint["epoch"])
print("Next epoch:", START_EPOCH)
print("Best validation accuracy:", best_val_acc)
print("Best epoch:", best_epoch)
print("Current learning rate:", optimizer.param_groups[0]["lr"])
print("Scheduler last epoch:", scheduler.last_epoch)

if START_EPOCH == 109:
    assert scheduler.last_epoch == 108

# Preserve the epoch-108 best checkpoint if no later epoch improves.
if not BEST_CKPT.exists():
    shutil.copy2(ORIGINAL_BEST, BEST_CKPT)

# ------------------------------------------------
# 7. Evaluation
# ------------------------------------------------

@torch.no_grad()
def evaluate():
    model.eval()

    correct = 0
    total = 0
    total_loss = 0.0

    for images, labels in val_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            outputs = model(images)
            loss = criterion(outputs, labels)

        total_loss += loss.item() * labels.size(0)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, 100.0 * correct / total

# ------------------------------------------------
# 8. Training function
# ------------------------------------------------

def train_one_epoch():
    model.train()

    correct = 0
    total = 0
    total_loss = 0.0

    for images, labels in train_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * labels.size(0)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, 100.0 * correct / total

# ------------------------------------------------
# 9. Atomic checkpoint saving
# ------------------------------------------------

def save_checkpoint(path, epoch, train_loss, train_acc,
                    val_loss, val_acc):

    state = {
        "epoch": epoch,
        "best_epoch": best_epoch,
        "train_accuracy": train_acc,
        "train_loss": train_loss,
        "val_acc": val_acc,
        "val_loss": val_loss,
        "best_val_acc": best_val_acc,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "config": CONFIG,
    }

    temp_path = path.with_suffix(path.suffix + ".tmp")
    torch.save(state, temp_path)
    os.replace(temp_path, path)

# ------------------------------------------------
# 10. Training history
# ------------------------------------------------

if not HISTORY_CSV.exists():
    with open(HISTORY_CSV, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "epoch",
            "learning_rate",
            "train_loss",
            "train_accuracy",
            "val_loss",
            "val_accuracy",
            "best_val_accuracy",
            "best_epoch",
            "epoch_time_seconds",
        ])

# ------------------------------------------------
# 11. Resume training
# ------------------------------------------------

print("\n========== TRAINING ==========")

for epoch in range(START_EPOCH, TARGET_EPOCH + 1):

    epoch_start = time.time()
    current_lr = optimizer.param_groups[0]["lr"]

    train_loss, train_acc = train_one_epoch()
    val_loss, val_acc = evaluate()

    improved = val_acc > best_val_acc

    if improved:
        best_val_acc = val_acc
        best_epoch = epoch

    # Step scheduler after validation, matching original procedure
    scheduler.step()

    epoch_time = time.time() - epoch_start

    # Save latest checkpoint after every completed epoch
    save_checkpoint(
        LATEST_CKPT,
        epoch,
        train_loss,
        train_acc,
        val_loss,
        val_acc,
    )

    # Update best checkpoint if validation improves
    if improved:
        save_checkpoint(
            BEST_CKPT,
            epoch,
            train_loss,
            train_acc,
            val_loss,
            val_acc,
        )

    # Append metrics
    with open(HISTORY_CSV, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            epoch,
            current_lr,
            train_loss,
            train_acc,
            val_loss,
            val_acc,
            best_val_acc,
            best_epoch,
            epoch_time,
        ])

    summary = {
        "completed_epoch": epoch,
        "best_epoch": best_epoch,
        "best_validation_accuracy": best_val_acc,
        "latest_validation_accuracy": val_acc,
        "target_epoch": TARGET_EPOCH,
        "latest_checkpoint": str(LATEST_CKPT),
        "best_checkpoint": str(BEST_CKPT),
    }

    with open(SUMMARY_JSON, "w") as f:
        json.dump(summary, f, indent=4)

    print(
        f"Epoch {epoch:03d}/{TARGET_EPOCH} | "
        f"LR {current_lr:.6f} | "
        f"Train Loss {train_loss:.4f} | "
        f"Train Acc {train_acc:.2f}% | "
        f"Val Loss {val_loss:.4f} | "
        f"Val Acc {val_acc:.2f}% | "
        f"Best {best_val_acc:.2f}% (Epoch {best_epoch}) | "
        f"Time {epoch_time:.1f}s"
    )

    # Release temporary GPU memory
    torch.cuda.empty_cache()

print("\n========== TRAINING FINISHED ==========")
print("Completed epoch:", TARGET_EPOCH)
print("Best validation accuracy:", best_val_acc)
print("Best epoch:", best_epoch)
print("Latest checkpoint:", LATEST_CKPT)
print("Best checkpoint:", BEST_CKPT)
print("History:", HISTORY_CSV)


In [ ]:
from pathlib import Path
import zipfile
import hashlib
import torch
import json

OUT = Path("/kaggle/working/JFAMP/retrain_to150")
CKPT_DIR = OUT / "checkpoints"

BEST_CKPT = CKPT_DIR / "resnet18_cifar100_best_150.pth"
LAST_CKPT = CKPT_DIR / "resnet18_cifar100_last_150.pth"

FILES = [
    BEST_CKPT,
    LAST_CKPT,
    OUT / "training_history.csv",
    OUT / "training_summary.json",
    OUT / "training_config.json",
]

# Verify all expected files
for f in FILES:
    assert f.exists(), f"Missing file: {f}"
    print(f"{f.name}: {f.stat().st_size / (1024**2):.2f} MiB")

# Inspect best checkpoint
best = torch.load(
    BEST_CKPT,
    map_location="cpu",
    weights_only=True
)

print("\n========== BEST CHECKPOINT ==========")
print("Epoch:", best["epoch"])
print("Best epoch:", best["best_epoch"])
print("Validation accuracy:", best["val_acc"])
print("Best validation accuracy:", best["best_val_acc"])

assert best["epoch"] == 139
assert abs(best["best_val_acc"] - 77.82) < 0.001

# Create SHA256 manifest
manifest = {}

for f in FILES:
    h = hashlib.sha256()
    with open(f, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    manifest[f.name] = h.hexdigest()

manifest_path = OUT / "sha256_manifest.json"

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=4)

# Create archive
archive_path = Path("/kaggle/working/JFAMP_retrain_to150_backup.zip")

with zipfile.ZipFile(
    archive_path, "w", compression=zipfile.ZIP_DEFLATED
) as z:
    for f in FILES + [manifest_path]:
        z.write(f, arcname=f.name)

print("\nBackup created:", archive_path)
print("Archive size: {:.2f} MiB".format(
    archive_path.stat().st_size / (1024**2)
))

In [ ]:

# ============================================================
# J-FAMP: FP32 Reference Baseline Evaluation
# Model: ResNet-18 CIFAR-100
# Checkpoint: Best model from epoch 139
# ============================================================

import json
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/JFAMP/retrain_to150")

BEST_CKPT = (
    WORK_ROOT / "checkpoints" /
    "resnet18_cifar100_best_150.pth"
)

DATA_DIR = Path("/kaggle/working/cifar100_data")

# Find original validation split
split_matches = list(
    INPUT_ROOT.rglob("data_split_indices.json")
)

assert len(split_matches) == 1, (
    f"Expected one split file, found {len(split_matches)}"
)

SPLIT_PATH = split_matches[0]

assert BEST_CKPT.exists(), (
    "Best checkpoint not found. Restore it from your saved backup."
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------
# 2. Load checkpoint
# ------------------------------------------------------------

checkpoint = torch.load(
    BEST_CKPT,
    map_location="cpu",
    weights_only=True
)

print("Checkpoint epoch:", checkpoint["epoch"])
print("Best epoch:", checkpoint["best_epoch"])
print("Stored validation accuracy:", checkpoint["val_acc"])

assert checkpoint["epoch"] == 139
assert checkpoint["best_epoch"] == 139

# ------------------------------------------------------------
# 3. Reconstruct CIFAR-style ResNet-18
# ------------------------------------------------------------

model = models.resnet18(weights=None)

model.conv1 = nn.Conv2d(
    3, 64,
    kernel_size=3,
    stride=1,
    padding=1,
    bias=False
)

model.maxpool = nn.Identity()
model.fc = nn.Linear(model.fc.in_features, 100)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(DEVICE)
model.eval()

# ------------------------------------------------------------
# 4. Load original validation split
# ------------------------------------------------------------

with open(SPLIT_PATH, "r") as f:
    split_data = json.load(f)

def get_indices(data, candidates):
    for key in candidates:
        if key in data:
            return data[key]
    raise KeyError(
        f"Could not find keys {candidates}. "
        f"Available keys: {list(data.keys())}"
    )

val_indices = get_indices(
    split_data,
    ["val_indices", "validation_indices", "val_idx", "val"]
)

assert len(val_indices) == 5000

# ------------------------------------------------------------
# 5. Validation dataset
# ------------------------------------------------------------

mean = (0.5071, 0.4867, 0.4408)
std = (0.2675, 0.2565, 0.2761)

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

val_dataset_full = datasets.CIFAR100(
    root=str(DATA_DIR),
    train=True,
    download=True,
    transform=val_transform
)

val_dataset = Subset(
    val_dataset_full,
    val_indices
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ------------------------------------------------------------
# 6. FP32 evaluation
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss()

total = 0
correct = 0
total_loss = 0.0

with torch.inference_mode():

    for images, labels in val_loader:

        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        # Explicit FP32 inference
        images = images.float()

        with torch.autocast(
            device_type="cuda",
            enabled=False
        ):
            outputs = model(images)
            loss = criterion(outputs, labels)

        predictions = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (predictions == labels).sum().item()
        total_loss += loss.item() * labels.size(0)

fp32_accuracy = 100.0 * correct / total
fp32_loss = total_loss / total

print("\n========== FP32 BASELINE ==========")
print("Checkpoint epoch:", checkpoint["epoch"])
print("Validation samples:", total)
print(f"FP32 validation accuracy: {fp32_accuracy:.4f}%")
print(f"FP32 validation loss: {fp32_loss:.6f}")
print(f"Stored checkpoint accuracy: {checkpoint['val_acc']:.4f}%")
print(
    f"Difference: "
    f"{fp32_accuracy - checkpoint['val_acc']:+.4f} percentage points"
)

# Save baseline result
baseline_result = {
    "model": "CIFAR-style ResNet-18",
    "dataset": "CIFAR-100",
    "checkpoint_epoch": checkpoint["epoch"],
    "validation_samples": total,
    "precision": "FP32",
    "validation_accuracy": fp32_accuracy,
    "validation_loss": fp32_loss,
    "stored_checkpoint_accuracy": checkpoint["val_acc"]
}

result_path = WORK_ROOT / "fp32_baseline.json"

with open(result_path, "w") as f:
    json.dump(baseline_result, f, indent=4)

print("\nSaved:", result_path)


In [ ]:
from pathlib import Path

PROJECT = Path("/kaggle/working/JFAMP")

DIRS = [
    PROJECT / "codec",
    PROJECT / "checkpoints",
    PROJECT / "baseline",
    PROJECT / "quantization/individual",
    PROJECT / "quantization/mixed_precision",
    PROJECT / "quantization/results",
    PROJECT / "reports",
]

for directory in DIRS:
    directory.mkdir(parents=True, exist_ok=True)

print("J-FAMP experiment directories prepared.")

for directory in DIRS:
    print(directory)

In [ ]:
%%writefile /kaggle/working/JFAMP/codec/e4m11_codec.py


"""E4M11 (1 sign, 4 exponent, 11 mantissa bits, bias 7) encode/decode.

Two encoder modes:

paper (default)
    Eq. 4 of the paper including the mantissa carry
    ("if m >= 2^11: set e <- e+1, m <- 0"), combined with the clipping/flush
    rules of Algorithm 1 (magnitude clipped to vmax, |w| < 2^-7 flushed to
    signed zero, exponent clamped to [-6, 7], fraction lower-clipped at 0 so
    that [2^-7, 2^-6) rounds up to the minimum normal 2^-6).

legacy
    Bit-exact reproduction of the original notebook: the fraction is clipped to
    [0, 1 - 2^-11] and the mantissa to [0, 2^11 - 1] (this equals Algorithm 1
    steps 7-8 literally) and there is no carry.

subnormal (NOT in the paper; exploratory)
    Same 16-bit layout and bias 7, but es == 0 stores a subnormal
    (-1)^s * m * 2^-17 instead of flushing to zero.
    Normals are encoded exactly as in paper mode.

The decoder supports three policies for exponent es == 15:
    clamp   - treat as es = 14
    ieee    - map to inf/NaN
    literal - use exponent 15 directly.
"""

from __future__ import annotations

import numpy as np

E_BITS = 4
M_BITS = 11
BIAS = 7

MANT_MASK = (1 << M_BITS) - 1
EXP_MAX_STORED = (1 << E_BITS) - 2
MAX_FINITE = (2.0 - 2.0 ** -M_BITS) * 2.0 ** BIAS
MIN_NORMAL = 2.0 ** (1 - BIAS)
FLUSH_THRESHOLD = 2.0 ** -BIAS
SUBNORMAL_STEP = 2.0 ** (1 - BIAS - M_BITS)

MODES = ("paper", "legacy", "subnormal")
ALL_ONES = ("clamp", "ieee", "literal")


def _as_f64(values) -> np.ndarray:
    if hasattr(values, "detach"):
        values = values.detach().cpu().numpy()

    x = np.asarray(values, dtype=np.float64)

    if not np.isfinite(x).all():
        raise ValueError("E4M11 encoder supports finite values only.")

    return x


def max_finite(bias: int = BIAS) -> float:
    return (
        (2.0 - 2.0 ** -M_BITS)
        * 2.0 ** (EXP_MAX_STORED - bias)
    )


def encode(
    values,
    mode: str = "paper",
    bias: int = BIAS
) -> np.ndarray:
    """FP32/FP64 array -> uint16 E4M11 words (same shape).

    bias (default 7 = paper) shifts the 14 usable binades to
    [2^(1-bias), 2^(15-bias)).

    Every other rule (flush below 2^-bias, clip to max_finite(bias),
    carry) is the paper rule re-expressed in terms of the bias.
    bias != 7 is a deviation from the paper (Table 1).
    """

    if mode not in MODES:
        raise ValueError(
            f"mode must be one of {MODES}, got {mode!r}"
        )

    x = _as_f64(values)
    shape = x.shape
    x = x.reshape(-1)

    sign = np.signbit(x).astype(np.uint16)
    a = np.minimum(np.abs(x), max_finite(bias))

    word = (sign << 15).astype(np.uint16)

    if mode == "subnormal":
        sub = a < 2.0 ** (1 - bias)

        ms = np.rint(
            a[sub] / 2.0 ** (1 - bias - M_BITS)
        ).astype(np.int64)

        # m == 2048 is exactly 2^-6:
        # es=1, m=0 (carry into the normal range)
        word[sub] |= np.where(
            ms >= (1 << M_BITS),
            1 << M_BITS,
            ms
        ).astype(np.uint16)

        normal = ~sub

    else:
        normal = a >= 2.0 ** -bias

    if not normal.any():
        return word.reshape(shape)

    v = a[normal]

    if mode == "legacy":
        e = np.floor(np.log2(v)).astype(np.int64)
    else:
        e = np.frexp(v)[1].astype(np.int64) - 1

    e = np.clip(
        e,
        1 - bias,
        EXP_MAX_STORED - bias
    )

    frac = np.ldexp(v, -e) - 1.0

    if mode == "legacy":
        frac = np.clip(
            frac,
            0.0,
            1.0 - 2.0 ** -M_BITS
        )

        m = np.clip(
            np.rint(frac * (1 << M_BITS)),
            0,
            MANT_MASK
        ).astype(np.int64)

    else:
        frac = np.maximum(frac, 0.0)

        m = np.rint(
            frac * (1 << M_BITS)
        ).astype(np.int64)

        carry = m >= (1 << M_BITS)

        e = e + carry
        m = np.where(carry, 0, m)

        over = e > EXP_MAX_STORED - bias

        e = np.where(
            over,
            EXP_MAX_STORED - bias,
            e
        )

        m = np.where(over, MANT_MASK, m)

    es = (e + bias).astype(np.uint16)

    word[normal] |= (
        (es << M_BITS)
        | m.astype(np.uint16)
    )

    return word.reshape(shape)


def decode(
    words,
    mode: str = "paper",
    all_ones: str = "clamp",
    bias: int = BIAS
) -> np.ndarray:
    """uint16 E4M11 words -> float32 array (same shape)."""

    if mode not in MODES:
        raise ValueError(
            f"mode must be one of {MODES}, got {mode!r}"
        )

    if all_ones not in ALL_ONES:
        raise ValueError(
            f"all_ones must be one of {ALL_ONES}, got {all_ones!r}"
        )

    if mode == "legacy":
        all_ones = "clamp"

    w = np.asarray(words).astype(np.uint16)

    sign = ((w >> 15) & 1).astype(bool)
    es = ((w >> M_BITS) & 0xF).astype(np.int64)
    m = (w & MANT_MASK).astype(np.float64)

    if all_ones == "literal":
        e_used = np.where(es == 0, 1, es)
    else:
        e_used = np.clip(
            es,
            1,
            EXP_MAX_STORED
        )

    val = (
        (1.0 + m / (1 << M_BITS))
        * np.exp2(e_used - bias)
    )

    if mode == "subnormal":
        val = np.where(
            es == 0,
            m * 2.0 ** (1 - bias - M_BITS),
            val
        )
    else:
        val = np.where(es == 0, 0.0, val)

    if all_ones == "ieee":
        val = np.where(
            es == 15,
            np.where(m == 0, np.inf, np.nan),
            val
        )

    val = np.where(sign, -val, val)

    return val.astype(np.float32)


def quantize_dequantize(
    values,
    mode: str = "paper",
    bias: int = BIAS
) -> np.ndarray:
    """Round-trip FP -> E4M11 -> float32."""

    return decode(
        encode(values, mode, bias),
        mode,
        bias=bias
    )


# -----------------------------------------------------------------
# Torch helpers
# -----------------------------------------------------------------

def encode_tensor(
    t,
    mode: str = "paper",
    bias: int = BIAS
) -> np.ndarray:
    """torch tensor -> uint16 numpy words (torch has no usable uint16 ops)."""

    return encode(
        t.detach().float().cpu().numpy(),
        mode,
        bias
    )


def decode_to_tensor(
    words,
    dtype=None,
    device=None,
    mode: str = "paper",
    all_ones: str = "clamp",
    bias: int = BIAS
):
    """uint16 words -> torch tensor (float32 by default)."""

    import torch

    out = torch.from_numpy(
        decode(words, mode, all_ones, bias)
    )

    return out.to(
        device=device,
        dtype=dtype or torch.float32
    )


def decode_torch(
    words_int,
    all_ones: str = "clamp",
    mode: str = "paper",
    bias: int = BIAS
):
    """Pure-torch decoder for int32/int64 tensors holding the 16-bit words.

    Runs on any device (useful for fault injection on GPU later).
    Equivalent to decode() (paper mode) for the same all_ones policy.
    """

    import torch

    w = words_int.to(torch.int64) & 0xFFFF

    sign = (w >> 15) & 1
    es = (w >> M_BITS) & 0xF

    m = (w & MANT_MASK).to(torch.float64)

    e_used = (
        es.clamp(1, EXP_MAX_STORED)
        if all_ones != "literal"
        else es.clamp(min=1)
    )

    val = (
        (1.0 + m / (1 << M_BITS))
        * torch.exp2(
            (e_used - bias).to(torch.float64)
        )
    )

    zero_val = (
        m * 2.0 ** (1 - bias - M_BITS)
        if mode == "subnormal"
        else torch.zeros_like(val)
    )

    val = torch.where(
        es == 0,
        zero_val,
        val
    )

    if all_ones == "ieee":
        special = torch.where(
            m == 0,
            torch.full_like(val, float("inf")),
            torch.full_like(val, float("nan"))
        )

        val = torch.where(
            es == 15,
            special,
            val
        )

    val = torch.where(
        sign == 1,
        -val,
        val
    )

    return val.to(torch.float32)


In [ ]:
import os

os.makedirs("/kaggle/working/JFAMP/codec", exist_ok=True)

In [ ]:
import sys
import numpy as np
import torch

sys.path.insert(0, "/kaggle/working/JFAMP/codec")
import e4m11_codec as e4

# 1. Verify codec configuration
assert e4.E_BITS == 4
assert e4.M_BITS == 11
assert e4.BIAS == 7
assert e4.MODES == ("paper", "legacy", "subnormal")

# 2. Select our primary experiment configuration
E4M11_MODE = "subnormal"
E4M11_BIAS = 7
E4M11_ALL_ONES = "clamp"

# 3. Test representative values
test_values = np.array([
    0.0, -0.0,
    2**-18, 2**-17, 2**-16, 2**-15,
    2**-7, 2**-6,
    0.001, 0.01, 0.1, 1.0, 10.0, 100.0,
    e4.MAX_FINITE
], dtype=np.float32)

words = e4.encode(
    test_values,
    mode=E4M11_MODE,
    bias=E4M11_BIAS
)

decoded = e4.decode(
    words,
    mode=E4M11_MODE,
    all_ones=E4M11_ALL_ONES,
    bias=E4M11_BIAS
)

# 4. Verify NumPy and Torch decoder equivalence
torch_words = torch.from_numpy(words.astype(np.int32))

torch_decoded = e4.decode_torch(
    torch_words,
    all_ones=E4M11_ALL_ONES,
    mode=E4M11_MODE,
    bias=E4M11_BIAS
)

np.testing.assert_allclose(
    decoded,
    torch_decoded.numpy(),
    rtol=1e-6,
    atol=1e-8,
    equal_nan=True
)

# 5. Verify subnormal values are preserved
subnormal_values = np.array(
    [2**-17, 2**-16, 2**-15],
    dtype=np.float64
)

sub_words = e4.encode(
    subnormal_values,
    mode="subnormal",
    bias=7
)

sub_decoded = e4.decode(
    sub_words,
    mode="subnormal",
    all_ones="clamp",
    bias=7
)

assert np.all(sub_words != 0), "Subnormal encoding failed."
assert np.all(sub_decoded > 0), "Subnormal decoding failed."

np.testing.assert_allclose(
    sub_decoded,
    subnormal_values,
    rtol=0,
    atol=0
)

print("E4M11 codec verification PASSED")
print("Primary mode:", E4M11_MODE)
print("Bias:", E4M11_BIAS)
print("All-ones policy:", E4M11_ALL_ONES)
print("NumPy/Torch decoder equivalence: PASSED")
print("Subnormal preservation: PASSED")

print("\nSample encoding:")
for x, w, y in zip(test_values, words, decoded):
    print(f"Input: {x:12.8f} | Word: 0x{int(w):04X} | Decoded: {y:12.8f}")

In [ ]:
import os

BASE = "/kaggle/working/JFAMP/retrain_to150"

for f in [
    "checkpoints/resnet18_cifar100_best_150.pth",
    "fp32_baseline.json",
]:
    path = os.path.join(BASE, f)
    print(f, "EXISTS" if os.path.exists(path) else "MISSING")

In [ ]:

# ================================================================
# JFAMP EXPERIMENT 1
# E4M11 SUBNORMAL WEIGHT-ONLY QUANTIZATION
# ResNet-18 | CIFAR-100 | Best checkpoint: Epoch 139
# ================================================================

import os, sys, json, copy, time, random, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

# ---------------- CONFIGURATION ----------------

SEED = 42
BATCH_SIZE = 128
NUM_WORKERS = 2

BASE = "/kaggle/working/JFAMP"
TRAIN_DIR = os.path.join(BASE, "retrain_to150")

CHECKPOINT_PATH = os.path.join(
    TRAIN_DIR, "checkpoints/resnet18_cifar100_best_150.pth"
)
BASELINE_PATH = os.path.join(TRAIN_DIR, "fp32_baseline.json")

SPLIT_PATH = (
    "/kaggle/input/datasets/crahulnaidu/"
    "jfamp-resnet18-cifar100-checkpoint/"
    "data_split_indices.json"
)

CODEC_DIR = os.path.join(BASE, "codec")
sys.path.insert(0, CODEC_DIR)

import e4m11_codec as e4

MODE = "subnormal"
BIAS = 7
ALL_ONES = "clamp"

FP32_BASELINE_ACC = 77.78

OUTPUT_DIR = os.path.join(
    BASE, "experiments/e4m11_subnormal"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ---------------- VALIDATE FILES ----------------

for path in [CHECKPOINT_PATH, BASELINE_PATH, SPLIT_PATH]:
    assert os.path.exists(path), f"Missing: {path}"

print("=" * 65)
print("JFAMP: E4M11 SUBNORMAL WEIGHT-ONLY QUANTIZATION")
print("=" * 65)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ---------------- LOAD BASELINE ----------------

with open(BASELINE_PATH) as f:
    baseline_data = json.load(f)

assert baseline_data["validation_samples"] == 5000
assert abs(
    baseline_data["validation_accuracy"] - FP32_BASELINE_ACC
) < 1e-6

print("\nCanonical FP32 accuracy:", FP32_BASELINE_ACC)

# ---------------- LOAD ORIGINAL SPLIT ----------------

with open(SPLIT_PATH) as f:
    split_data = json.load(f)

assert split_data["seed"] == 42

train_indices = split_data["train_indices"]
val_indices = split_data["validation_indices"]

assert len(train_indices) == 45000
assert len(val_indices) == 5000
assert len(set(val_indices)) == 5000

print("Original split verified.")
print("Training indices:", len(train_indices))
print("Validation indices:", len(val_indices))

# ---------------- LOAD CHECKPOINT ----------------

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

state_dict = checkpoint.get(
    "model_state_dict",
    checkpoint.get("state_dict", checkpoint)
)

# ---------------- RECREATE MODEL ----------------

model = models.resnet18(weights=None)

model.conv1 = nn.Conv2d(
    3, 64, kernel_size=3,
    stride=1, padding=1, bias=False
)
model.maxpool = nn.Identity()
model.fc = nn.Linear(model.fc.in_features, 100)

model.load_state_dict(state_dict, strict=True)
model = model.to(device)
model.eval()

print("\nModel loaded successfully.")
print("Checkpoint epoch:", checkpoint.get("epoch"))

# ---------------- CIFAR-100 VALIDATION DATA ----------------

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

data_root = os.path.join(BASE, "data")
os.makedirs(data_root, exist_ok=True)

dataset = datasets.CIFAR100(
    root=data_root,
    train=True,
    download=True,
    transform=eval_transform
)

val_dataset = Subset(dataset, val_indices)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

# ---------------- EVALUATION FUNCTION ----------------

def evaluate(model, loader):
    model.eval()

    criterion = nn.CrossEntropyLoss(reduction="sum")

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images.float())
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            correct += (
                outputs.argmax(dim=1) == labels
            ).sum().item()
            total += labels.size(0)

    return {
        "accuracy": 100.0 * correct / total,
        "loss": total_loss / total,
        "correct": correct,
        "total": total
    }

# ---------------- WEIGHT QUANTIZATION ----------------

quantized_model = copy.deepcopy(model).to(device)
quantized_model.eval()

sqnr_signal = 0.0
sqnr_noise = 0.0

total_weights = 0
subnormal_count = 0
zero_count = 0
saturated_count = 0

layer_results = []

print("\nQuantizing convolutional and linear weights...")
print("Codec mode:", MODE)

for name, module in quantized_model.named_modules():

    if not isinstance(module, (nn.Conv2d, nn.Linear)):
        continue

    original = (
        module.weight.detach().float().cpu().numpy()
    )

    words = e4.encode(
        original,
        mode=MODE,
        bias=BIAS
    )

    reconstructed = e4.decode(
        words,
        mode=MODE,
        all_ones=ALL_ONES,
        bias=BIAS
    )

    module.weight.data.copy_(
        torch.from_numpy(reconstructed).to(
            device=module.weight.device,
            dtype=module.weight.dtype
        )
    )

    original64 = original.astype(np.float64)
    reconstructed64 = reconstructed.astype(np.float64)

    signal = np.sum(original64 ** 2)
    noise = np.sum((original64 - reconstructed64) ** 2)

    sqnr_signal += signal
    sqnr_noise += noise

    total_weights += original.size

    unsigned_words = words.astype(np.uint16)

    exponent = (
        (unsigned_words >> e4.M_BITS) & 0xF
    )
    mantissa = unsigned_words & e4.MANT_MASK

    layer_subnormal = np.sum(
        (exponent == 0) & (mantissa != 0)
    )
    layer_zero = np.sum(
        (exponent == 0) & (mantissa == 0)
    )
    layer_saturated = np.sum(
        np.abs(original64) > e4.max_finite(BIAS)
    )

    subnormal_count += int(layer_subnormal)
    zero_count += int(layer_zero)
    saturated_count += int(layer_saturated)

    layer_results.append({
        "layer": name,
        "weights": original.size,
        "subnormal": int(layer_subnormal),
        "zero": int(layer_zero),
        "saturated": int(layer_saturated),
        "signal_power": float(signal),
        "noise_power": float(noise)
    })

    print(
        f"{name:35s} "
        f"Weights={original.size:9,d} "
        f"Subnormal={int(layer_subnormal):8,d} "
        f"Zero={int(layer_zero):8,d}"
    )

# ---------------- SQNR ----------------

global_sqnr = (
    10 * np.log10(sqnr_signal / sqnr_noise)
    if sqnr_noise > 0 else float("inf")
)

# ---------------- EVALUATE ----------------

print("\nEvaluating E4M11 model...")

start = time.time()
metrics = evaluate(quantized_model, val_loader)
elapsed = time.time() - start

accuracy_change = metrics["accuracy"] - FP32_BASELINE_ACC

# ---------------- RESULTS ----------------

results = {
    "experiment": "E4M11_subnormal_weight_only",
    "model": "ResNet18_CIFAR100",
    "checkpoint_epoch": checkpoint.get("epoch"),
    "validation_samples": metrics["total"],
    "fp32_baseline_accuracy": FP32_BASELINE_ACC,
    "quantized_accuracy": metrics["accuracy"],
    "accuracy_change_pp": accuracy_change,
    "quantized_loss": metrics["loss"],
    "global_weight_sqnr_db": float(global_sqnr),
    "total_quantized_weights": total_weights,
    "subnormal_weights": subnormal_count,
    "subnormal_percentage": 100 * subnormal_count / total_weights,
    "zero_weights": zero_count,
    "zero_percentage": 100 * zero_count / total_weights,
    "saturated_weights": saturated_count,
    "saturated_percentage": 100 * saturated_count / total_weights,
    "encoding": "E4M11",
    "mode": MODE,
    "bias": BIAS,
    "all_ones": ALL_ONES,
    "activation_precision": "FP32",
    "inference_precision": "FP32 after weight reconstruction",
    "evaluation_seconds": elapsed
}

# ---------------- SAVE RESULTS ----------------

with open(
    os.path.join(OUTPUT_DIR, "e4m11_subnormal_results.json"),
    "w"
) as f:
    json.dump(results, f, indent=4)

pd.DataFrame(layer_results).to_csv(
    os.path.join(OUTPUT_DIR, "e4m11_layerwise_statistics.csv"),
    index=False
)

torch.save(
    {
        "model_state_dict": quantized_model.state_dict(),
        "encoding": "E4M11",
        "mode": MODE,
        "bias": BIAS,
        "all_ones": ALL_ONES,
        "checkpoint_epoch": checkpoint.get("epoch"),
        "validation_accuracy": metrics["accuracy"]
    },
    os.path.join(
        OUTPUT_DIR,
        "resnet18_e4m11_subnormal_dequantized_fp32.pth"
    )
)

config = {
    "checkpoint_path": CHECKPOINT_PATH,
    "split_path": SPLIT_PATH,
    "codec_path": os.path.join(CODEC_DIR, "e4m11_codec.py"),
    "mode": MODE,
    "bias": BIAS,
    "all_ones": ALL_ONES,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "quantized_parameters": "Conv2d and Linear weights only",
    "excluded_parameters": "Biases and BatchNorm parameters",
    "activation_precision": "FP32",
    "inference_precision": "FP32"
}

with open(
    os.path.join(OUTPUT_DIR, "experiment_config.json"),
    "w"
) as f:
    json.dump(config, f, indent=4)

# ---------------- FINAL REPORT ----------------

print("\n" + "=" * 65)
print("E4M11 SUBNORMAL RESULTS")
print("=" * 65)

print(f"FP32 baseline accuracy : {FP32_BASELINE_ACC:.4f}%")
print(f"E4M11 accuracy         : {metrics['accuracy']:.4f}%")
print(f"Accuracy change        : {accuracy_change:+.4f} pp")
print(f"E4M11 validation loss  : {metrics['loss']:.6f}")
print(f"Global weight SQNR     : {global_sqnr:.4f} dB")

print("\nWeight statistics:")
print(f"Total weights          : {total_weights:,}")
print(
    f"Subnormal weights      : {subnormal_count:,} "
    f"({100*subnormal_count/total_weights:.4f}%)"
)
print(
    f"Zero weights           : {zero_count:,} "
    f"({100*zero_count/total_weights:.4f}%)"
)
print(
    f"Saturated weights      : {saturated_count:,} "
    f"({100*saturated_count/total_weights:.6f}%)"
)

print("\nTheoretical weight storage:")
print("FP32: 32 bits/weight")
print("E4M11: 16 bits/weight")
print("Theoretical reduction: 50% (excluding metadata/alignment)")

print("\nEvaluation time:", round(elapsed, 2), "seconds")
print("\nResults saved to:", OUTPUT_DIR)

print("\nCOMPLETE RESULT SUMMARY:")
print(json.dumps(results, indent=4))

print("\nNOTE: Weight-only simulation; reconstructed weights and")
print("activations are FP32 during inference.")


In [ ]:

# ================================================================
# JFAMP — PRECISION BASELINE COLLECTION
# FP16 | BF16 | INT8 | INT4 | FP8 E4M3FN | FP8 E5M2
# Same checkpoint, same validation split, no retraining.
# ================================================================

import os, json, time, copy, math, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset

warnings.filterwarnings("ignore")

# ---------------- CONFIGURATION ----------------
ROOT = "/kaggle/working/JFAMP"
CKPT = ROOT + "/retrain_to150/checkpoints/resnet18_cifar100_best_150.pth"
SPLIT = "/kaggle/input/datasets/crahulnaidu/jfamp-resnet18-cifar100-checkpoint/data_split_indices.json"
BASELINE = ROOT + "/retrain_to150/fp32_baseline.json"
OUT = ROOT + "/experiments/precision_comparison"

os.makedirs(OUT, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 256
NUM_WORKERS = 2

print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ---------------- LOAD CANONICAL BASELINE ----------------
with open(BASELINE) as f:
    baseline = json.load(f)

FP32_ACC = float(baseline["accuracy"])
print(f"Canonical FP32 accuracy: {FP32_ACC:.4f}%")

# ---------------- LOAD CHECKPOINT ----------------
checkpoint = torch.load(CKPT, map_location="cpu", weights_only=False)
state = checkpoint.get("model_state_dict", checkpoint.get("state_dict", checkpoint))

# ---------------- MODEL ----------------
def build_model():
    model = models.resnet18(weights=None)
    model.conv1 = nn.Conv2d(
        3, 64, kernel_size=3, stride=1, padding=1, bias=False
    )
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, 100)
    return model

base_model = build_model()
base_model.load_state_dict(state)
base_model.eval()

# ---------------- ORIGINAL VALIDATION SPLIT ----------------
with open(SPLIT) as f:
    split = json.load(f)

val_indices = split["validation_indices"]

mean = (0.5071, 0.4867, 0.4408)
std = (0.2675, 0.2565, 0.2761)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

dataset = datasets.CIFAR100(
    root="/kaggle/working/cifar100",
    train=True,
    download=True,
    transform=transform
)

val_dataset = Subset(dataset, val_indices)

loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

# ---------------- EVALUATION ----------------
@torch.no_grad()
def evaluate(model):
    model = model.to(DEVICE)
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0
    criterion = nn.CrossEntropyLoss(reduction="sum")

    start = time.time()

    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        logits = model(x)
        total_loss += criterion(logits, y).item()
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)

    elapsed = time.time() - start
    model.cpu()

    return {
        "accuracy": 100.0 * correct / total,
        "loss": total_loss / total,
        "evaluation_seconds": elapsed
    }

# ---------------- WEIGHT QUANTIZERS ----------------
def quantize_symmetric_per_tensor(w, bits):
    qmax = (2 ** (bits - 1)) - 1
    scale = w.abs().max().clamp(min=1e-12) / qmax
    q = torch.round(w / scale).clamp(-qmax, qmax)
    return q * scale

def quantize_symmetric_per_channel(w, bits):
    qmax = (2 ** (bits - 1)) - 1
    dims = tuple(range(1, w.ndim))
    maximum = w.abs().amax(dim=dims, keepdim=True).clamp(min=1e-12)
    scale = maximum / qmax
    q = torch.round(w / scale).clamp(-qmax, qmax)
    return q * scale

def quantize_fp(w, dtype):
    return w.to(dtype).to(torch.float32)

def collect_weight_stats(original, reconstructed):
    a = original.detach().float().cpu()
    b = reconstructed.detach().float().cpu()

    signal = torch.sum(a * a).item()
    noise = torch.sum((a - b) ** 2).item()

    sqnr = (
        10 * math.log10(signal / noise)
        if noise > 0 else float("inf")
    )

    return {
        "sqnr_db": sqnr,
        "zero_percentage": 100.0 * (b == 0).sum().item() / b.numel(),
        "weight_count": b.numel()
    }

def apply_weight_format(model, format_name):
    total_signal = 0.0
    total_noise = 0.0
    total_zeros = 0
    total_weights = 0

    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            original = module.weight.data.float().clone()

            if format_name == "FP16":
                reconstructed = quantize_fp(original, torch.float16)

            elif format_name == "BF16":
                reconstructed = quantize_fp(original, torch.bfloat16)

            elif format_name == "INT8":
                reconstructed = quantize_symmetric_per_channel(original, 8)

            elif format_name == "INT4":
                reconstructed = quantize_symmetric_per_tensor(original, 4)

            elif format_name == "FP8_E4M3FN":
                if not hasattr(torch, "float8_e4m3fn"):
                    raise RuntimeError("FP8 E4M3FN unavailable in this PyTorch build")
                reconstructed = quantize_fp(original, torch.float8_e4m3fn)

            elif format_name == "FP8_E5M2":
                if not hasattr(torch, "float8_e5m2"):
                    raise RuntimeError("FP8 E5M2 unavailable in this PyTorch build")
                reconstructed = quantize_fp(original, torch.float8_e5m2)

            else:
                raise ValueError(format_name)

            module.weight.data.copy_(reconstructed.to(module.weight.dtype))

            total_signal += torch.sum(original ** 2).item()
            total_noise += torch.sum((original - reconstructed) ** 2).item()
            total_zeros += (reconstructed == 0).sum().item()
            total_weights += original.numel()

    sqnr = (
        10 * math.log10(total_signal / total_noise)
        if total_noise > 0 else float("inf")
    )

    return {
        "sqnr_db": sqnr,
        "zero_weight_percentage": 100.0 * total_zeros / total_weights,
        "total_weights": total_weights
    }

# ---------------- RUN COMPARISONS ----------------
formats = [
    ("FP16", 16),
    ("BF16", 16),
    ("INT8", 8),
    ("INT4", 4),
    ("FP8_E4M3FN", 8),
    ("FP8_E5M2", 8)
]

results = []

# Include canonical FP32 baseline
results.append({
    "format": "FP32",
    "nominal_bits_per_weight": 32,
    "accuracy": FP32_ACC,
    "accuracy_change_pp": 0.0,
    "loss": baseline.get("loss", None),
    "sqnr_db": float("inf"),
    "zero_weight_percentage": 0.0,
    "theoretical_storage_reduction_pct": 0.0,
    "evaluation_seconds": None,
    "status": "baseline"
})

for fmt, bits in formats:
    print(f"\nRunning {fmt} ...")

    model = build_model()
    model.load_state_dict(state)
    model.eval()

    try:
        stats = apply_weight_format(model, fmt)
        metrics = evaluate(model)

        row = {
            "format": fmt,
            "nominal_bits_per_weight": bits,
            "accuracy": metrics["accuracy"],
            "accuracy_change_pp": metrics["accuracy"] - FP32_ACC,
            "loss": metrics["loss"],
            "sqnr_db": stats["sqnr_db"],
            "zero_weight_percentage": stats["zero_weight_percentage"],
            "theoretical_storage_reduction_pct": 100.0 * (1 - bits / 32),
            "evaluation_seconds": metrics["evaluation_seconds"],
            "status": "success"
        }

        results.append(row)

        print(
            f"Accuracy: {row['accuracy']:.4f}% | "
            f"Change: {row['accuracy_change_pp']:+.4f} pp | "
            f"SQNR: {row['sqnr_db']:.2f} dB"
        )

    except Exception as e:
        print(f"{fmt} skipped: {repr(e)}")
        results.append({
            "format": fmt,
            "nominal_bits_per_weight": bits,
            "status": "unsupported_or_failed",
            "error": repr(e)
        })

    finally:
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

# ---------------- SAVE RESULTS ----------------
df = pd.DataFrame(results)

csv_path = OUT + "/precision_comparison.csv"
json_path = OUT + "/precision_comparison.json"

df.to_csv(csv_path, index=False)

with open(json_path, "w") as f:
    json.dump(results, f, indent=2, allow_nan=True)

print("\n" + "=" * 65)
print("PRECISION COMPARISON COMPLETE")
print("=" * 65)

display(df)

print("\nSaved:")
print(csv_path)
print(json_path)

print("\nIMPORTANT:")
print("These are weight-only quantization simulations.")
print("All reconstructed weights use FP32 inference.")
print("No native low-precision hardware speedup or power result is claimed.")
print("No retraining was performed.")


In [ ]:

# =====================================================================
# JFAMP — PRECISION COMPARISON EXPERIMENT
#
# Formats:
#   FP32   : Original reference
#   FP16   : IEEE half precision
#   BF16   : Brain floating point
#   INT8   : Symmetric per-output-channel weight quantization
#   INT4   : Symmetric per-tensor weight quantization
#   FP8 E4M3FN
#   FP8 E5M2
#
# Method:
#   Weight-only quantization -> FP32 reconstruction -> FP32 inference
#
# Same epoch-139 checkpoint and original validation split.
# No retraining. Original checkpoint remains untouched.
# =====================================================================

import os
import json
import time
import math
import copy
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------------------

ROOT = "/kaggle/working/JFAMP"

CKPT = (
    ROOT +
    "/retrain_to150/checkpoints/resnet18_cifar100_best_150.pth"
)

SPLIT = (
    "/kaggle/input/datasets/crahulnaidu/"
    "jfamp-resnet18-cifar100-checkpoint/data_split_indices.json"
)

BASELINE = ROOT + "/retrain_to150/fp32_baseline.json"

OUT = ROOT + "/experiments/precision_comparison"

os.makedirs(OUT, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 256
NUM_WORKERS = 2

print("=" * 70)
print("JFAMP PRECISION COMPARISON")
print("=" * 70)

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# ---------------------------------------------------------------------
# 2. LOAD CHECKPOINT
# ---------------------------------------------------------------------

if not os.path.exists(CKPT):
    raise FileNotFoundError(f"Checkpoint not found: {CKPT}")

checkpoint = torch.load(
    CKPT,
    map_location="cpu",
    weights_only=False
)

if "model_state_dict" in checkpoint:
    state = checkpoint["model_state_dict"]
elif "state_dict" in checkpoint:
    state = checkpoint["state_dict"]
else:
    state = checkpoint

print("\nCheckpoint loaded:", CKPT)

if isinstance(checkpoint, dict):
    print("Checkpoint epoch:", checkpoint.get("epoch", "Not recorded"))


# ---------------------------------------------------------------------
# 3. MODEL ARCHITECTURE
# ---------------------------------------------------------------------

def build_model():

    model = models.resnet18(weights=None)

    model.conv1 = nn.Conv2d(
        3,
        64,
        kernel_size=3,
        stride=1,
        padding=1,
        bias=False
    )

    model.maxpool = nn.Identity()

    model.fc = nn.Linear(
        model.fc.in_features,
        100
    )

    return model


base_model = build_model()
base_model.load_state_dict(state)
base_model.eval()

print("ResNet-18 CIFAR-100 architecture loaded successfully.")


# ---------------------------------------------------------------------
# 4. LOAD ORIGINAL VALIDATION SPLIT
# ---------------------------------------------------------------------

with open(SPLIT, "r") as f:
    split = json.load(f)

if "validation_indices" not in split:
    raise KeyError(
        "Expected 'validation_indices' in split JSON. "
        f"Available keys: {list(split.keys())}"
    )

val_indices = split["validation_indices"]

print("\nOriginal validation samples:", len(val_indices))

mean = (0.5071, 0.4867, 0.4408)
std = (0.2675, 0.2565, 0.2761)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

DATA_ROOT = "/kaggle/working/cifar100"

dataset = datasets.CIFAR100(
    root=DATA_ROOT,
    train=True,
    download=True,
    transform=transform
)

val_dataset = Subset(
    dataset,
    val_indices
)

loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

print("Validation loader ready:", len(val_dataset), "samples")


# ---------------------------------------------------------------------
# 5. EVALUATION FUNCTION
# ---------------------------------------------------------------------

@torch.no_grad()
def evaluate(model):

    model = model.to(DEVICE)
    model.eval()

    criterion = nn.CrossEntropyLoss(reduction="sum")

    total_loss = 0.0
    correct = 0
    total = 0

    start = time.time()

    for images, labels in loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        outputs = model(images)

        total_loss += criterion(
            outputs,
            labels
        ).item()

        predictions = outputs.argmax(dim=1)

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    elapsed = time.time() - start

    model.cpu()

    return {
        "accuracy": 100.0 * correct / total,
        "loss": total_loss / total,
        "evaluation_seconds": elapsed,
        "samples": total
    }


# ---------------------------------------------------------------------
# 6. ROBUST BASELINE JSON READER
# ---------------------------------------------------------------------

def find_accuracy(obj, path=""):

    preferred_keys = [
        "accuracy",
        "fp32_accuracy",
        "baseline_accuracy",
        "validation_accuracy",
        "val_accuracy",
        "test_accuracy",
        "accuracy_percent",
        "fp32_baseline_accuracy"
    ]

    if isinstance(obj, dict):

        # First search preferred keys at this level.
        for key in preferred_keys:

            if key in obj:

                value = obj[key]

                if isinstance(value, (int, float)):
                    return float(value), path + "/" + key

        # Then recursively inspect nested dictionaries.
        for key, value in obj.items():

            if isinstance(value, (dict, list)):

                result = find_accuracy(
                    value,
                    path + "/" + key
                )

                if result is not None:
                    return result

    elif isinstance(obj, list):

        for i, value in enumerate(obj):

            result = find_accuracy(
                value,
                path + f"/{i}"
            )

            if result is not None:
                return result

    return None


FP32_ACC = None
baseline = {}

if os.path.exists(BASELINE):

    with open(BASELINE, "r") as f:
        baseline = json.load(f)

    result = find_accuracy(baseline)

    if result is not None:

        FP32_ACC, selected_key = result

        if 0 <= FP32_ACC <= 1:
            FP32_ACC *= 100

        print("\nBaseline accuracy loaded from:", selected_key)

    else:

        print("\nNo recognizable accuracy field in baseline JSON.")
        print("Will evaluate the FP32 checkpoint directly.")

else:

    print("\nBaseline JSON not found.")
    print("Will evaluate the FP32 checkpoint directly.")


# ---------------------------------------------------------------------
# 7. EVALUATE FP32 REFERENCE IF NECESSARY
# ---------------------------------------------------------------------

if FP32_ACC is None:

    print("\nEvaluating FP32 reference...")

    fp32_model = build_model()
    fp32_model.load_state_dict(state)

    fp32_metrics = evaluate(fp32_model)

    FP32_ACC = fp32_metrics["accuracy"]

    print(f"FP32 accuracy: {FP32_ACC:.4f}%")

    with open(
        OUT + "/fp32_reference_evaluated.json",
        "w"
    ) as f:

        json.dump(
            fp32_metrics,
            f,
            indent=2
        )

    del fp32_model

else:

    print(f"\nLoaded FP32 accuracy: {FP32_ACC:.4f}%")


# ---------------------------------------------------------------------
# 8. QUANTIZATION FUNCTIONS
# ---------------------------------------------------------------------

def quantize_symmetric_per_tensor(w, bits):

    qmax = (2 ** (bits - 1)) - 1

    scale = (
        w.abs().max().clamp(min=1e-12) / qmax
    )

    q = torch.round(w / scale)

    q = q.clamp(-qmax, qmax)

    return q * scale


def quantize_symmetric_per_channel(w, bits):

    qmax = (2 ** (bits - 1)) - 1

    dims = tuple(range(1, w.ndim))

    maximum = (
        w.abs()
        .amax(dim=dims, keepdim=True)
        .clamp(min=1e-12)
    )

    scale = maximum / qmax

    q = torch.round(w / scale)

    q = q.clamp(-qmax, qmax)

    return q * scale


def quantize_fp(w, dtype):

    return w.to(dtype).to(torch.float32)


# ---------------------------------------------------------------------
# 9. APPLY WEIGHT QUANTIZATION AND COLLECT STATISTICS
# ---------------------------------------------------------------------

def apply_weight_format(model, format_name):

    total_signal = 0.0
    total_noise = 0.0
    total_zeros = 0
    total_weights = 0

    for module in model.modules():

        if isinstance(module, (nn.Conv2d, nn.Linear)):

            original = (
                module.weight.data
                .float()
                .clone()
            )

            if format_name == "FP16":

                reconstructed = quantize_fp(
                    original,
                    torch.float16
                )

            elif format_name == "BF16":

                reconstructed = quantize_fp(
                    original,
                    torch.bfloat16
                )

            elif format_name == "INT8":

                reconstructed = quantize_symmetric_per_channel(
                    original,
                    8
                )

            elif format_name == "INT4":

                reconstructed = quantize_symmetric_per_tensor(
                    original,
                    4
                )

            elif format_name == "FP8_E4M3FN":

                if not hasattr(torch, "float8_e4m3fn"):
                    raise RuntimeError("FP8 E4M3FN unavailable")

                reconstructed = quantize_fp(
                    original,
                    torch.float8_e4m3fn
                )

            elif format_name == "FP8_E5M2":

                if not hasattr(torch, "float8_e5m2"):
                    raise RuntimeError("FP8 E5M2 unavailable")

                reconstructed = quantize_fp(
                    original,
                    torch.float8_e5m2
                )

            else:

                raise ValueError(
                    f"Unknown format: {format_name}"
                )

            module.weight.data.copy_(
                reconstructed.to(module.weight.dtype)
            )

            total_signal += torch.sum(
                original ** 2
            ).item()

            total_noise += torch.sum(
                (original - reconstructed) ** 2
            ).item()

            total_zeros += (
                reconstructed == 0
            ).sum().item()

            total_weights += original.numel()

    sqnr = (
        10 * math.log10(total_signal / total_noise)
        if total_noise > 0
        else float("inf")
    )

    return {
        "sqnr_db": sqnr,
        "zero_weight_percentage": (
            100.0 * total_zeros / total_weights
        ),
        "total_weights": total_weights
    }


# ---------------------------------------------------------------------
# 10. RUN PRECISION EXPERIMENTS
# ---------------------------------------------------------------------

formats = [
    ("FP16", 16),
    ("BF16", 16),
    ("INT8", 8),
    ("INT4", 4),
    ("FP8_E4M3FN", 8),
    ("FP8_E5M2", 8)
]

results = []

# FP32 reference row
results.append({

    "format": "FP32",

    "nominal_bits_per_weight": 32,

    "accuracy": FP32_ACC,

    "accuracy_change_pp": 0.0,

    "loss": baseline.get("loss"),

    "sqnr_db": None,

    "zero_weight_percentage": 0.0,

    "theoretical_storage_reduction_pct": 0.0,

    "evaluation_seconds": None,

    "status": "baseline"

})


for fmt, bits in formats:

    print("\n" + "-" * 65)
    print("Running:", fmt)
    print("-" * 65)

    model = build_model()
    model.load_state_dict(state)
    model.eval()

    try:

        stats = apply_weight_format(
            model,
            fmt
        )

        metrics = evaluate(model)

        row = {

            "format": fmt,

            "nominal_bits_per_weight": bits,

            "accuracy": metrics["accuracy"],

            "accuracy_change_pp": (
                metrics["accuracy"] - FP32_ACC
            ),

            "loss": metrics["loss"],

            "sqnr_db": stats["sqnr_db"],

            "zero_weight_percentage": (
                stats["zero_weight_percentage"]
            ),

            "theoretical_storage_reduction_pct": (
                100.0 * (1 - bits / 32)
            ),

            "evaluation_seconds": (
                metrics["evaluation_seconds"]
            ),

            "status": "success"

        }

        results.append(row)

        print(
            f"Accuracy: {row['accuracy']:.4f}%"
        )

        print(
            f"Accuracy change: {row['accuracy_change_pp']:+.4f} pp"
        )

        print(
            f"SQNR: {row['sqnr_db']:.4f} dB"
        )

        print(
            f"Zero weights: {row['zero_weight_percentage']:.4f}%"
        )

        print(
            f"Theoretical storage reduction: "
            f"{row['theoretical_storage_reduction_pct']:.2f}%"
        )

    except Exception as e:

        print(f"ERROR in {fmt}: {repr(e)}")

        results.append({

            "format": fmt,

            "nominal_bits_per_weight": bits,

            "status": "unsupported_or_failed",

            "error": repr(e)

        })

    finally:

        del model

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()


# ---------------------------------------------------------------------
# 11. SAVE RESULTS
# ---------------------------------------------------------------------

df = pd.DataFrame(results)

csv_path = OUT + "/precision_comparison.csv"

json_path = OUT + "/precision_comparison.json"

df.to_csv(
    csv_path,
    index=False
)

with open(json_path, "w") as f:

    json.dump(
        results,
        f,
        indent=2,
        allow_nan=True
    )


# ---------------------------------------------------------------------
# 12. FINAL REPORT
# ---------------------------------------------------------------------

print("\n" + "=" * 70)
print("JFAMP PRECISION COMPARISON COMPLETED")
print("=" * 70)

print(f"\nFP32 reference accuracy: {FP32_ACC:.4f}%")

display(df)

print("\nSaved files:")

print(csv_path)

print(json_path)

print("\nExperiment configuration:")
print("Checkpoint: epoch-139 best checkpoint")
print("Dataset: CIFAR-100")
print("Validation samples:", len(val_dataset))
print("Validation split: original validation_indices")
print("Quantization scope: Conv2d and Linear weights")
print("Inference: FP32 after weight reconstruction")
print("Retraining: None")

print("\nIMPORTANT INTERPRETATION:")
print("- These are weight-only numerical quantization simulations.")
print("- FP16/BF16/FP8/INT8/INT4 weights are reconstructed to FP32.")
print("- All inference is performed using FP32 arithmetic.")
print("- Theoretical storage reduction excludes scales, metadata and alignment.")
print("- No hardware latency, power or native low-precision throughput is measured.")
print("- Original checkpoint and E4M11 experiment files remain unchanged.")

print("\nUse Kaggle Quick Save. Do not use Save and Run All.")


In [2]:

# =====================================================================
# JFAMP DAY 2 — LAYER-WISE QUANTIZATION PROFILING
#
# Model: ResNet-18 / CIFAR-100
# Checkpoint: Epoch 139 (best)
# Validation: Original 5,000-sample validation split
#
# 21 layers x 7 datatypes = 147 isolated-layer experiments
#
# Each experiment:
#   1. Start from the original FP32 checkpoint.
#   2. Quantize exactly ONE Conv2d/Linear weight tensor.
#   3. Keep all other weights and all activations in FP32.
#   4. Evaluate the complete validation set.
#   5. Measure SQNR, MSE, zero weights, accuracy and loss.
#
# Results are saved incrementally to protect against interruptions.
# No retraining. No changes to original checkpoint or codec.
# =====================================================================

import os
import sys
import json
import time
import math
import copy
import inspect
import warnings
import hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------------------

ROOT = "/kaggle/working/JFAMP"

CHECKPOINT_PATH = (
    ROOT + "/retrain_to150/checkpoints/"
    "resnet18_cifar100_best_150.pth"
)

SPLIT_PATH = (
    "/kaggle/input/datasets/crahulnaidu/"
    "jfamp-resnet18-cifar100-checkpoint/"
    "data_split_indices.json"
)

CODEC_PATH = ROOT + "/codec"

OUTPUT_DIR = (
    ROOT + "/experiments/layerwise_quantization"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

sys.path.insert(0, CODEC_PATH)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

BATCH_SIZE = 256
NUM_WORKERS = 2

# Exact teammate codec configuration
E4M11_MODE = "subnormal"
E4M11_BIAS = 7
E4M11_ALL_ONES = "clamp"

# All candidate formats
FORMATS = [
    ("FP16", 16),
    ("BF16", 16),
    ("INT8", 8),
    ("INT4", 4),
    ("FP8_E4M3FN", 8),
    ("FP8_E5M2", 8),
    ("E4M11_SUBNORMAL", 16),
]

CSV_PATH = OUTPUT_DIR + "/layerwise_quantization.csv"
JSON_PATH = OUTPUT_DIR + "/layerwise_quantization.json"
CONFIG_PATH = OUTPUT_DIR + "/experiment_config.json"

print("=" * 75)
print("JFAMP — LAYER-WISE QUANTIZATION PROFILING")
print("=" * 75)

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# ---------------------------------------------------------------------
# 2. CHECK REQUIRED FILES
# ---------------------------------------------------------------------

for path in [CHECKPOINT_PATH, SPLIT_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(path)

# Import exact teammate E4M11 implementation.
try:
    import e4m11_codec

    encode_tensor = e4m11_codec.encode_tensor
    decode_to_tensor = e4m11_codec.decode_to_tensor

    print("\nExact E4M11 codec imported:", e4m11_codec.__file__)

except Exception as e:
    raise RuntimeError(
        "Could not import the existing E4M11 codec. "
        "Check JFAMP/codec/e4m11_codec.py. "
        f"Original error: {repr(e)}"
    )


# ---------------------------------------------------------------------
# 3. LOAD CHECKPOINT
# ---------------------------------------------------------------------

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

if "model_state_dict" in checkpoint:
    state = checkpoint["model_state_dict"]
elif "state_dict" in checkpoint:
    state = checkpoint["state_dict"]
else:
    state = checkpoint

checkpoint_epoch = checkpoint.get("epoch", 139)

print("\nCheckpoint epoch:", checkpoint_epoch)


# ---------------------------------------------------------------------
# 4. BUILD MODEL
# ---------------------------------------------------------------------

def build_model():

    model = models.resnet18(weights=None)

    model.conv1 = nn.Conv2d(
        3, 64,
        kernel_size=3,
        stride=1,
        padding=1,
        bias=False
    )

    model.maxpool = nn.Identity()

    model.fc = nn.Linear(
        model.fc.in_features,
        100
    )

    return model


base_model = build_model()
base_model.load_state_dict(state)
base_model.eval()

# All experiments start from this immutable FP32 model.
print("FP32 model loaded successfully.")


# ---------------------------------------------------------------------
# 5. ORIGINAL VALIDATION DATASET
# ---------------------------------------------------------------------

with open(SPLIT_PATH, "r") as f:
    split = json.load(f)

if "validation_indices" not in split:
    raise KeyError(
        "Original split must contain validation_indices. "
        f"Found keys: {list(split.keys())}"
    )

val_indices = split["validation_indices"]

mean = (0.5071, 0.4867, 0.4408)
std = (0.2675, 0.2565, 0.2761)

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

dataset = datasets.CIFAR100(
    root="/kaggle/working/cifar100",
    train=True,
    download=True,
    transform=eval_transform
)

val_dataset = Subset(dataset, val_indices)

loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

print("Validation samples:", len(val_dataset))


# ---------------------------------------------------------------------
# 6. EVALUATION FUNCTION
# ---------------------------------------------------------------------

@torch.inference_mode()
def evaluate(model):

    model = model.to(DEVICE)
    model.eval()

    criterion = nn.CrossEntropyLoss(reduction="sum")

    correct = 0
    total = 0
    total_loss = 0.0

    start = time.time()

    for images, labels in loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        outputs = model(images)

        total_loss += criterion(
            outputs,
            labels
        ).item()

        correct += (
            outputs.argmax(1) == labels
        ).sum().item()

        total += labels.size(0)

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.time() - start

    model.cpu()

    return {
        "accuracy": 100.0 * correct / total,
        "loss": total_loss / total,
        "evaluation_seconds": elapsed,
        "samples": total
    }


# ---------------------------------------------------------------------
# 7. QUANTIZATION FUNCTIONS
# ---------------------------------------------------------------------

def quantize_symmetric_per_tensor(w, bits):

    qmax = (2 ** (bits - 1)) - 1

    scale = (
        w.abs().max().clamp(min=1e-12) / qmax
    )

    q = torch.round(w / scale).clamp(-qmax, qmax)

    return q * scale


def quantize_symmetric_per_channel(w, bits):

    qmax = (2 ** (bits - 1)) - 1

    dims = tuple(range(1, w.ndim))

    maximum = (
        w.abs()
        .amax(dim=dims, keepdim=True)
        .clamp(min=1e-12)
    )

    scale = maximum / qmax

    q = torch.round(w / scale).clamp(-qmax, qmax)

    return q * scale


def quantize_fp(w, dtype):

    return w.to(dtype).to(torch.float32)


# ---------------------------------------------------------------------
# 8. EXACT E4M11 CODEC ADAPTER
# ---------------------------------------------------------------------

def call_codec_function(function, value):

    """
    Call the supplied codec while passing only supported options.
    No encoding or decoding logic is reimplemented here.
    """

    signature = inspect.signature(function)

    available = signature.parameters

    kwargs = {}

    options = {
        "mode": E4M11_MODE,
        "bias": E4M11_BIAS,
        "all_ones": E4M11_ALL_ONES,
    }

    for key, val in options.items():

        if key in available:
            kwargs[key] = val

    return function(value, **kwargs)


def quantize_e4m11(w):

    encoded = call_codec_function(
        encode_tensor,
        w.detach().cpu()
    )

    decoded = call_codec_function(
        decode_to_tensor,
        encoded
    )

    if isinstance(decoded, tuple):
        decoded = decoded[0]

    reconstructed = torch.as_tensor(
        decoded,
        dtype=torch.float32
    )

    if reconstructed.shape != w.shape:
        raise RuntimeError(
            "E4M11 decoded tensor shape mismatch: "
            f"{reconstructed.shape} vs {w.shape}"
        )

    return reconstructed


# ---------------------------------------------------------------------
# 9. QUANTIZATION DISPATCH
# ---------------------------------------------------------------------

def quantize_weight(w, format_name):

    w = w.detach().float().cpu()

    if format_name == "FP16":

        return quantize_fp(w, torch.float16)

    elif format_name == "BF16":

        return quantize_fp(w, torch.bfloat16)

    elif format_name == "INT8":

        return quantize_symmetric_per_channel(w, 8)

    elif format_name == "INT4":

        return quantize_symmetric_per_tensor(w, 4)

    elif format_name == "FP8_E4M3FN":

        if not hasattr(torch, "float8_e4m3fn"):
            raise RuntimeError("FP8 E4M3FN unavailable")

        return quantize_fp(w, torch.float8_e4m3fn)

    elif format_name == "FP8_E5M2":

        if not hasattr(torch, "float8_e5m2"):
            raise RuntimeError("FP8 E5M2 unavailable")

        return quantize_fp(w, torch.float8_e5m2)

    elif format_name == "E4M11_SUBNORMAL":

        return quantize_e4m11(w)

    else:

        raise ValueError(format_name)


# ---------------------------------------------------------------------
# 10. IDENTIFY ALL QUANTIZABLE LAYERS
# ---------------------------------------------------------------------

layer_inventory = []

for name, module in base_model.named_modules():

    if isinstance(module, (nn.Conv2d, nn.Linear)):

        layer_inventory.append({
            "layer": name,
            "layer_type": type(module).__name__,
            "weight_shape": list(module.weight.shape),
            "parameters": module.weight.numel(),
        })

print("\nLayer inventory:")
display(pd.DataFrame(layer_inventory))

print("\nTotal quantizable layers:", len(layer_inventory))

if len(layer_inventory) != 21:
    print(
        "WARNING: Expected 21 layers based on the current "
        "ResNet-18 configuration."
    )


# ---------------------------------------------------------------------
# 11. FP32 REFERENCE EVALUATION
# ---------------------------------------------------------------------

print("\nEvaluating FP32 reference...")

fp32_reference = evaluate(
    copy.deepcopy(base_model)
)

FP32_ACC = fp32_reference["accuracy"]
FP32_LOSS = fp32_reference["loss"]

print(f"FP32 accuracy: {FP32_ACC:.4f}%")
print(f"FP32 loss:     {FP32_LOSS:.6f}")


# ---------------------------------------------------------------------
# 12. MEASURE WEIGHT QUANTIZATION ERROR
# ---------------------------------------------------------------------

def weight_statistics(original, reconstructed):

    original = original.detach().float().cpu()
    reconstructed = reconstructed.detach().float().cpu()

    error = original - reconstructed

    signal_power = torch.sum(original ** 2).item()
    noise_power = torch.sum(error ** 2).item()

    mse = torch.mean(error ** 2).item()

    if noise_power == 0:
        sqnr = float("inf")
    elif signal_power == 0:
        sqnr = float("-inf")
    else:
        sqnr = 10.0 * math.log10(
            signal_power / noise_power
        )

    zero_percentage = (
        100.0 *
        (reconstructed == 0).sum().item() /
        reconstructed.numel()
    )

    return {
        "sqnr_db": sqnr,
        "weight_mse": mse,
        "zero_weight_percentage": zero_percentage,
    }


# ---------------------------------------------------------------------
# 13. EXPERIMENT CONFIGURATION
# ---------------------------------------------------------------------

config = {
    "experiment": "JFAMP_layerwise_quantization_profiling",
    "model": "ResNet18_CIFAR100",
    "checkpoint": CHECKPOINT_PATH,
    "checkpoint_epoch": checkpoint_epoch,
    "validation_samples": len(val_dataset),
    "validation_split_key": "validation_indices",
    "validation_accuracy_fp32": FP32_ACC,
    "validation_loss_fp32": FP32_LOSS,
    "quantization_scope": "One Conv2d/Linear weight tensor at a time",
    "other_layers": "FP32",
    "activations": "FP32",
    "inference": "FP32",
    "formats": [
        {
            "name": name,
            "nominal_bits": bits
        }
        for name, bits in FORMATS
    ],
    "e4m11": {
        "mode": E4M11_MODE,
        "bias": E4M11_BIAS,
        "all_ones": E4M11_ALL_ONES,
        "implementation": "Exact supplied codec"
    },
    "storage_note": (
        "Theoretical weight storage excludes scale metadata, "
        "packing, alignment and protection overhead."
    )
}

with open(CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=2)


# ---------------------------------------------------------------------
# 14. INCREMENTAL RESULT STORAGE
# ---------------------------------------------------------------------

results = []

completed_keys = set()

def save_results():

    df = pd.DataFrame(results)

    df.to_csv(
        CSV_PATH,
        index=False
    )

    with open(JSON_PATH, "w") as f:
        json.dump(
            results,
            f,
            indent=2,
            allow_nan=True
        )


# ---------------------------------------------------------------------
# 15. MAIN LAYER-WISE EXPERIMENT
# ---------------------------------------------------------------------

total_experiments = len(layer_inventory) * len(FORMATS)

completed = 0
failed = 0

experiment_start = time.time()

print("\n" + "=" * 75)
print("STARTING LAYER-WISE QUANTIZATION EXPERIMENTS")
print("=" * 75)

print("Total planned experiments:", total_experiments)

for layer_info in layer_inventory:

    layer_name = layer_info["layer"]

    # Locate the original layer.
    original_module = dict(
        base_model.named_modules()
    )[layer_name]

    original_weight = (
        original_module.weight.detach()
        .float()
        .cpu()
        .clone()
    )

    for format_name, bits in FORMATS:

        completed += 1

        print(
            f"\n[{completed}/{total_experiments}] "
            f"{layer_name} | {format_name}"
        )

        try:

            # Quantize this layer only.
            reconstructed = quantize_weight(
                original_weight,
                format_name
            )

            # Compute local quantization statistics.
            stats = weight_statistics(
                original_weight,
                reconstructed
            )

            # Start with a fresh copy of the FP32 model.
            test_model = copy.deepcopy(base_model)

            test_module = dict(
                test_model.named_modules()
            )[layer_name]

            test_module.weight.data.copy_(
                reconstructed.to(
                    test_module.weight.dtype
                )
            )

            # Evaluate the complete network.
            metrics = evaluate(test_model)

            parameter_count = original_weight.numel()

            theoretical_bytes = (
                parameter_count * bits / 8.0
            )

            fp32_bytes = parameter_count * 4.0

            storage_reduction = (
                100.0 *
                (1.0 - theoretical_bytes / fp32_bytes)
            )

            row = {

                "layer": layer_name,

                "layer_type": layer_info["layer_type"],

                "weight_shape": str(
                    layer_info["weight_shape"]
                ),

                "parameter_count": parameter_count,

                "format": format_name,

                "nominal_bits": bits,

                "weight_mse": stats["weight_mse"],

                "sqnr_db": stats["sqnr_db"],

                "zero_weight_percentage": (
                    stats["zero_weight_percentage"]
                ),

                "fp32_accuracy": FP32_ACC,

                "quantized_accuracy": metrics["accuracy"],

                "accuracy_change_pp": (
                    metrics["accuracy"] - FP32_ACC
                ),

                "fp32_loss": FP32_LOSS,

                "quantized_loss": metrics["loss"],

                "loss_change": (
                    metrics["loss"] - FP32_LOSS
                ),

                "theoretical_weight_bytes": theoretical_bytes,

                "fp32_weight_bytes": fp32_bytes,

                "storage_reduction_pct": storage_reduction,

                "evaluation_seconds": (
                    metrics["evaluation_seconds"]
                ),

                "status": "success",

                "error": None

            }

            results.append(row)

            print(
                f"Accuracy: {metrics['accuracy']:.4f}% | "
                f"Change: {row['accuracy_change_pp']:+.4f} pp | "
                f"SQNR: {stats['sqnr_db']:.2f} dB | "
                f"MSE: {stats['weight_mse']:.3e}"
            )

            # Free model memory.
            del test_model
            del reconstructed

            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

        except Exception as e:

            failed += 1

            print("EXPERIMENT FAILED:", repr(e))

            results.append({

                "layer": layer_name,

                "layer_type": layer_info["layer_type"],

                "weight_shape": str(
                    layer_info["weight_shape"]
                ),

                "parameter_count": (
                    layer_info["parameters"]
                ),

                "format": format_name,

                "nominal_bits": bits,

                "status": "failed",

                "error": repr(e)

            })

        # Save after EVERY combination.
        save_results()

        del original_weight


# ---------------------------------------------------------------------
# 16. SAVE FINAL LAYER INVENTORY AND SUMMARY
# ---------------------------------------------------------------------

layer_inventory_path = (
    OUTPUT_DIR + "/layer_inventory.csv"
)

pd.DataFrame(layer_inventory).to_csv(
    layer_inventory_path,
    index=False
)

elapsed = time.time() - experiment_start

successful = [
    r for r in results
    if r.get("status") == "success"
]

summary = {

    "experiment": "JFAMP_layerwise_quantization_profiling",

    "total_layers": len(layer_inventory),

    "total_formats": len(FORMATS),

    "planned_experiments": total_experiments,

    "successful_experiments": len(successful),

    "failed_experiments": failed,

    "fp32_accuracy": FP32_ACC,

    "fp32_loss": FP32_LOSS,

    "validation_samples": len(val_dataset),

    "total_runtime_seconds": elapsed,

    "results_csv": CSV_PATH,

    "results_json": JSON_PATH,

    "layer_inventory_csv": layer_inventory_path

}

with open(
    OUTPUT_DIR + "/experiment_summary.json",
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


# ---------------------------------------------------------------------
# 17. FINAL REPORT
# ---------------------------------------------------------------------

df = pd.DataFrame(results)

print("\n" + "=" * 75)
print("JFAMP LAYER-WISE QUANTIZATION PROFILING COMPLETED")
print("=" * 75)

print("Total layers:", len(layer_inventory))
print("Planned experiments:", total_experiments)
print("Successful:", len(successful))
print("Failed:", failed)

print(f"FP32 accuracy: {FP32_ACC:.4f}%")

print(
    "Total runtime:",
    f"{elapsed / 60:.2f} minutes"
)

print("\nResults by format:")

if not df.empty:

    successful_df = df[
        df["status"] == "success"
    ]

    display(
        successful_df.groupby("format").agg(
            layers_measured=("layer", "count"),
            mean_accuracy=("quantized_accuracy", "mean"),
            mean_accuracy_change=("accuracy_change_pp", "mean"),
            mean_sqnr=("sqnr_db", "mean"),
            mean_weight_mse=("weight_mse", "mean"),
            mean_zero_percentage=("zero_weight_percentage", "mean"),
            mean_storage_reduction=("storage_reduction_pct", "mean")
        ).round(5)
    )

print("\nSaved files:")

print(CSV_PATH)
print(JSON_PATH)
print(CONFIG_PATH)
print(layer_inventory_path)
print(OUTPUT_DIR + "/experiment_summary.json")

print("\nIMPORTANT:")
print("1. Only one layer was quantized in each experiment.")
print("2. All other weights remained FP32.")
print("3. Activations and inference arithmetic remained FP32.")
print("4. Accuracy changes are relative to the FP32 reference.")
print("5. Storage figures exclude scale metadata and packing overhead.")
print("6. These are isolated-layer measurements, not a final mixed-precision model.")
print("7. No retraining or fault injection was performed.")

print("\nUse Kaggle Quick Save. Do not use Save and Run All.")


JFAMP — LAYER-WISE QUANTIZATION PROFILING
Device: cuda
GPU: Tesla T4


FileNotFoundError: /kaggle/working/JFAMP/retrain_to150/checkpoints/resnet18_cifar100_best_150.pth

In [3]:
import os

ROOT = "/kaggle/working/JFAMP"

print("JFAMP exists:", os.path.exists(ROOT))

if os.path.exists(ROOT):
    print("\nContents of JFAMP:")
    print(os.listdir(ROOT))

    print("\nSearching for checkpoint files:")
    for current_dir, dirs, files in os.walk(ROOT):
        for filename in files:
            if filename.endswith((".pth", ".pt", ".ckpt")):
                print(os.path.join(current_dir, filename))

JFAMP exists: True

Contents of JFAMP:
['experiments']

Searching for checkpoint files:


In [4]:
import os

SEARCH_ROOTS = [
    "/kaggle/input",
    "/kaggle/working",
]

for root in SEARCH_ROOTS:
    print(f"\n{'='*60}")
    print("Searching:", root)

    for current_dir, dirs, files in os.walk(root):
        for filename in files:
            if filename.endswith((".pth", ".pt", ".ckpt")):
                path = os.path.join(current_dir, filename)
                print(path)


Searching: /kaggle/input
/kaggle/input/datasets/crahulnaidu/jfamp-resnet18-cifar100-checkpoint/resnet18_cifar100_best.pth
/kaggle/input/datasets/crahulnaidu/jfamp-resnet18-cifar100-checkpoint/resnet18_cifar100_last.pth

Searching: /kaggle/working


In [6]:
import os

for current_dir, dirs, files in os.walk("/kaggle/input"):
    for filename in files:
        if any(x in filename.lower() for x in [
            "best_150",
            "last_150",
            "training_history",
            "fp32_baseline",
        ]):
            print(os.path.join(current_dir, filename))

In [7]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for filename in files:
        if any(key in filename.lower() for key in [
            "best_150",
            "last_150",
            "training_history",
            "fp32_baseline",
        ]):
            print(os.path.join(root, filename))

/kaggle/input/notebooks/crahulnaidu/notebook08ddb3dde9/JFAMP/retrain_to150/fp32_baseline.json
/kaggle/input/notebooks/crahulnaidu/notebook08ddb3dde9/JFAMP/retrain_to150/training_history.csv
/kaggle/input/notebooks/crahulnaidu/notebook08ddb3dde9/JFAMP/retrain_to150/checkpoints/resnet18_cifar100_best_150.pth
/kaggle/input/notebooks/crahulnaidu/notebook08ddb3dde9/JFAMP/retrain_to150/checkpoints/resnet18_cifar100_last_150.pth


In [8]:
import os
import shutil

SOURCE = "/kaggle/input/notebooks/crahulnaidu/notebook08ddb3dde9/JFAMP"
DEST = "/kaggle/working/JFAMP"

os.makedirs(DEST, exist_ok=True)

# Restore retraining outputs
shutil.copytree(
    os.path.join(SOURCE, "retrain_to150"),
    os.path.join(DEST, "retrain_to150"),
    dirs_exist_ok=True
)

# Restore other available experiment folders
for folder in ["codec", "data", "experiments"]:
    src = os.path.join(SOURCE, folder)
    dst = os.path.join(DEST, folder)

    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("Restored:", folder)
    else:
        print("Not found in saved output:", folder)

# Verify checkpoint and baseline
required = [
    "retrain_to150/checkpoints/resnet18_cifar100_best_150.pth",
    "retrain_to150/checkpoints/resnet18_cifar100_last_150.pth",
    "retrain_to150/fp32_baseline.json",
]

print("\nVERIFICATION")
for relative_path in required:
    path = os.path.join(DEST, relative_path)
    print(
        "FOUND" if os.path.isfile(path) else "MISSING",
        relative_path,
        f"({os.path.getsize(path)/1024**2:.2f} MB)" if os.path.isfile(path) else ""
    )

Restored: codec
Restored: data
Restored: experiments

VERIFICATION
FOUND retrain_to150/checkpoints/resnet18_cifar100_best_150.pth (85.70 MB)
FOUND retrain_to150/checkpoints/resnet18_cifar100_last_150.pth (85.70 MB)
FOUND retrain_to150/fp32_baseline.json (0.00 MB)


In [9]:
import os
import json
import torch

ROOT = "/kaggle/working/JFAMP"

CKPT_PATH = (
    f"{ROOT}/retrain_to150/checkpoints/"
    "resnet18_cifar100_best_150.pth"
)
BASELINE_PATH = f"{ROOT}/retrain_to150/fp32_baseline.json"

print("Checkpoint exists:", os.path.isfile(CKPT_PATH))
print("Baseline exists:", os.path.isfile(BASELINE_PATH))

checkpoint = torch.load(
    CKPT_PATH,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint keys:", list(checkpoint.keys()))

with open(BASELINE_PATH) as f:
    baseline = json.load(f)

print("\nFP32 baseline:")
print(json.dumps(baseline, indent=2))

print("\nCodec files:")
print(os.listdir(f"{ROOT}/codec"))

print("\nExisting experiment folders:")
print(os.listdir(f"{ROOT}/experiments"))

Checkpoint exists: True
Baseline exists: True

Checkpoint keys: ['epoch', 'best_epoch', 'train_accuracy', 'train_loss', 'val_acc', 'val_loss', 'best_val_acc', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'scaler_state_dict', 'config']

FP32 baseline:
{
  "model": "CIFAR-style ResNet-18",
  "dataset": "CIFAR-100",
  "checkpoint_epoch": 139,
  "validation_samples": 5000,
  "precision": "FP32",
  "validation_accuracy": 77.78,
  "validation_loss": 0.9143250974655152,
  "stored_checkpoint_accuracy": 77.82
}

Codec files:
['e4m11_codec.py', '__pycache__']

Existing experiment folders:
['e4m11_subnormal', 'precision_comparison', 'layerwise_quantization']


In [10]:
# ================================================================
# JFAMP: LAYER-WISE QUANTIZATION PROFILING
# ResNet-18 / CIFAR-100 / Original 5,000-sample validation split
#
# Formats:
# FP16, BF16, INT8, INT4, FP8 E4M3FN, FP8 E5M2, E4M11 subnormal
#
# One layer is quantized at a time. All remaining layers stay FP32.
# Results are saved after every successful layer-format evaluation.
# ================================================================

import os
import sys
import json
import time
import copy
import inspect
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset

warnings.filterwarnings("ignore")

# ------------------------------------------------
# 1. Paths and configuration
# ------------------------------------------------

ROOT = "/kaggle/working/JFAMP"

CKPT_PATH = (
    f"{ROOT}/retrain_to150/checkpoints/"
    "resnet18_cifar100_best_150.pth"
)

BASELINE_PATH = f"{ROOT}/retrain_to150/fp32_baseline.json"
SPLIT_PATH = (
    "/kaggle/input/datasets/crahulnaidu/"
    "jfamp-resnet18-cifar100-checkpoint/data_split_indices.json"
)

CODEC_DIR = f"{ROOT}/codec"
OUTPUT_DIR = f"{ROOT}/experiments/layerwise_quantization"

os.makedirs(OUTPUT_DIR, exist_ok=True)

CSV_PATH = f"{OUTPUT_DIR}/layerwise_quantization.csv"
JSON_PATH = f"{OUTPUT_DIR}/layerwise_quantization.json"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 256
NUM_WORKERS = 2

E4M11_MODE = "subnormal"
E4M11_BIAS = 7
E4M11_ALL_ONES = "clamp"

FORMATS = [
    ("FP16", 16),
    ("BF16", 16),
    ("INT8", 8),
    ("INT4", 4),
    ("FP8_E4M3FN", 8),
    ("FP8_E5M2", 8),
    ("E4M11_SUBNORMAL", 16),
]

print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------
# 2. Validate required files
# ------------------------------------------------

for path in [CKPT_PATH, BASELINE_PATH, SPLIT_PATH]:
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Required file not found: {path}")

# ------------------------------------------------
# 3. Load the exact teammate E4M11 codec
# ------------------------------------------------

sys.path.insert(0, CODEC_DIR)

import e4m11_codec as codec

if not hasattr(codec, "encode_tensor"):
    raise AttributeError("Codec is missing encode_tensor().")

if not hasattr(codec, "decode_to_tensor"):
    raise AttributeError("Codec is missing decode_to_tensor().")

encode_fn = codec.encode_tensor
decode_fn = codec.decode_to_tensor

print("\nE4M11 codec:")
print("encode_tensor:", inspect.signature(encode_fn))
print("decode_to_tensor:", inspect.signature(decode_fn))

def accepted_kwargs(fn, requested):
    """Pass only supported keyword arguments, including **kwargs APIs."""
    sig = inspect.signature(fn)
    params = sig.parameters

    accepts_extra = any(
        p.kind == inspect.Parameter.VAR_KEYWORD
        for p in params.values()
    )

    return {
        key: value
        for key, value in requested.items()
        if key in params or accepts_extra
    }

def codec_quantize_tensor(weight):
    """
    Use the supplied codec without changing its implementation.
    Return the decoded/dequantized FP32 weight tensor.
    """
    original_shape = tuple(weight.shape)
    original_device = weight.device

    source = weight.detach().to("cpu", dtype=torch.float32).contiguous()

    requested = {
        "mode": E4M11_MODE,
        "bias": E4M11_BIAS,
        "all_ones": E4M11_ALL_ONES,
    }

    encode_kwargs = accepted_kwargs(encode_fn, requested)

    # Do not silently use a different E4M11 mode.
    encode_signature = inspect.signature(encode_fn)
    encode_params = encode_signature.parameters
    encode_has_kwargs = any(
        p.kind == inspect.Parameter.VAR_KEYWORD
        for p in encode_params.values()
    )

    if "mode" not in encode_params and not encode_has_kwargs:
        raise RuntimeError(
            "encode_tensor does not expose a mode argument. "
            "Cannot guarantee the requested subnormal mode."
        )

    encoded = encode_fn(source, **encode_kwargs)

    decode_kwargs = accepted_kwargs(decode_fn, requested)
    decoded = decode_fn(encoded, **decode_kwargs)

    if isinstance(decoded, tuple):
        decoded = decoded[0]

    if isinstance(decoded, np.ndarray):
        decoded = torch.from_numpy(decoded)

    if not torch.is_tensor(decoded):
        decoded = torch.as_tensor(decoded)

    decoded = decoded.to(dtype=torch.float32).reshape(original_shape)

    if not torch.isfinite(decoded).all():
        raise RuntimeError("E4M11 decoder returned non-finite values.")

    return decoded.to(original_device)

# Verify codec on a small tensor before the full experiment.
codec_test = torch.tensor(
    [0.0, 0.125, 0.5, 1.0, -0.25, 2.0],
    dtype=torch.float32
)

codec_test_decoded = codec_quantize_tensor(codec_test)

print("\nE4M11 test input: ", codec_test.tolist())
print("E4M11 decoded:   ", codec_test_decoded.tolist())

# ------------------------------------------------
# 4. Locate the CIFAR-100 dataset
# ------------------------------------------------

candidate_roots = [
    f"{ROOT}/data",
    f"{ROOT}/cifar_100",
    f"{ROOT}/cifar_100data",
    "/kaggle/input/datasets/crahulnaidu/"
    "jfamp-resnet18-cifar100-checkpoint",
]

DATA_ROOT = None

for candidate in candidate_roots:
    expected = os.path.join(candidate, "cifar-100-python")

    if os.path.isfile(os.path.join(expected, "train")):
        DATA_ROOT = candidate
        break

# Search restored JFAMP folders and Kaggle input if needed.
if DATA_ROOT is None:
    search_roots = [ROOT, "/kaggle/input"]

    for search_root in search_roots:
        for current_dir, dirs, files in os.walk(search_root):
            if (
                "train" in files
                and os.path.basename(current_dir) == "cifar-100-python"
            ):
                DATA_ROOT = os.path.dirname(current_dir)
                break

        if DATA_ROOT is not None:
            break

if DATA_ROOT is None:
    raise FileNotFoundError(
        "Could not locate cifar-100-python/train. "
        "Check the restored CIFAR-100 data directory."
    )

print("\nCIFAR-100 root:", DATA_ROOT)

# ------------------------------------------------
# 5. Load the original validation split
# ------------------------------------------------

with open(SPLIT_PATH, "r") as f:
    split = json.load(f)

if "validation_indices" not in split:
    raise KeyError(
        "Expected 'validation_indices' in the original split JSON."
    )

validation_indices = split["validation_indices"]

if len(validation_indices) != 5000:
    raise ValueError(
        f"Expected 5000 validation samples, got {len(validation_indices)}"
    )

mean = (0.5071, 0.4867, 0.4408)
std = (0.2675, 0.2565, 0.2761)

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

full_dataset = datasets.CIFAR100(
    root=DATA_ROOT,
    train=True,
    download=False,
    transform=eval_transform,
)

validation_dataset = Subset(
    full_dataset,
    validation_indices
)

if len(validation_dataset) != 5000:
    raise RuntimeError("Validation subset size mismatch.")

loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

print("Validation samples:", len(validation_dataset))

# ------------------------------------------------
# 6. Reconstruct the exact CIFAR-style ResNet-18
# ------------------------------------------------

model = models.resnet18(weights=None, num_classes=100)
model.conv1 = nn.Conv2d(
    3, 64, kernel_size=3, stride=1, padding=1, bias=False
)
model.maxpool = nn.Identity()

checkpoint = torch.load(
    CKPT_PATH,
    map_location="cpu",
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model = model.float().cpu().eval()

print("\nCheckpoint epoch:", checkpoint.get("epoch"))
print("Best epoch:", checkpoint.get("best_epoch"))
print("Stored best validation accuracy:", checkpoint.get("best_val_acc"))

# ------------------------------------------------
# 7. Evaluation function
# ------------------------------------------------

criterion = nn.CrossEntropyLoss()

def evaluate(model_to_evaluate):
    model_to_evaluate = model_to_evaluate.to(DEVICE).eval()

    total = 0
    correct = 0
    total_loss = 0.0

    start = time.time()

    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(
                DEVICE,
                non_blocking=True
            )
            labels = labels.to(
                DEVICE,
                non_blocking=True
            )

            outputs = model_to_evaluate(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * labels.size(0)
            correct += (
                outputs.argmax(dim=1) == labels
            ).sum().item()

            total += labels.size(0)

    elapsed = time.time() - start

    accuracy = 100.0 * correct / total
    average_loss = total_loss / total

    model_to_evaluate.cpu()

    return {
        "accuracy": accuracy,
        "loss": average_loss,
        "seconds": elapsed,
    }

# ------------------------------------------------
# 8. Load and verify FP32 baseline
# ------------------------------------------------

with open(BASELINE_PATH, "r") as f:
    baseline_json = json.load(f)

stored_baseline = float(
    baseline_json["validation_accuracy"]
)

print("\nStored FP32 baseline:", stored_baseline)

baseline_result = evaluate(model)

FP32_ACCURACY = baseline_result["accuracy"]
FP32_LOSS = baseline_result["loss"]

print("Re-evaluated FP32 accuracy:", FP32_ACCURACY)
print("Re-evaluated FP32 loss:", FP32_LOSS)

if abs(FP32_ACCURACY - stored_baseline) > 0.05:
    raise RuntimeError(
        "FP32 baseline mismatch. Stop before quantization."
    )

print("FP32 baseline verification passed.")

# ------------------------------------------------
# 9. Quantization implementations
# ------------------------------------------------

def quantize_weight(weight, format_name):
    """
    Simulated weight quantization.
    Returns a reconstructed FP32 tensor.
    """

    w = weight.detach().to(
        device="cpu",
        dtype=torch.float32
    )

    if format_name == "FP16":
        return w.to(torch.float16).to(torch.float32)

    if format_name == "BF16":
        return w.to(torch.bfloat16).to(torch.float32)

    if format_name == "INT8":
        # Symmetric per-output-channel quantization.
        flat = w.reshape(w.shape[0], -1)
        max_abs = flat.abs().amax(dim=1).clamp(min=1e-12)

        scale = max_abs / 127.0
        scale = scale.reshape(
            [w.shape[0]] + [1] * (w.ndim - 1)
        )

        q = torch.round(w / scale).clamp(-127, 127)
        return (q * scale).to(torch.float32)

    if format_name == "INT4":
        # Symmetric per-tensor quantization.
        max_abs = w.abs().max().clamp(min=1e-12)
        scale = max_abs / 7.0

        q = torch.round(w / scale).clamp(-7, 7)
        return (q * scale).to(torch.float32)

    if format_name == "FP8_E4M3FN":
        if not hasattr(torch, "float8_e4m3fn"):
            raise RuntimeError("This PyTorch build lacks float8_e4m3fn.")
        return w.to(torch.float8_e4m3fn).to(torch.float32)

    if format_name == "FP8_E5M2":
        if not hasattr(torch, "float8_e5m2"):
            raise RuntimeError("This PyTorch build lacks float8_e5m2.")
        return w.to(torch.float8_e5m2).to(torch.float32)

    if format_name == "E4M11_SUBNORMAL":
        return codec_quantize_tensor(w)

    raise ValueError(f"Unknown format: {format_name}")

# ------------------------------------------------
# 10. Layer inventory
# ------------------------------------------------

layers = []

for name, module in model.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        layers.append({
            "name": name,
            "type": module.__class__.__name__,
            "shape": list(module.weight.shape),
            "num_weights": module.weight.numel(),
        })

if not layers:
    raise RuntimeError("No Conv2d or Linear layers found.")

inventory_df = pd.DataFrame(layers)
inventory_path = f"{OUTPUT_DIR}/layer_inventory.csv"
inventory_df.to_csv(inventory_path, index=False)

print("\nQuantizable layers:", len(layers))
print(inventory_df[["name", "type", "num_weights"]].to_string(index=False))

# ------------------------------------------------
# 11. Resume support
# ------------------------------------------------

if os.path.isfile(CSV_PATH):
    previous_df = pd.read_csv(CSV_PATH)

    if "status" in previous_df.columns:
        previous_df = previous_df[
            previous_df["status"] == "success"
        ]

    rows = previous_df.to_dict("records")

    completed = {
        (str(row["layer"]), str(row["format"]))
        for row in rows
    }

    print("\nExisting successful combinations:", len(completed))
else:
    rows = []
    completed = set()

def save_results():
    df = pd.DataFrame(rows)
    df.to_csv(CSV_PATH, index=False)

    with open(JSON_PATH, "w") as f:
        json.dump(
            df.to_dict("records"),
            f,
            indent=2,
            default=str
        )

# ------------------------------------------------
# 12. Run isolated layer-wise experiments
# ------------------------------------------------

total_experiments = len(layers) * len(FORMATS)
run_count = 0

print("\nExpected total combinations:", total_experiments)
print("Already completed:", len(completed))

for layer_index, layer_info in enumerate(layers, start=1):

    layer_name = layer_info["name"]

    # Fetch original layer weights from the untouched FP32 model.
    original_module = dict(model.named_modules())[layer_name]
    original_weight = (
        original_module.weight.detach().cpu().clone()
    )

    print(
        f"\n[{layer_index}/{len(layers)}] "
        f"Layer: {layer_name} "
        f"({layer_info['num_weights']:,} weights)"
    )

    for format_name, bits in FORMATS:

        key = (layer_name, format_name)

        if key in completed:
            print("  SKIP:", format_name, "(already completed)")
            continue

        run_count += 1
        print(f"  Running {format_name} ...", end=" ", flush=True)

        start_time = time.time()

        try:
            quantized_weight = quantize_weight(
                original_weight,
                format_name
            )

            if quantized_weight.shape != original_weight.shape:
                raise RuntimeError("Quantized weight shape mismatch.")

            if not torch.isfinite(quantized_weight).all():
                raise RuntimeError("Quantized weight contains non-finite values.")

            difference = (
                original_weight - quantized_weight
            )

            mse = torch.mean(difference ** 2).item()

            signal_power = torch.mean(
                original_weight ** 2
            ).item()

            if mse == 0:
                sqnr = float("inf")
            elif signal_power == 0:
                sqnr = float("-inf")
            else:
                sqnr = 10.0 * np.log10(signal_power / mse)

            zero_percentage = (
                100.0
                * (quantized_weight == 0).sum().item()
                / quantized_weight.numel()
            )

            # Clone the original FP32 model.
            test_model = copy.deepcopy(model)

            test_module = dict(
                test_model.named_modules()
            )[layer_name]

            # Only this layer's weight is replaced.
            with torch.no_grad():
                test_module.weight.copy_(quantized_weight)

            result = evaluate(test_model)

            fp32_bytes = original_weight.numel() * 4
            theoretical_bytes = (
                original_weight.numel() * bits / 8.0
            )

            elapsed = time.time() - start_time

            row = {
                "layer": layer_name,
                "layer_type": layer_info["type"],
                "weight_shape": str(layer_info["shape"]),
                "num_weights": layer_info["num_weights"],
                "format": format_name,
                "bits_per_weight": bits,

                "mse": mse,
                "sqnr_db": sqnr,
                "zero_weight_percentage": zero_percentage,

                "fp32_accuracy": FP32_ACCURACY,
                "quantized_accuracy": result["accuracy"],
                "accuracy_delta_pp": (
                    result["accuracy"] - FP32_ACCURACY
                ),

                "fp32_loss": FP32_LOSS,
                "quantized_loss": result["loss"],
                "loss_delta": result["loss"] - FP32_LOSS,

                "fp32_weight_bytes": fp32_bytes,
                "theoretical_weight_bytes": theoretical_bytes,
                "theoretical_storage_reduction_percentage": (
                    100.0 * (1.0 - theoretical_bytes / fp32_bytes)
                ),

                "evaluation_seconds": result["seconds"],
                "total_experiment_seconds": elapsed,
                "status": "success",
                "error": "",
            }

            rows.append(row)
            completed.add(key)

            print(
                f"Accuracy={result['accuracy']:.2f}% | "
                f"Delta={row['accuracy_delta_pp']:+.2f} pp | "
                f"SQNR={sqnr:.2f} dB"
            )

        except Exception as exc:

            elapsed = time.time() - start_time

            row = {
                "layer": layer_name,
                "layer_type": layer_info["type"],
                "weight_shape": str(layer_info["shape"]),
                "num_weights": layer_info["num_weights"],
                "format": format_name,
                "bits_per_weight": bits,
                "status": "error",
                "error": repr(exc),
                "total_experiment_seconds": elapsed,
            }

            rows.append(row)

            print("FAILED:", repr(exc))

        # Save after every combination, including errors.
        save_results()

        # Release temporary model and tensors.
        if "test_model" in locals():
            del test_model

        if "quantized_weight" in locals():
            del quantized_weight

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

# ------------------------------------------------
# 13. Final summary
# ------------------------------------------------

final_df = pd.DataFrame(rows)
final_df.to_csv(CSV_PATH, index=False)

with open(JSON_PATH, "w") as f:
    json.dump(
        final_df.to_dict("records"),
        f,
        indent=2,
        default=str
    )

successful = (
    final_df[final_df["status"] == "success"]
    if "status" in final_df.columns
    else pd.DataFrame()
)

failed = (
    final_df[final_df["status"] == "error"]
    if "status" in final_df.columns
    else pd.DataFrame()
)

summary = {
    "model": "CIFAR-style ResNet-18",
    "dataset": "CIFAR-100",
    "checkpoint_epoch": checkpoint.get("epoch"),
    "best_epoch": checkpoint.get("best_epoch"),
    "validation_samples": len(validation_dataset),
    "fp32_accuracy": FP32_ACCURACY,
    "fp32_loss": FP32_LOSS,
    "quantizable_layers": len(layers),
    "formats": [x[0] for x in FORMATS],
    "expected_combinations": total_experiments,
    "successful_combinations": len(successful),
    "failed_combinations": len(failed),
    "experiment_scope": "one layer quantized at a time",
    "activations_batchnorm_biases": "FP32",
    "e4m11_mode": E4M11_MODE,
}

with open(f"{OUTPUT_DIR}/experiment_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 65)
print("LAYER-WISE QUANTIZATION PROFILING FINISHED")
print("=" * 65)

print(json.dumps(summary, indent=2))

if len(successful) > 0:
    print("\nMean results by precision format:")
    display(
        successful.groupby("format").agg(
            mean_accuracy=("quantized_accuracy", "mean"),
            mean_accuracy_delta_pp=("accuracy_delta_pp", "mean"),
            mean_sqnr_db=("sqnr_db", "mean"),
            mean_zero_weight_percentage=("zero_weight_percentage", "mean"),
            evaluated_layers=("layer", "count"),
        ).round(4)
    )

if len(failed) > 0:
    print("\nFailed combinations:")
    display(failed[["layer", "format", "error"]])

print("\nSaved files:")
print(CSV_PATH)
print(JSON_PATH)
print(f"{OUTPUT_DIR}/layer_inventory.csv")
print(f"{OUTPUT_DIR}/experiment_summary.json")

Device: cuda
GPU: Tesla T4

E4M11 codec:
encode_tensor: (t, mode: 'str' = 'paper', bias: 'int' = 7) -> 'np.ndarray'
decode_to_tensor: (words, dtype=None, device=None, mode: 'str' = 'paper', all_ones: 'str' = 'clamp', bias: 'int' = 7)

E4M11 test input:  [0.0, 0.125, 0.5, 1.0, -0.25, 2.0]
E4M11 decoded:    [0.0, 0.125, 0.5, 1.0, -0.25, 2.0]

CIFAR-100 root: /kaggle/working/JFAMP/data
Validation samples: 5000

Checkpoint epoch: 139
Best epoch: 139
Stored best validation accuracy: 77.82

Stored FP32 baseline: 77.78
Re-evaluated FP32 accuracy: 77.78
Re-evaluated FP32 loss: 0.9143250495910644
FP32 baseline verification passed.

Quantizable layers: 21
                 name   type  num_weights
                conv1 Conv2d         1728
       layer1.0.conv1 Conv2d        36864
       layer1.0.conv2 Conv2d        36864
       layer1.1.conv1 Conv2d        36864
       layer1.1.conv2 Conv2d        36864
       layer2.0.conv1 Conv2d        73728
       layer2.0.conv2 Conv2d       147456
layer2.0.d

,mean_accuracy,mean_accuracy_delta_pp,mean_sqnr_db,mean_zero_weight_percentage,evaluated_layers
format,,,,,
BF16,77.7867,0.0067,55.6313,0.0000,21
E4M11_SUBNORMAL,77.7800,0.0000,74.2754,2.1573,21
FP16,77.7810,0.0010,73.6969,0.0942,21
FP8_E4M3FN,77.7524,-0.0276,26.1738,11.9441,21
FP8_E5M2,77.7505,-0.0295,25.5409,2.4272,21
INT4,77.3895,-0.3905,7.5037,64.9887,21
INT8,77.7724,-0.0076,39.5538,3.5323,21



Saved files:
/kaggle/working/JFAMP/experiments/layerwise_quantization/layerwise_quantization.csv
/kaggle/working/JFAMP/experiments/layerwise_quantization/layerwise_quantization.json
/kaggle/working/JFAMP/experiments/layerwise_quantization/layer_inventory.csv
/kaggle/working/JFAMP/experiments/layerwise_quantization/experiment_summary.json


In [11]:
import os
import pandas as pd
import numpy as np

ROOT = "/kaggle/working/JFAMP"
CSV_PATH = (
    f"{ROOT}/experiments/layerwise_quantization/"
    "layerwise_quantization.csv"
)

df = pd.read_csv(CSV_PATH)
df = df[df["status"] == "success"].copy()

# ------------------------------------------------
# 1. Verify experiment completeness
# ------------------------------------------------

print("=" * 70)
print("JFAMP LAYER-WISE QUANTIZATION SENSITIVITY ANALYSIS")
print("=" * 70)

print("Successful experiments:", len(df))
print("Unique layers:", df["layer"].nunique())
print("Precision formats:", df["format"].nunique())

# ------------------------------------------------
# 2. Rank layers by accuracy sensitivity for each format
# ------------------------------------------------

formats = [
    "INT4",
    "FP8_E4M3FN",
    "FP8_E5M2",
    "INT8",
    "BF16",
    "FP16",
    "E4M11_SUBNORMAL",
]

for fmt in formats:
    subset = df[df["format"] == fmt].copy()

    subset = subset.sort_values(
        "accuracy_delta_pp",
        ascending=True
    )

    print("\n" + "=" * 70)
    print(f"FORMAT: {fmt}")
    print("=" * 70)

    print("\nMost sensitive layers (largest accuracy degradation):")
    display(
        subset[
            [
                "layer",
                "accuracy_delta_pp",
                "quantized_accuracy",
                "sqnr_db",
                "zero_weight_percentage",
            ]
        ].head(5).round(4)
    )

    print("\nMost tolerant layers:")
    display(
        subset[
            [
                "layer",
                "accuracy_delta_pp",
                "quantized_accuracy",
                "sqnr_db",
                "zero_weight_percentage",
            ]
        ].tail(5).sort_values(
            "accuracy_delta_pp",
            ascending=False
        ).round(4)
    )

# ------------------------------------------------
# 3. Build a layer x precision sensitivity matrix
# ------------------------------------------------

accuracy_matrix = df.pivot(
    index="layer",
    columns="format",
    values="accuracy_delta_pp"
)

sqnr_matrix = df.pivot(
    index="layer",
    columns="format",
    values="sqnr_db"
)

zero_matrix = df.pivot(
    index="layer",
    columns="format",
    values="zero_weight_percentage"
)

print("\n" + "=" * 70)
print("LAYER-WISE ACCURACY SENSITIVITY MATRIX (pp)")
print("=" * 70)

display(accuracy_matrix[formats].round(3))

print("\n" + "=" * 70)
print("LAYER-WISE SQNR MATRIX (dB)")
print("=" * 70)

display(sqnr_matrix[formats].round(2))

# ------------------------------------------------
# 4. Count layers satisfying accuracy-loss thresholds
# ------------------------------------------------

print("\n" + "=" * 70)
print("LAYERS WITHIN ACCURACY-LOSS THRESHOLDS")
print("=" * 70)

thresholds = [0.0, 0.1, 0.25, 0.5, 1.0]

threshold_rows = []

for fmt in formats:
    subset = df[df["format"] == fmt]

    row = {"format": fmt}

    # Accuracy degradation is negative delta.
    degradation = -subset["accuracy_delta_pp"]

    for threshold in thresholds:
        row[f"loss_le_{threshold}_pp"] = int(
            (degradation <= threshold).sum()
        )

    threshold_rows.append(row)

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df)

# ------------------------------------------------
# 5. Identify layers that are candidates for INT4
# ------------------------------------------------

int4 = df[df["format"] == "INT4"].copy()

int4["accuracy_loss_pp"] = -int4["accuracy_delta_pp"]

print("\n" + "=" * 70)
print("INT4 CANDIDATE LAYERS")
print("=" * 70)

for threshold in [0.1, 0.25, 0.5, 1.0]:
    candidates = int4[
        int4["accuracy_loss_pp"] <= threshold
    ]

    print(
        f"Accuracy loss <= {threshold:.2f} pp: "
        f"{len(candidates)}/{len(int4)} layers"
    )

    print(candidates["layer"].tolist())

# ------------------------------------------------
# 6. Export analysis tables
# ------------------------------------------------

OUT = f"{ROOT}/experiments/layerwise_quantization"

accuracy_matrix.to_csv(
    f"{OUT}/layer_accuracy_sensitivity_matrix.csv"
)

sqnr_matrix.to_csv(
    f"{OUT}/layer_sqnr_matrix.csv"
)

zero_matrix.to_csv(
    f"{OUT}/layer_zero_weight_matrix.csv"
)

threshold_df.to_csv(
    f"{OUT}/accuracy_threshold_summary.csv",
    index=False
)

print("\nAnalysis tables saved successfully.")

JFAMP LAYER-WISE QUANTIZATION SENSITIVITY ANALYSIS
Successful experiments: 147
Unique layers: 21
Precision formats: 7

FORMAT: INT4

Most sensitive layers (largest accuracy degradation):


,layer,accuracy_delta_pp,quantized_accuracy,sqnr_db,zero_weight_percentage
73,layer3.0.conv1,-1.10,76.68,4.9488,70.6061
10,layer1.0.conv1,-1.06,76.72,6.3504,89.3256
52,layer2.0.downsample.0,-1.04,76.74,6.7013,78.1250
24,layer1.1.conv1,-0.52,77.26,6.1197,76.8717
94,layer3.1.conv1,-0.48,77.30,6.2042,63.2902



Most tolerant layers:


,layer,accuracy_delta_pp,quantized_accuracy,sqnr_db,zero_weight_percentage
122,layer4.0.downsample.0,0.08,77.86,5.9202,64.4493
136,layer4.1.conv2,-0.02,77.76,7.2835,59.5336
115,layer4.0.conv2,-0.08,77.70,9.2543,46.7114
108,layer4.0.conv1,-0.08,77.70,7.8630,53.4281
129,layer4.1.conv1,-0.18,77.60,9.9909,42.7637



FORMAT: FP8_E4M3FN

Most sensitive layers (largest accuracy degradation):


,layer,accuracy_delta_pp,quantized_accuracy,sqnr_db,zero_weight_percentage
53,layer2.0.downsample.0,-0.24,77.54,30.3738,5.0781
4,conv1,-0.18,77.60,31.0158,34.8958
95,layer3.1.conv1,-0.10,77.68,23.0106,10.4585
74,layer3.0.conv1,-0.10,77.68,26.1299,7.1988
60,layer2.1.conv1,-0.06,77.72,25.7835,8.5497


... (25 outputs left out)

format,INT4,FP8_E4M3FN,FP8_E5M2,INT8,BF16,FP16,E4M11_SUBNORMAL
layer,,,,,,,
conv1,13.72,31.02,25.43,45.33,55.84,74.45,79.10
fc,7.47,31.55,25.45,35.92,55.70,73.64,79.53
layer1.0.conv1,6.35,28.24,25.63,37.44,55.65,73.73,75.67
layer1.0.conv2,7.01,27.55,25.53,38.32,55.63,73.73,75.62
layer1.1.conv1,6.12,27.13,25.51,38.35,55.56,73.55,75.21
layer1.1.conv2,8.42,26.85,25.44,39.63,55.66,73.73,74.95
layer2.0.conv1,8.34,27.48,25.50,39.68,55.59,73.64,75.60
layer2.0.conv2,8.40,26.48,25.58,39.68,55.64,73.72,74.63
layer2.0.downsample.0,6.70,30.37,25.53,40.87,55.70,73.65,78.62



LAYERS WITHIN ACCURACY-LOSS THRESHOLDS


,format,loss_le_0.0_pp,loss_le_0.1_pp,loss_le_0.25_pp,loss_le_0.5_pp,loss_le_1.0_pp
0,INT4,1,4,7,17,18
1,FP8_E4M3FN,9,19,21,21,21
2,FP8_E5M2,9,19,21,21,21
3,INT8,8,21,21,21,21
4,BF16,20,21,21,21,21
5,FP16,21,21,21,21,21
6,E4M11_SUBNORMAL,21,21,21,21,21



INT4 CANDIDATE LAYERS
Accuracy loss <= 0.10 pp: 4/21 layers
['layer4.0.conv1', 'layer4.0.conv2', 'layer4.0.downsample.0', 'layer4.1.conv2']
Accuracy loss <= 0.25 pp: 7/21 layers
['layer2.1.conv2', 'layer4.0.conv1', 'layer4.0.conv2', 'layer4.0.downsample.0', 'layer4.1.conv1', 'layer4.1.conv2', 'fc']
Accuracy loss <= 0.50 pp: 17/21 layers
['conv1', 'layer1.0.conv2', 'layer1.1.conv2', 'layer2.0.conv1', 'layer2.0.conv2', 'layer2.1.conv1', 'layer2.1.conv2', 'layer3.0.conv2', 'layer3.0.downsample.0', 'layer3.1.conv1', 'layer3.1.conv2', 'layer4.0.conv1', 'layer4.0.conv2', 'layer4.0.downsample.0', 'layer4.1.conv1', 'layer4.1.conv2', 'fc']
Accuracy loss <= 1.00 pp: 18/21 layers
['conv1', 'layer1.0.conv2', 'layer1.1.conv1', 'layer1.1.conv2', 'layer2.0.conv1', 'layer2.0.conv2', 'layer2.1.conv1', 'layer2.1.conv2', 'layer3.0.conv2', 'layer3.0.downsample.0', 'layer3.1.conv1', 'layer3.1.conv2', 'layer4.0.conv1', 'layer4.0.conv2', 'layer4.0.downsample.0', 'layer4.1.conv1', 'layer4.1.conv2', 'fc']

An

In [ ]:

# ================================================================
# JFAMP EXPERIMENT 5
# Layer-wise single-fault profiling: FP32 bit flips + random corruption
#
# 21 Conv2d/Linear weight layers
# 100 independent realizations per layer per fault mechanism
# 4,200 total trials
#
# Preserves:
#   - recovered checkpoint
#   - previous quantization and precision experiment outputs
#
# New outputs:
#   /kaggle/working/JFAMP/experiments/fault_profiling/
# ================================================================

import os
import sys
import json
import time
import copy
import random
import hashlib
import platform
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset

# -----------------------------
# 1. Reproducibility and paths
# -----------------------------

SEED = 20261111
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

ROOT = Path("/kaggle/working/JFAMP")
CHECKPOINT = ROOT / "retrain_to150/checkpoints/resnet18_cifar100_best_150.pth"
SPLIT_FILE = Path("/kaggle/input/datasets/crahulnaidu/jfamp-resnet18-cifar100-checkpoint/data_split_indices.json")

OUT = ROOT / "experiments/fault_profiling"
OUT.mkdir(parents=True, exist_ok=True)

assert CHECKPOINT.exists(), f"Missing checkpoint: {CHECKPOINT}"
assert SPLIT_FILE.exists(), f"Missing original split file: {SPLIT_FILE}"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_REALIZATIONS = 100
BATCH_SIZE = 256
NUM_WORKERS = 2

# -----------------------------
# 2. Load original split
# -----------------------------

with open(SPLIT_FILE, "r") as f:
    split = json.load(f)

assert len(split["train_indices"]) == 45000
assert len(split["validation_indices"]) == 5000

val_indices = split["validation_indices"]

split_hash = hashlib.sha256(
    json.dumps(split, sort_keys=True).encode()
).hexdigest()

print("Split SHA256:", split_hash)

# -----------------------------
# 3. Dataset and validation loader
# -----------------------------

mean = (0.5071, 0.4867, 0.4408)
std = (0.2675, 0.2565, 0.2761)

transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean, std)
])

data_root = ROOT / "data"
data_root.mkdir(parents=True, exist_ok=True)

# Reuse the existing CIFAR-100 cache when available.
# If the dataset is not cached, torchvision will attempt to download it.
base_dataset = torchvision.datasets.CIFAR100(
    root=str(data_root),
    train=True,
    transform=transform,
    download=True
)

val_dataset = Subset(base_dataset, val_indices)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0)
)

assert len(val_dataset) == 5000

# -----------------------------
# 4. Reconstruct model architecture
# -----------------------------

from torchvision.models import resnet18

def build_model():
    model = resnet18(weights=None)
    model.conv1 = nn.Conv2d(
        3, 64, kernel_size=3, stride=1, padding=1, bias=False
    )
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, 100)
    return model

checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

if isinstance(checkpoint, dict):
    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    else:
        state_dict = checkpoint
else:
    raise TypeError("Unexpected checkpoint format.")

# Handle DataParallel-style prefixes, if present.
state_dict = {
    k.removeprefix("module."): v
    for k, v in state_dict.items()
}

model = build_model()
model.load_state_dict(state_dict, strict=True)
model = model.to(DEVICE).eval()

# Keep a CPU copy as the immutable reference.
reference_state = {
    k: v.detach().cpu().clone()
    for k, v in model.state_dict().items()
}

del checkpoint, state_dict

# -----------------------------
# 5. Evaluation function
# -----------------------------

criterion = nn.CrossEntropyLoss(reduction="sum")

@torch.inference_mode()
def evaluate(model_to_eval):
    model_to_eval.eval()

    total_correct = 0
    total_samples = 0
    total_loss = 0.0
    nonfinite_outputs = 0

    for images, labels in val_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        outputs = model_to_eval(images)

        if not torch.isfinite(outputs).all():
            nonfinite_outputs += int(images.size(0))
            continue

        total_loss += criterion(outputs, labels).item()
        predictions = outputs.argmax(dim=1)
        total_correct += (predictions == labels).sum().item()
        total_samples += labels.numel()

    if nonfinite_outputs > 0:
        return {
            "accuracy": float("nan"),
            "loss": float("nan"),
            "nonfinite_samples": nonfinite_outputs
        }

    return {
        "accuracy": 100.0 * total_correct / total_samples,
        "loss": total_loss / total_samples,
        "nonfinite_samples": 0
    }

# -----------------------------
# 6. Establish clean FP32 baseline
# -----------------------------

baseline = evaluate(model)

print("\n========== CLEAN FP32 BASELINE ==========")
print("Accuracy:", round(baseline["accuracy"], 4), "%")
print("Loss:", baseline["loss"])
print("Validation samples:", len(val_dataset))

# Verify consistency with the previously recorded baseline.
baseline_file = ROOT / "retrain_to150/fp32_baseline.json"

if baseline_file.exists():
    with open(baseline_file, "r") as f:
        previous_baseline = json.load(f)

    previous_acc = float(previous_baseline["validation_accuracy"])
    print("Previously recorded accuracy:", previous_acc, "%")
    print("Current minus previous (pp):",
          round(baseline["accuracy"] - previous_acc, 4))

# -----------------------------
# 7. Identify eligible layers
# -----------------------------

eligible_layers = []

for name, module in model.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        if module.weight is not None:
            eligible_layers.append(name)

assert len(eligible_layers) == 21, (
    f"Expected 21 Conv2d/Linear layers, found {len(eligible_layers)}"
)

print("\nEligible layers:", len(eligible_layers))
for name in eligible_layers:
    print("  ", name)

# -----------------------------
# 8. Fault injection helpers
# -----------------------------

def get_module_by_name(model_obj, layer_name):
    modules = dict(model_obj.named_modules())
    return modules[layer_name]


def make_faulty_model(layer_name, mechanism, trial_seed):
    """
    Create a fresh model from the immutable FP32 reference.

    Exactly one weight element is modified:
      bitflip:
        Flip one uniformly selected bit (0..31) in one FP32 weight.
      random_corruption:
        Replace one selected weight with a value sampled from
        the same layer's empirical weight distribution.
    """

    faulty = build_model()
    faulty.load_state_dict(reference_state, strict=True)

    module = get_module_by_name(faulty, layer_name)
    weight = module.weight

    generator = torch.Generator(device="cpu")
    generator.manual_seed(trial_seed)

    flat = weight.detach().view(-1)
    num_weights = flat.numel()

    weight_index = int(
        torch.randint(
            low=0,
            high=num_weights,
            size=(1,),
            generator=generator
        ).item()
    )

    original_value = float(flat[weight_index].item())
    bit_position = None
    replacement_value = None

    if mechanism == "bitflip":

        # View the original FP32 storage as individual bytes.
        # This avoids signed-int32 overflow.
        raw_bytes = weight.detach().view(torch.uint8).view(-1)

        bit_position = int(
            torch.randint(0, 32, (1,), generator=generator).item()
        )

        # Kaggle's x86 environment uses little-endian storage.
        byte_index = weight_index * 4 + bit_position // 8
        bit_in_byte = bit_position % 8

        old_byte = int(raw_bytes[byte_index].item())
        new_byte = old_byte ^ (1 << bit_in_byte)

        # uint8 assignment is always in the range 0..255.
        raw_bytes[byte_index] = new_byte

        replacement_value = float(flat[weight_index].item())

    elif mechanism == "random_corruption":

        source_index = int(
            torch.randint(
                0,
                num_weights,
                (1,),
                generator=generator
            ).item()
        )

        if num_weights > 1 and source_index == weight_index:
            source_index = (source_index + 1) % num_weights

        replacement_value = float(flat[source_index].item())
        flat[weight_index] = replacement_value

    else:
        raise ValueError(f"Unknown mechanism: {mechanism}")

    faulty = faulty.to(DEVICE).eval()

    return faulty, {
        "weight_index": weight_index,
        "original_value": original_value,
        "replacement_value": replacement_value,
        "bit_position": bit_position
    }


# -----------------------------
# 9. Run all fault trials
# -----------------------------

mechanisms = ["bitflip", "random_corruption"]
records = []

total_trials = (
    len(eligible_layers) *
    len(mechanisms) *
    N_REALIZATIONS
)

print("\n========== FAULT PROFILING ==========")
print("Mechanisms:", mechanisms)
print("Realizations per layer/mechanism:", N_REALIZATIONS)
print("Total trials:", total_trials)
print("Evaluation samples per trial:", len(val_dataset))

start_time = time.time()
completed = 0

for layer_idx, layer_name in enumerate(eligible_layers):

    print(f"\n[{layer_idx + 1}/{len(eligible_layers)}] Layer: {layer_name}")

    for mechanism_idx, mechanism in enumerate(mechanisms):

        for trial in range(N_REALIZATIONS):

            trial_seed = (
                SEED
                + layer_idx * 1_000_000
                + mechanism_idx * 100_000
                + trial
            )

            faulty_model, fault_info = make_faulty_model(
                layer_name,
                mechanism,
                trial_seed
            )

            try:
                result = evaluate(faulty_model)

                if result["nonfinite_samples"] > 0:
                    accuracy_drop = float("nan")
                    loss_increase = float("nan")
                    failed = True
                else:
                    accuracy_drop = (
                        baseline["accuracy"] - result["accuracy"]
                    )
                    loss_increase = (
                        result["loss"] - baseline["loss"]
                    )
                    failed = False

                records.append({
                    "layer": layer_name,
                    "mechanism": mechanism,
                    "trial": trial,
                    "seed": trial_seed,
                    "weight_index": fault_info["weight_index"],
                    "original_value": fault_info["original_value"],
                    "replacement_value": fault_info["replacement_value"],
                    "bit_position": fault_info["bit_position"],
                    "accuracy": result["accuracy"],
                    "accuracy_drop_pp": accuracy_drop,
                    "loss": result["loss"],
                    "loss_increase": loss_increase,
                    "nonfinite_samples": result["nonfinite_samples"],
                    "failed": failed
                })

            except Exception as exc:
                records.append({
                    "layer": layer_name,
                    "mechanism": mechanism,
                    "trial": trial,
                    "seed": trial_seed,
                    "weight_index": fault_info["weight_index"],
                    "original_value": fault_info["original_value"],
                    "replacement_value": fault_info["replacement_value"],
                    "bit_position": fault_info["bit_position"],
                    "accuracy": np.nan,
                    "accuracy_drop_pp": np.nan,
                    "loss": np.nan,
                    "loss_increase": np.nan,
                    "nonfinite_samples": np.nan,
                    "failed": True,
                    "error": repr(exc)
                })

            finally:
                del faulty_model

            completed += 1

            if completed % 100 == 0:
                elapsed = time.time() - start_time
                print(
                    f"Completed {completed}/{total_trials} "
                    f"({100 * completed / total_trials:.1f}%) | "
                    f"Elapsed: {elapsed / 60:.1f} min"
                )

            if DEVICE.type == "cuda" and completed % 100 == 0:
                torch.cuda.empty_cache()

# -----------------------------
# 10. Save raw trial results
# -----------------------------

raw_df = pd.DataFrame(records)

raw_path = OUT / "fault_profiling_raw.csv"
raw_df.to_csv(raw_path, index=False)

# -----------------------------
# 11. Aggregate layer-wise results
# -----------------------------

valid_df = raw_df[
    (~raw_df["failed"]) &
    raw_df["accuracy_drop_pp"].notna()
].copy()

summary_rows = []

for (layer, mechanism), group in raw_df.groupby(
    ["layer", "mechanism"],
    sort=False
):
    valid = group[
        (~group["failed"]) &
        group["accuracy_drop_pp"].notna()
    ]

    if len(valid) > 0:
        drops = valid["accuracy_drop_pp"]
        loss_changes = valid["loss_increase"]

        summary_rows.append({
            "layer": layer,
            "mechanism": mechanism,
            "requested_trials": N_REALIZATIONS,
            "successful_trials": len(valid),
            "failed_trials": len(group) - len(valid),
            "mean_accuracy_drop_pp": drops.mean(),
            "std_accuracy_drop_pp": drops.std(ddof=1),
            "median_accuracy_drop_pp": drops.median(),
            "worst_accuracy_drop_pp": drops.max(),
            "best_accuracy_drop_pp": drops.min(),
            "mean_loss_increase": loss_changes.mean(),
            "std_loss_increase": loss_changes.std(ddof=1),
            "catastrophic_trials": int(
                group["nonfinite_samples"].fillna(0).gt(0).sum()
            )
        })
    else:
        summary_rows.append({
            "layer": layer,
            "mechanism": mechanism,
            "requested_trials": N_REALIZATIONS,
            "successful_trials": 0,
            "failed_trials": len(group),
            "mean_accuracy_drop_pp": np.nan,
            "std_accuracy_drop_pp": np.nan,
            "median_accuracy_drop_pp": np.nan,
            "worst_accuracy_drop_pp": np.nan,
            "best_accuracy_drop_pp": np.nan,
            "mean_loss_increase": np.nan,
            "std_loss_increase": np.nan,
            "catastrophic_trials": int(
                group["nonfinite_samples"].fillna(0).gt(0).sum()
            )
        })

summary_df = pd.DataFrame(summary_rows)

summary_path = OUT / "fault_profiling_layer_summary.csv"
summary_df.to_csv(summary_path, index=False)

# -----------------------------
# 12. Mechanism-level summary
# -----------------------------

mechanism_summary = (
    summary_df.groupby("mechanism")
    .agg(
        layers=("layer", "nunique"),
        successful_trials=("successful_trials", "sum"),
        failed_trials=("failed_trials", "sum"),
        mean_layer_degradation=("mean_accuracy_drop_pp", "mean"),
        worst_layer_degradation=("worst_accuracy_drop_pp", "max"),
        mean_layer_loss_increase=("mean_loss_increase", "mean")
    )
    .reset_index()
)

mechanism_summary_path = OUT / "fault_profiling_mechanism_summary.csv"
mechanism_summary.to_csv(mechanism_summary_path, index=False)

# -----------------------------
# 13. Save experiment metadata
# -----------------------------

elapsed_seconds = time.time() - start_time

metadata = {
    "experiment": "JFAMP layer-wise single-fault profiling",
    "checkpoint": str(CHECKPOINT),
    "checkpoint_sha256": hashlib.sha256(
        CHECKPOINT.read_bytes()
    ).hexdigest(),
    "split_file": str(SPLIT_FILE),
    "split_sha256": split_hash,
    "baseline_accuracy_percent": baseline["accuracy"],
    "baseline_loss": baseline["loss"],
    "validation_samples": len(val_dataset),
    "eligible_layers": eligible_layers,
    "number_of_layers": len(eligible_layers),
    "fault_mechanisms": mechanisms,
    "granularity": "one weight element per realization",
    "bitflip_model": (
        "One uniformly selected bit among the 32 bits of one FP32 weight"
    ),
    "random_corruption_model": (
        "Replace one selected FP32 weight with a random value sampled "
        "from the empirical weight distribution of the same layer"
    ),
    "realizations_per_layer_per_mechanism": N_REALIZATIONS,
    "total_requested_trials": total_trials,
    "completed_trials": len(raw_df),
    "successful_trials": int((~raw_df["failed"]).sum()),
    "failed_trials": int(raw_df["failed"].sum()),
    "batch_size": BATCH_SIZE,
    "device": str(DEVICE),
    "torch_version": torch.__version__,
    "torchvision_version": torchvision.__version__,
    "python_version": platform.python_version(),
    "seed": SEED,
    "elapsed_seconds": elapsed_seconds,
    "outputs": {
        "raw": str(raw_path),
        "layer_summary": str(summary_path),
        "mechanism_summary": str(mechanism_summary_path)
    }
}

metadata_path = OUT / "fault_profiling_metadata.json"

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

# -----------------------------
# 14. Final report
# -----------------------------

print("\n\n========== JFAMP FAULT PROFILING COMPLETE ==========")
print("Baseline accuracy:", round(baseline["accuracy"], 4), "%")
print("Requested trials:", total_trials)
print("Completed records:", len(raw_df))
print("Successful trials:", int((~raw_df["failed"]).sum()))
print("Failed trials:", int(raw_df["failed"].sum()))
print("Elapsed time:", round(elapsed_seconds / 60, 2), "minutes")

print("\n--- Layer-wise summary ---")
display(
    summary_df.sort_values(
        ["mechanism", "mean_accuracy_drop_pp"],
        ascending=[True, False]
    ).reset_index(drop=True)
)

print("\n--- Mechanism summary ---")
display(mechanism_summary)

print("\nSaved files:")
print(raw_path)
print(summary_path)
print(mechanism_summary_path)
print(metadata_path)

print("\nPrevious experiment directories were not modified.")


Split SHA256: b632b03ccc8166092cbf5c9e02a41c754248533ef1f60292c0404e3a8d80d0e7

========== CLEAN FP32 BASELINE ==========
Accuracy: 77.78 %
Loss: 0.914325048828125
Validation samples: 5000
Previously recorded accuracy: 77.78 %
Current minus previous (pp): 0.0

Eligible layers: 21
   conv1
   layer1.0.conv1
   layer1.0.conv2
   layer1.1.conv1
   layer1.1.conv2
   layer2.0.conv1
   layer2.0.conv2
   layer2.0.downsample.0
   layer2.1.conv1
   layer2.1.conv2
   layer3.0.conv1
   layer3.0.conv2
   layer3.0.downsample.0
   layer3.1.conv1
   layer3.1.conv2
   layer4.0.conv1
   layer4.0.conv2
   layer4.0.downsample.0
   layer4.1.conv1
   layer4.1.conv2
   fc

========== FAULT PROFILING ==========
Mechanisms: ['bitflip', 'random_corruption']
Realizations per layer/mechanism: 100
Total trials: 4200
Evaluation samples per trial: 5000

[1/21] Layer: conv1
Completed 100/4200 (2.4%) | Elapsed: 4.0 min
Completed 200/4200 (4.8%) | Elapsed: 7.9 min

[2/21] Layer: layer1.0.conv1
Completed 300/4200 (7.1%

In [15]:

from pathlib import Path
import pandas as pd
import json
import os

OUT = Path("/kaggle/working/JFAMP/experiments/fault_profiling")
OUT.mkdir(parents=True, exist_ok=True)

print("========== KERNEL STATUS ==========")

print("records exists:", "records" in globals())
print("completed exists:", "completed" in globals())
print("eligible_layers exists:", "eligible_layers" in globals())

if "records" in globals():
    print("Records in memory:", len(records))

if "completed" in globals():
    print("Completed counter:", completed)

if "raw_df" in globals():
    print("raw_df exists:", len(raw_df))

print("\n========== EXISTING SAVED FILES ==========")

for p in OUT.glob("*"):
    print(p.name, f"({p.stat().st_size / 1024:.2f} KB)")

# Preserve any results still available in memory.
if "records" in globals() and len(records) > 0:
    partial_df = pd.DataFrame(records)
    partial_path = OUT / "fault_profiling_partial_recovery.csv"
    partial_df.to_csv(partial_path, index=False)

    print("\nPARTIAL RESULTS SAVED:")
    print(partial_path)
    print("Rows:", len(partial_df))

    if {"layer", "mechanism"}.issubset(partial_df.columns):
        print("\nTrials by layer and mechanism:")
        display(
            partial_df.groupby(["layer", "mechanism"])
            .size()
            .reset_index(name="trials")
        )
else:
    print("\nNo in-memory trial records are currently available.")


========== KERNEL STATUS ==========
records exists: True
completed exists: True
eligible_layers exists: True
Records in memory: 4200
Completed counter: 4200
raw_df exists: 4200

========== EXISTING SAVED FILES ==========
fault_profiling_mechanism_summary.csv (0.25 KB)
fault_profiling_metadata.json (2.08 KB)
fault_profiling_raw.csv (628.09 KB)
fault_profiling_layer_summary.csv (6.80 KB)

PARTIAL RESULTS SAVED:
/kaggle/working/JFAMP/experiments/fault_profiling/fault_profiling_partial_recovery.csv
Rows: 4200

Trials by layer and mechanism:


,layer,mechanism,trials
0,conv1,bitflip,100
1,conv1,random_corruption,100
2,fc,bitflip,100
3,fc,random_corruption,100
4,layer1.0.conv1,bitflip,100
5,layer1.0.conv1,random_corruption,100
6,layer1.0.conv2,bitflip,100
7,layer1.0.conv2,random_corruption,100
8,layer1.1.conv1,bitflip,100
9,layer1.1.conv1,random_corruption,100


In [16]:

from pathlib import Path
import pandas as pd
import numpy as np

OUT = Path("/kaggle/working/JFAMP/experiments/fault_profiling")

raw = pd.read_csv(OUT / "fault_profiling_raw.csv")
summary = pd.read_csv(OUT / "fault_profiling_layer_summary.csv")
mechanisms = pd.read_csv(OUT / "fault_profiling_mechanism_summary.csv")

# ------------------------------------------------
# 1. Verify completeness
# ------------------------------------------------

print("========== COMPLETENESS CHECK ==========")
print("Raw records:", len(raw))
print("Unique layers:", raw["layer"].nunique())
print("Mechanisms:", raw["mechanism"].unique())
print("Total failed trials:", raw["failed"].sum())

counts = raw.groupby(["layer", "mechanism"]).size()
print("Minimum trials per group:", counts.min())
print("Maximum trials per group:", counts.max())

assert len(raw) == 4200
assert raw["layer"].nunique() == 21
assert len(counts) == 42
assert (counts == 100).all()

print("PASS: All 4,200 requested trials are present.")

# ------------------------------------------------
# 2. Mechanism-level comparison
# ------------------------------------------------

print("\n========== MECHANISM COMPARISON ==========")
display(mechanisms)

# ------------------------------------------------
# 3. Most fault-sensitive layers
# ------------------------------------------------

for mechanism in ["bitflip", "random_corruption"]:

    subset = summary[
        summary["mechanism"] == mechanism
    ].copy()

    subset = subset.sort_values(
        "mean_accuracy_drop_pp",
        ascending=False
    )

    print(f"\n========== MOST SENSITIVE: {mechanism} ==========")

    display(
        subset[
            [
                "layer",
                "successful_trials",
                "failed_trials",
                "mean_accuracy_drop_pp",
                "std_accuracy_drop_pp",
                "median_accuracy_drop_pp",
                "worst_accuracy_drop_pp",
                "mean_loss_increase",
                "catastrophic_trials"
            ]
        ].head(10)
    )

    print(f"\n========== MOST TOLERANT: {mechanism} ==========")

    display(
        subset.sort_values(
            "mean_accuracy_drop_pp",
            ascending=True
        )[
            [
                "layer",
                "mean_accuracy_drop_pp",
                "std_accuracy_drop_pp",
                "median_accuracy_drop_pp",
                "worst_accuracy_drop_pp",
                "catastrophic_trials"
            ]
        ].head(10)
    )

# ------------------------------------------------
# 4. Compare mechanisms layer by layer
# ------------------------------------------------

comparison = summary.pivot(
    index="layer",
    columns="mechanism",
    values="mean_accuracy_drop_pp"
).reset_index()

comparison["bitflip_minus_random"] = (
    comparison["bitflip"] -
    comparison["random_corruption"]
)

print("\n========== LAYER-WISE MECHANISM COMPARISON ==========")
display(
    comparison.sort_values(
        "bitflip_minus_random",
        ascending=False
    )
)

comparison.to_csv(
    OUT / "fault_mechanism_layer_comparison.csv",
    index=False
)

# ------------------------------------------------
# 5. Overall fault statistics
# ------------------------------------------------

print("\n========== OVERALL FAULT STATISTICS ==========")

for mechanism in ["bitflip", "random_corruption"]:

    group = raw[
        (raw["mechanism"] == mechanism) &
        (~raw["failed"].astype(bool))
    ]

    drops = group["accuracy_drop_pp"].dropna()

    print(f"\n{mechanism}")
    print("Valid trials:", len(drops))
    print("Mean degradation:", drops.mean(), "pp")
    print("Median degradation:", drops.median(), "pp")
    print("Std deviation:", drops.std(), "pp")
    print("Worst observed degradation:", drops.max(), "pp")
    print("Best observed degradation:", drops.min(), "pp")

# ------------------------------------------------
# 6. Bit-position analysis
# ------------------------------------------------

bit_trials = raw[
    (raw["mechanism"] == "bitflip") &
    raw["bit_position"].notna()
].copy()

bit_trials["bit_position"] = bit_trials[
    "bit_position"
].astype(int)

bit_summary = (
    bit_trials.groupby("bit_position")
    .agg(
        trials=("accuracy_drop_pp", "count"),
        mean_accuracy_drop_pp=("accuracy_drop_pp", "mean"),
        mean_loss_increase=("loss_increase", "mean"),
        catastrophic_trials=("nonfinite_samples",
                             lambda x: x.fillna(0).gt(0).sum())
    )
    .reset_index()
)

print("\n========== BIT-POSITION SENSITIVITY ==========")
display(bit_summary)

bit_summary.to_csv(
    OUT / "fault_bit_position_summary.csv",
    index=False
)

print("\nAnalysis files saved in:", OUT)


========== COMPLETENESS CHECK ==========
Raw records: 4200
Unique layers: 21
Mechanisms: ['bitflip' 'random_corruption']
Total failed trials: 19
Minimum trials per group: 100
Maximum trials per group: 100
PASS: All 4,200 requested trials are present.

========== MECHANISM COMPARISON ==========


,mechanism,layers,successful_trials,failed_trials,mean_layer_degradation,worst_layer_degradation,mean_layer_loss_increase
0,bitflip,21,2081,19,1.348014,77.24,inf
1,random_corruption,21,2100,0,0.001895,1.20,0.000052



========== MOST SENSITIVE: bitflip ==========


,layer,successful_trials,failed_trials,mean_accuracy_drop_pp,std_accuracy_drop_pp,median_accuracy_drop_pp,worst_accuracy_drop_pp,mean_loss_increase,catastrophic_trials
28,layer3.1.conv2,100,0,5.093400,18.695101,0.0,76.64,9.339970e+33,0
32,layer4.0.conv2,100,0,4.308600,15.870787,0.0,71.96,inf,0
18,layer2.1.conv2,100,0,3.069400,15.112715,0.0,76.90,1.231253e+31,0
22,layer3.0.conv2,100,0,3.053400,15.031054,0.0,76.60,4.370415e+33,0
26,layer3.1.conv1,100,0,2.295400,13.115636,0.0,76.74,4.363283e+33,0
38,layer4.1.conv2,100,0,1.670600,7.411482,0.0,39.86,1.196145e+33,0
8,layer1.1.conv2,93,7,1.650108,11.195406,0.0,76.82,1.238522e+24,7
24,layer3.0.downsample.0,100,0,1.537000,10.787392,0.0,76.78,inf,0
10,layer2.0.conv1,97,3,0.797320,7.799497,0.0,76.82,7.108560e+31,3
16,layer2.1.conv1,98,2,0.782653,7.756038,0.0,76.78,2.932834e+32,2



========== MOST TOLERANT: bitflip ==========


,layer,mean_accuracy_drop_pp,std_accuracy_drop_pp,median_accuracy_drop_pp,worst_accuracy_drop_pp,catastrophic_trials
2,layer1.0.conv1,-0.000408,0.007587,0.0,0.04,2
6,layer1.1.conv1,0.000202,0.010099,0.0,0.06,1
12,layer2.0.conv2,0.001414,0.012207,0.0,0.08,1
0,conv1,0.007677,0.042734,0.0,0.34,1
14,layer2.0.downsample.0,0.019596,0.089487,0.0,0.62,1
36,layer4.1.conv1,0.506000,5.055964,0.0,50.56,0
4,layer1.0.conv2,0.613333,5.850926,0.0,58.18,1
30,layer4.0.conv1,0.686600,6.861961,0.0,68.62,0
40,fc,0.698200,6.907607,0.0,69.08,0
34,layer4.0.downsample.0,0.746600,7.474082,0.0,74.74,0


... (5 outputs left out)

mechanism,layer,bitflip,random_corruption,bitflip_minus_random
15,layer3.1.conv2,5.093400,0.0002,5.093200
17,layer4.0.conv2,4.308600,0.0000,4.308600
10,layer2.1.conv2,3.069400,-0.0028,3.072200
12,layer3.0.conv2,3.053400,-0.0020,3.055400
14,layer3.1.conv1,2.295400,-0.0014,2.296800
20,layer4.1.conv2,1.670600,0.0000,1.670600
5,layer1.1.conv2,1.650108,0.0012,1.648908
13,layer3.0.downsample.0,1.537000,-0.0036,1.540600
6,layer2.0.conv1,0.797320,-0.0034,0.800720
9,layer2.1.conv1,0.782653,-0.0008,0.783453



========== OVERALL FAULT STATISTICS ==========

bitflip
Valid trials: 2081
Mean degradation: 1.3525612686208555 pp
Median degradation: 0.0 pp
Std deviation: 9.623398029733103 pp
Worst observed degradation: 77.24 pp
Best observed degradation: -0.1200000000000045 pp

random_corruption
Valid trials: 2100
Mean degradation: 0.0018952380952385306 pp
Median degradation: 0.0 pp
Std deviation: 0.043850285332754736 pp
Worst observed degradation: 1.2000000000000028 pp
Best observed degradation: -0.0999999999999943 pp

========== BIT-POSITION SENSITIVITY ==========


,bit_position,trials,mean_accuracy_drop_pp,mean_loss_increase,catastrophic_trials
0,0,69,0.000000,4.444952e-09,0
1,1,80,0.000000,6.580353e-09,0
2,2,65,0.000000,1.126803e-08,0
3,3,54,0.000000,1.328080e-08,0
4,4,56,0.000000,1.727513e-08,0
5,5,54,0.000000,1.311126e-08,0
6,6,76,0.000000,8.412411e-09,0
7,7,74,0.000000,9.443953e-09,0
8,8,80,0.000000,1.457214e-08,0
9,9,62,0.000000,1.353602e-08,0



Analysis files saved in: /kaggle/working/JFAMP/experiments/fault_profiling


In [17]:

# ================================================================
# JFAMP EXPERIMENT 6: PROTECTION EFFECTIVENESS PROFILING
# ResNet-18 + CIFAR-100 | Epoch-139 best checkpoint
#
# Protection candidates:
#   1. No protection
#   2. Single parity bit
#   3. SECDED (Hamming 38,32 + overall parity = 39 bits)
#   4. DMR (dual modular redundancy, detection only)
#
# Fault models:
#   A. Single data-bit flip
#   B. Random same-layer weight replacement
#
# Scope: all Conv2d and Linear weight tensors (21 layers expected)
# Trials: 100 per layer per fault model
#
# This is codeword-level software simulation, not FPGA execution.
# ================================================================

import os
import json
import time
import random
import hashlib
import platform
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch

# -------------------- 1. REPRODUCIBILITY --------------------------

SEED = 42
TRIALS_PER_LAYER = 100

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ROOT = Path("/kaggle/working/JFAMP")
CHECKPOINT = ROOT / "retrain_to150/checkpoints/resnet18_cifar100_best_150.pth"

OUT_DIR = ROOT / "experiments/protection_profiling"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert CHECKPOINT.exists(), f"Checkpoint not found: {CHECKPOINT}"

print("=" * 72)
print("JFAMP EXPERIMENT 6: PROTECTION PROFILING")
print("=" * 72)
print("Checkpoint:", CHECKPOINT)
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("Trials per layer and fault type:", TRIALS_PER_LAYER)
print("Output:", OUT_DIR)

# -------------------- 2. LOAD CHECKPOINT --------------------------

checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

def extract_state_dict(obj):
    """Find a model state dictionary in common checkpoint formats."""
    if isinstance(obj, dict):
        for key in (
            "state_dict",
            "model_state_dict",
            "model",
            "net",
            "weights"
        ):
            candidate = obj.get(key)
            if isinstance(candidate, dict) and candidate:
                if any(torch.is_tensor(v) for v in candidate.values()):
                    return candidate

        if any(torch.is_tensor(v) for v in obj.values()):
            return obj

    raise RuntimeError(
        "Could not locate a model state_dict in the checkpoint."
    )

state_dict = extract_state_dict(checkpoint)

# Remove DataParallel prefix if present.
clean_state = {}
for key, value in state_dict.items():
    new_key = key[7:] if key.startswith("module.") else key
    clean_state[new_key] = value

# Extract only Conv2d/Linear-like weight tensors.
# Weight tensors are identified by their dimensionality:
# 4-D = convolution, 2-D = linear.
layers = {}

for name, tensor in clean_state.items():
    if (
        name.endswith(".weight")
        and torch.is_tensor(tensor)
        and tensor.ndim in (2, 4)
        and tensor.is_floating_point()
    ):
        array = tensor.detach().cpu().to(torch.float32).numpy()
        layers[name.rsplit(".weight", 1)[0]] = np.ascontiguousarray(
            array.reshape(-1).copy()
        )

assert len(layers) > 0, "No Conv2d/Linear weight tensors found."

print("\nLoaded weight layers:", len(layers))
print("Total weights:", sum(len(v) for v in layers.values()))

if len(layers) != 21:
    print(
        "WARNING: Expected 21 layers based on previous profiling. "
        "Please verify the layer inventory."
    )

# -------------------- 3. SECDED IMPLEMENTATION --------------------

# Hamming(38,32):
#   Data positions: all positions 1..38 except 1,2,4,8,16,32.
#   Six Hamming parity bits.
#   Position 39: overall parity bit.
#
# Even parity convention.

HAMMING_PARITY_POSITIONS = {1, 2, 4, 8, 16, 32}
DATA_POSITIONS = [
    p for p in range(1, 39)
    if p not in HAMMING_PARITY_POSITIONS
]

assert len(DATA_POSITIONS) == 32

def parity_of_bits(bits):
    result = 0
    for bit in bits:
        result ^= int(bit)
    return result

def encode_secded(data_word):
    """Encode a 32-bit data word into a 39-bit SECDED codeword."""
    code = [0] * 40  # positions 1..39

    for i, pos in enumerate(DATA_POSITIONS):
        code[pos] = (int(data_word) >> i) & 1

    # Six Hamming parity bits.
    for parity_pos in HAMMING_PARITY_POSITIONS:
        covered = [
            code[pos]
            for pos in range(1, 39)
            if (pos & parity_pos) and pos != parity_pos
        ]
        code[parity_pos] = parity_of_bits(covered)

    # Overall parity over positions 1..38.
    code[39] = parity_of_bits(code[1:39])

    return code

def decode_secded(code):
    """
    Decode SECDED codeword.

    Returns:
        decoded_data_word
        status:
            clean
            corrected
            detected_double
            detected_uncorrectable
    """
    code = list(code)

    syndrome = 0

    for parity_pos in HAMMING_PARITY_POSITIONS:
        parity = parity_of_bits([
            code[pos]
            for pos in range(1, 39)
            if pos & parity_pos
        ])
        if parity:
            syndrome |= parity_pos

    overall_check = parity_of_bits(code[1:40])

    status = "clean"

    if syndrome != 0 and overall_check == 1:
        # Single-bit error in Hamming positions.
        if 1 <= syndrome <= 38:
            code[syndrome] ^= 1
            status = "corrected"
        else:
            status = "detected_uncorrectable"

    elif syndrome == 0 and overall_check == 1:
        # Single-bit error in the overall parity bit.
        code[39] ^= 1
        status = "corrected"

    elif syndrome != 0 and overall_check == 0:
        # Even-numbered error pattern: double-bit error detected.
        status = "detected_double"

    decoded = 0
    for i, pos in enumerate(DATA_POSITIONS):
        decoded |= (int(code[pos]) << i)

    return decoded, status

# -------------------- 4. PARITY IMPLEMENTATION --------------------

def encode_parity(data_word):
    data_bits = [(int(data_word) >> i) & 1 for i in range(32)]
    return parity_of_bits(data_bits)

def check_parity(data_word, stored_parity):
    current_parity = encode_parity(data_word)
    return current_parity != int(stored_parity)

# -------------------- 5. BIT MANIPULATION HELPERS ----------------

def float32_to_uint32(value):
    return int(np.asarray(np.float32(value)).view(np.uint32).item())

def uint32_to_float32(value):
    return np.asarray(np.uint32(value)).view(np.float32).item()

def replace_data_bits_in_codeword(code, new_word):
    """Replace only the 32 data positions, leaving ECC bits unchanged."""
    result = list(code)
    for i, pos in enumerate(DATA_POSITIONS):
        result[pos] = (int(new_word) >> i) & 1
    return result

def bit_flip_data_word(word, bit_position):
    return int(word) ^ (1 << int(bit_position))

# -------------------- 6. SELF-TEST SECDED -------------------------

print("\nRunning SECDED correctness checks...")

test_word = 0xA5C39E71
test_code = encode_secded(test_word)

decoded, status = decode_secded(test_code)
assert decoded == test_word and status == "clean"

# Test all 32 data-bit positions.
for bit in range(32):
    corrupted = list(test_code)
    corrupted[DATA_POSITIONS[bit]] ^= 1
    decoded, status = decode_secded(corrupted)
    assert decoded == test_word and status == "corrected"

# Test all six Hamming parity bits.
for pos in HAMMING_PARITY_POSITIONS:
    corrupted = list(test_code)
    corrupted[pos] ^= 1
    decoded, status = decode_secded(corrupted)
    assert decoded == test_word and status == "corrected"

# Test overall parity bit.
corrupted = list(test_code)
corrupted[39] ^= 1
decoded, status = decode_secded(corrupted)
assert decoded == test_word and status == "corrected"

# Test a representative double-bit error.
corrupted = list(test_code)
corrupted[DATA_POSITIONS[0]] ^= 1
corrupted[DATA_POSITIONS[1]] ^= 1
decoded, status = decode_secded(corrupted)
assert status == "detected_double"

print("SECDED tests passed: 32 data bits + 7 parity bits + double-bit detection.")

# -------------------- 7. PROTECTION TRIALS -----------------------

records = []
start_time = time.time()

fault_types = ["single_bit_flip", "random_weight_replacement"]

for layer_number, (layer_name, weights) in enumerate(layers.items(), start=1):

    n_weights = len(weights)

    if n_weights < 2:
        continue

    print(
        f"\n[{layer_number}/{len(layers)}] {layer_name} "
        f"({n_weights:,} weights)"
    )

    for fault_type in fault_types:

        for trial in range(TRIALS_PER_LAYER):

            # Select the original weight.
            weight_index = random.randrange(n_weights)
            original_value = np.float32(weights[weight_index])
            original_word = float32_to_uint32(original_value)

            # Generate a corrupted data word.
            if fault_type == "single_bit_flip":
                bit_position = random.randrange(32)
                corrupted_word = bit_flip_data_word(
                    original_word,
                    bit_position
                )
                fault_description = f"data_bit_{bit_position}"

            else:
                replacement_index = random.randrange(n_weights - 1)
                if replacement_index >= weight_index:
                    replacement_index += 1

                replacement_value = np.float32(weights[replacement_index])
                corrupted_word = float32_to_uint32(replacement_value)

                fault_description = "same_layer_weight_replacement"

            actual_data_changed = corrupted_word != original_word

            # ---------------- NO PROTECTION ----------------
            no_protection_detected = False
            no_protection_corrected = False
            no_protection_residual = actual_data_changed

            # ---------------- PARITY ------------------------
            original_parity = encode_parity(original_word)

            parity_detected = check_parity(
                corrupted_word,
                original_parity
            )

            # Parity detects, but cannot correct the corrupted data.
            parity_corrected = False
            parity_residual = actual_data_changed

            # ---------------- SECDED ------------------------
            original_codeword = encode_secded(original_word)

            corrupted_codeword = replace_data_bits_in_codeword(
                original_codeword,
                corrupted_word
            )

            decoded_word, secded_status = decode_secded(
                corrupted_codeword
            )

            secded_detected = secded_status != "clean"
            secded_corrected = decoded_word == original_word
            secded_residual = decoded_word != original_word

            if not actual_data_changed:
                # An identical replacement is not a fault.
                secded_detected = False
                secded_corrected = False
                secded_residual = False
                secded_status = "no_data_change"

            # ---------------- DMR ----------------------------
            # Two copies are stored. One copy is corrupted.
            # A mismatch is detectable, but DMR alone cannot
            # identify which copy is correct.
            dmr_copy_a = original_word
            dmr_copy_b = corrupted_word

            dmr_detected = dmr_copy_a != dmr_copy_b
            dmr_corrected = False
            dmr_residual = dmr_copy_b != original_word

            if not actual_data_changed:
                dmr_detected = False
                dmr_residual = False

            # ---------------- RECORD --------------------------
            common = {
                "layer": layer_name,
                "fault_type": fault_type,
                "trial": trial,
                "weight_index": weight_index,
                "fault_description": fault_description,
                "actual_data_changed": bool(actual_data_changed),
                "secded_status": secded_status,
            }

            strategies = {
                "none": (
                    no_protection_detected,
                    no_protection_corrected,
                    no_protection_residual
                ),
                "parity": (
                    parity_detected,
                    parity_corrected,
                    parity_residual
                ),
                "secded": (
                    secded_detected,
                    secded_corrected,
                    secded_residual
                ),
                "dmr": (
                    dmr_detected,
                    dmr_corrected,
                    dmr_residual
                ),
            }

            for strategy, (detected, corrected, residual) in strategies.items():
                records.append({
                    **common,
                    "strategy": strategy,
                    "detected": bool(detected),
                    "corrected": bool(corrected),
                    "residual_data_error": bool(residual),
                    "undetected_error": bool(
                        actual_data_changed and not detected
                    ),
                })

    print("  Completed both fault models.")

# -------------------- 8. BUILD DATAFRAMES ------------------------

raw_df = pd.DataFrame(records)

assert not raw_df.empty
assert raw_df["layer"].nunique() == len(layers)

print("\n" + "=" * 72)
print("TRIAL GENERATION COMPLETE")
print("=" * 72)
print("Rows:", len(raw_df))
print("Layers:", raw_df["layer"].nunique())
print("Fault types:", raw_df["fault_type"].nunique())
print("Strategies:", raw_df["strategy"].nunique())

# -------------------- 9. SUMMARY METRICS -------------------------

# Each row represents one protection strategy applied to one fault trial.
# The denominator for detection/correction rates is the number of
# trials in which the stored data actually changed.

summary = (
    raw_df
    .groupby(["fault_type", "layer", "strategy"], as_index=False)
    .agg(
        trials=("trial", "count"),
        actual_faults=("actual_data_changed", "sum"),
        detected_faults=("detected", "sum"),
        corrected_faults=("corrected", "sum"),
        residual_errors=("residual_data_error", "sum"),
        undetected_errors=("undetected_error", "sum"),
    )
)

def safe_rate(numerator, denominator):
    return (
        float(numerator) / float(denominator)
        if denominator else 0.0
    )

summary["detection_rate_pct"] = summary.apply(
    lambda r: 100 * safe_rate(
        r["detected_faults"], r["actual_faults"]
    ),
    axis=1
)

summary["correction_rate_pct"] = summary.apply(
    lambda r: 100 * safe_rate(
        r["corrected_faults"], r["actual_faults"]
    ),
    axis=1
)

summary["residual_error_rate_pct"] = summary.apply(
    lambda r: 100 * safe_rate(
        r["residual_errors"], r["actual_faults"]
    ),
    axis=1
)

summary["undetected_error_rate_pct"] = summary.apply(
    lambda r: 100 * safe_rate(
        r["undetected_errors"], r["actual_faults"]
    ),
    axis=1
)

# -------------------- 10. STORAGE OVERHEAD -----------------------

storage_overhead = {
    "none": {
        "data_bits": 32,
        "stored_bits": 32,
        "additional_bits": 0,
        "overhead_pct": 0.0,
        "description": "32-bit data word"
    },
    "parity": {
        "data_bits": 32,
        "stored_bits": 33,
        "additional_bits": 1,
        "overhead_pct": 100 * 1 / 32,
        "description": "32-bit data + 1 parity bit"
    },
    "secded": {
        "data_bits": 32,
        "stored_bits": 39,
        "additional_bits": 7,
        "overhead_pct": 100 * 7 / 32,
        "description": "32-bit data + 6 Hamming bits + 1 overall parity bit"
    },
    "dmr": {
        "data_bits": 32,
        "stored_bits": 64,
        "additional_bits": 32,
        "overhead_pct": 100.0,
        "description": "Two 32-bit copies; comparison detects mismatch"
    }
}

summary["storage_overhead_pct"] = summary["strategy"].map(
    lambda s: storage_overhead[s]["overhead_pct"]
)

# -------------------- 11. MECHANISM-LEVEL SUMMARY -----------------

mechanism_summary = (
    raw_df
    .groupby(["fault_type", "strategy"], as_index=False)
    .agg(
        trials=("trial", "count"),
        actual_faults=("actual_data_changed", "sum"),
        detected_faults=("detected", "sum"),
        corrected_faults=("corrected", "sum"),
        residual_errors=("residual_data_error", "sum"),
        undetected_errors=("undetected_error", "sum"),
    )
)

for metric, numerator in [
    ("detection_rate_pct", "detected_faults"),
    ("correction_rate_pct", "corrected_faults"),
    ("residual_error_rate_pct", "residual_errors"),
    ("undetected_error_rate_pct", "undetected_errors"),
]:
    mechanism_summary[metric] = mechanism_summary.apply(
        lambda r: 100 * safe_rate(
            r[numerator], r["actual_faults"]
        ),
        axis=1
    )

mechanism_summary["storage_overhead_pct"] = (
    mechanism_summary["strategy"].map(
        lambda s: storage_overhead[s]["overhead_pct"]
    )
)

# -------------------- 12. SAVE RESULTS ---------------------------

raw_path = OUT_DIR / "protection_profiling_raw.csv"
layer_path = OUT_DIR / "protection_layer_summary.csv"
mechanism_path = OUT_DIR / "protection_mechanism_summary.csv"
storage_path = OUT_DIR / "protection_storage_overhead.json"
metadata_path = OUT_DIR / "protection_profiling_metadata.json"

raw_df.to_csv(raw_path, index=False)
summary.to_csv(layer_path, index=False)
mechanism_summary.to_csv(mechanism_path, index=False)

with open(storage_path, "w") as f:
    json.dump(storage_overhead, f, indent=2)

metadata = {
    "experiment": "JFAMP Experiment 6 - Protection Profiling",
    "checkpoint": str(CHECKPOINT),
    "checkpoint_sha256": hashlib.sha256(
        CHECKPOINT.read_bytes()
    ).hexdigest(),
    "seed": SEED,
    "trials_per_layer_per_fault_type": TRIALS_PER_LAYER,
    "layers_profiled": int(raw_df["layer"].nunique()),
    "fault_types": fault_types,
    "protection_strategies": list(storage_overhead.keys()),
    "total_strategy_trial_records": int(len(raw_df)),
    "device": DEVICE,
    "torch_version": torch.__version__,
    "python_version": platform.python_version(),
    "secdec_codeword": {
        "data_bits": 32,
        "hamming_parity_bits": 6,
        "overall_parity_bits": 1,
        "total_bits": 39,
        "single_error_correction": True,
        "double_error_detection": True
    },
    "methodology_notes": [
        "Protection effectiveness is evaluated at stored-weight/codeword level.",
        "Single-bit faults are injected into data bits.",
        "Random replacement uses another sampled weight from the same layer.",
        "Parity detects selected error patterns but does not correct data.",
        "SECDED is evaluated using an explicit Hamming decoder.",
        "DMR compares two copies and detects disagreement but does not correct it.",
        "No end-to-end model accuracy recovery is claimed by this experiment.",
        "Storage overhead is theoretical and excludes alignment and controller costs.",
        "No physical FPGA area, power, or timing is measured."
    ],
    "output_files": [
        str(raw_path),
        str(layer_path),
        str(mechanism_path),
        str(storage_path)
    ]
}

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

# -------------------- 13. DISPLAY RESULTS ------------------------

print("\n" + "=" * 72)
print("PROTECTION PROFILING RESULTS")
print("=" * 72)

print("\nMECHANISM-LEVEL SUMMARY")
display(
    mechanism_summary[
        [
            "fault_type",
            "strategy",
            "actual_faults",
            "detection_rate_pct",
            "correction_rate_pct",
            "residual_error_rate_pct",
            "undetected_error_rate_pct",
            "storage_overhead_pct"
        ]
    ].round(4)
)

print("\nLAYER-LEVEL SUMMARY: FIRST 20 ROWS")
display(
    summary[
        [
            "fault_type",
            "layer",
            "strategy",
            "detection_rate_pct",
            "correction_rate_pct",
            "residual_error_rate_pct",
            "undetected_error_rate_pct"
        ]
    ].head(20).round(4)
)

print("\nSECDED STATUS DISTRIBUTION")
display(
    raw_df[raw_df["strategy"] == "secded"]
    .groupby(["fault_type", "secded_status"])
    .size()
    .reset_index(name="count")
)

print("\nSTORAGE OVERHEAD")
display(pd.DataFrame(storage_overhead).T)

elapsed = time.time() - start_time

print("\n" + "=" * 72)
print("EXPERIMENT 6 COMPLETED")
print("=" * 72)
print(f"Elapsed time: {elapsed:.2f} seconds")
print(f"Layers profiled: {raw_df['layer'].nunique()}")
print(f"Raw strategy-trial records: {len(raw_df)}")
print("\nSaved files:")
for path in [
    raw_path,
    layer_path,
    mechanism_path,
    storage_path,
    metadata_path
]:
    print(f"  {path} ({path.stat().st_size / 1024:.2f} KB)")

print("\nIMPORTANT:")
print("This experiment measures codeword-level protection behavior.")
print("It does not yet measure full-network accuracy after protection.")
print("Keep all existing JFAMP experiment outputs unchanged.")


JFAMP EXPERIMENT 6: PROTECTION PROFILING
Checkpoint: /kaggle/working/JFAMP/retrain_to150/checkpoints/resnet18_cifar100_best_150.pth
Device: cuda
PyTorch: 2.11.0+cu130
Trials per layer and fault type: 100
Output: /kaggle/working/JFAMP/experiments/protection_profiling

Loaded weight layers: 21
Total weights: 11210432

Running SECDED correctness checks...
SECDED tests passed: 32 data bits + 7 parity bits + double-bit detection.

[1/21] conv1 (1,728 weights)
  Completed both fault models.

[2/21] layer1.0.conv1 (36,864 weights)
  Completed both fault models.

[3/21] layer1.0.conv2 (36,864 weights)
  Completed both fault models.

[4/21] layer1.1.conv1 (36,864 weights)
  Completed both fault models.

[5/21] layer1.1.conv2 (36,864 weights)
  Completed both fault models.

[6/21] layer2.0.conv1 (73,728 weights)
  Completed both fault models.

[7/21] layer2.0.conv2 (147,456 weights)
  Completed both fault models.

[8/21] layer2.0.downsample.0 (8,192 weights)
  Completed both fault models.

[9/21

,fault_type,strategy,actual_faults,detection_rate_pct,correction_rate_pct,residual_error_rate_pct,undetected_error_rate_pct,storage_overhead_pct
0,random_weight_replacement,dmr,2100,100.0000,0.0,100.0,0.0000,100.000
1,random_weight_replacement,none,2100,0.0000,0.0,100.0,100.0000,0.000
2,random_weight_replacement,parity,2100,48.1905,0.0,100.0,51.8095,3.125
3,random_weight_replacement,secded,2100,99.0476,0.0,100.0,0.9524,21.875
4,single_bit_flip,dmr,2100,100.0000,0.0,100.0,0.0000,100.000
5,single_bit_flip,none,2100,0.0000,0.0,100.0,100.0000,0.000
6,single_bit_flip,parity,2100,100.0000,0.0,100.0,0.0000,3.125
7,single_bit_flip,secded,2100,100.0000,100.0,0.0,0.0000,21.875



LAYER-LEVEL SUMMARY: FIRST 20 ROWS


,fault_type,layer,strategy,detection_rate_pct,correction_rate_pct,residual_error_rate_pct,undetected_error_rate_pct
0,random_weight_replacement,conv1,dmr,100.0,0.0,100.0,0.0
1,random_weight_replacement,conv1,none,0.0,0.0,100.0,100.0
2,random_weight_replacement,conv1,parity,51.0,0.0,100.0,49.0
3,random_weight_replacement,conv1,secded,99.0,0.0,100.0,1.0
4,random_weight_replacement,fc,dmr,100.0,0.0,100.0,0.0
5,random_weight_replacement,fc,none,0.0,0.0,100.0,100.0
6,random_weight_replacement,fc,parity,49.0,0.0,100.0,51.0
7,random_weight_replacement,fc,secded,95.0,0.0,100.0,5.0
8,random_weight_replacement,layer1.0.conv1,dmr,100.0,0.0,100.0,0.0
9,random_weight_replacement,layer1.0.conv1,none,0.0,0.0,100.0,100.0



SECDED STATUS DISTRIBUTION


,fault_type,secded_status,count
0,random_weight_replacement,clean,20
1,random_weight_replacement,corrected,617
2,random_weight_replacement,detected_double,1068
3,random_weight_replacement,detected_uncorrectable,395
4,single_bit_flip,corrected,2100



STORAGE OVERHEAD


,data_bits,stored_bits,additional_bits,overhead_pct,description
none,32,32,0,0.0,32-bit data word
parity,32,33,1,3.125,32-bit data + 1 parity bit
secded,32,39,7,21.875,32-bit data + 6 Hamming bits + 1 overall parit...
dmr,32,64,32,100.0,Two 32-bit copies; comparison detects mismatch



EXPERIMENT 6 COMPLETED
Elapsed time: 0.77 seconds
Layers profiled: 21
Raw strategy-trial records: 16800

Saved files:
  /kaggle/working/JFAMP/experiments/protection_profiling/protection_profiling_raw.csv (1828.86 KB)
  /kaggle/working/JFAMP/experiments/protection_profiling/protection_layer_summary.csv (14.65 KB)
  /kaggle/working/JFAMP/experiments/protection_profiling/protection_mechanism_summary.csv (0.87 KB)
  /kaggle/working/JFAMP/experiments/protection_profiling/protection_storage_overhead.json (0.67 KB)
  /kaggle/working/JFAMP/experiments/protection_profiling/protection_profiling_metadata.json (1.85 KB)

IMPORTANT:
This experiment measures codeword-level protection behavior.
It does not yet measure full-network accuracy after protection.
Keep all existing JFAMP experiment outputs unchanged.


In [18]:

# =====================================================================
# JFAMP EXPERIMENT 7
# JOINT MIXED-PRECISION + SELECTIVE-PROTECTION OPTIMIZATION
#
# Inputs:
#   Experiment 3: layerwise_quantization.csv
#   Experiment 5: fault_profiling_layer_summary.csv
#   Experiment 6: protection_layer_summary.csv
#   Original FP32 checkpoint and baseline
#
# Solver: scipy.optimize.milp
#
# Output:
#   One selected datatype + protection strategy per layer
#   Optimization report, candidate table, and solver metadata
#
# Important:
#   - Isolated-layer quantization loss is an additive screening proxy.
#   - FP32 fault sensitivity is used as a proxy across datatypes.
#   - Final selected configuration MUST be validated end-to-end.
# =====================================================================

import os
import re
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import lil_matrix

# =====================================================================
# 1. CONFIGURATION
# =====================================================================

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path("/kaggle/working/JFAMP")

QUANT_FILE = (
    ROOT / "experiments/layerwise_quantization/"
    "layerwise_quantization.csv"
)

FAULT_FILE = (
    ROOT / "experiments/fault_profiling/"
    "fault_profiling_layer_summary.csv"
)

PROTECTION_FILE = (
    ROOT / "experiments/protection_profiling/"
    "protection_layer_summary.csv"
)

BASELINE_FILE = ROOT / "retrain_to150/fp32_baseline.json"

OUTPUT_DIR = ROOT / "experiments/joint_optimization"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Initial design constraints: editable.
STORAGE_BUDGET_FRACTION = 0.60
MAX_QUANTIZATION_LOSS_PP = 1.0

# Relative importance of residual fault risk in the objective.
LAMBDA_FAULT = 1.0

# Assumed fault-model mixture.
# These are design assumptions, not measured occurrence probabilities.
FAULT_WEIGHTS = {
    "bitflip": 0.50,
    "random_corruption": 0.50,
}

# Candidate datatype widths.
# E4M11 = 1 sign + 4 exponent + 11 mantissa = 16 bits.
DATATYPES = {
    "FP32": 32,
    "FP16": 16,
    "BF16": 16,
    "INT8": 8,
    "INT4": 4,
    "FP8_E4M3FN": 8,
    "FP8_E5M2": 8,
    "E4M11_SUBNORMAL": 16,
}

PROTECTIONS = ["none", "parity", "secded", "dmr"]

print("=" * 76)
print("JFAMP EXPERIMENT 7: JOINT OPTIMIZATION")
print("=" * 76)

for path in [
    QUANT_FILE,
    FAULT_FILE,
    PROTECTION_FILE,
    BASELINE_FILE,
]:
    assert path.exists(), f"Required input file missing: {path}"
    print("Found:", path)

# =====================================================================
# 2. LOAD INPUT DATA
# =====================================================================

quant_df = pd.read_csv(QUANT_FILE)
fault_df = pd.read_csv(FAULT_FILE)
protection_df = pd.read_csv(PROTECTION_FILE)

with open(BASELINE_FILE, "r") as f:
    baseline_data = json.load(f)

# Support the key used in the existing baseline file.
BASELINE_ACCURACY = float(
    baseline_data.get(
        "validation_accuracy",
        baseline_data.get("accuracy", 77.78)
    )
)

print("\nBaseline validation accuracy:", BASELINE_ACCURACY)
print("\nInput dimensions:")
print("Quantization:", quant_df.shape)
print("Fault profiling:", fault_df.shape)
print("Protection profiling:", protection_df.shape)

# =====================================================================
# 3. COLUMN DISCOVERY HELPERS
# =====================================================================

def normalize_name(value):
    return re.sub(r"[^a-z0-9]", "", str(value).lower())

def find_column(df, candidates, required=True, description="column"):
    normalized = {
        normalize_name(col): col for col in df.columns
    }

    for candidate in candidates:
        key = normalize_name(candidate)
        if key in normalized:
            return normalized[key]

    if required:
        raise ValueError(
            f"Could not identify {description}.\n"
            f"Available columns: {list(df.columns)}\n"
            f"Expected one of: {candidates}"
        )

    return None

# =====================================================================
# 4. QUANTIZATION PROFILE
# =====================================================================

q_layer_col = find_column(
    quant_df,
    ["layer", "layer_name", "module"],
    description="quantization layer column"
)

q_format_col = find_column(
    quant_df,
    ["format", "datatype", "precision", "quantization_format",
     "data_type", "scheme"],
    description="quantization format column"
)

q_delta_col = find_column(
    quant_df,
    [
        "accuracy_delta_pp",
        "accuracy_change_pp",
        "delta_accuracy_pp",
        "delta_pp",
        "accuracy_delta",
        "accuracy_change",
        "accuracy_drop_pp",
        "accuracy_loss_pp",
    ],
    required=False,
    description="quantization accuracy delta column"
)

q_accuracy_col = find_column(
    quant_df,
    ["accuracy", "accuracy_pct", "validation_accuracy"],
    required=False,
    description="quantization accuracy column"
)

if q_delta_col is None and q_accuracy_col is None:
    raise ValueError(
        "Quantization CSV needs an accuracy delta or accuracy column."
    )

def normalize_format(value):
    s = normalize_name(value)

    aliases = {
        "fp32": "FP32",
        "float32": "FP32",
        "fp16": "FP16",
        "float16": "FP16",
        "half": "FP16",
        "bf16": "BF16",
        "bfloat16": "BF16",
        "int8": "INT8",
        "int4": "INT4",
        "fpe4m3fn": "FP8_E4M3FN",
        "fp8e4m3fn": "FP8_E4M3FN",
        "fp8e4m3": "FP8_E4M3FN",
        "fpe5m2": "FP8_E5M2",
        "fp8e5m2": "FP8_E5M2",
        "e4m11": "E4M11_SUBNORMAL",
        "e4m11subnormal": "E4M11_SUBNORMAL",
        "e4m11sub": "E4M11_SUBNORMAL",
    }

    return aliases.get(s, str(value).upper())

quant_df["_layer"] = quant_df[q_layer_col].astype(str)
quant_df["_format"] = quant_df[q_format_col].apply(normalize_format)

if q_delta_col is not None:
    quant_df["_raw_delta"] = pd.to_numeric(
        quant_df[q_delta_col], errors="coerce"
    )
    # Accuracy delta is measured relative to FP32:
    # positive = improvement, negative = degradation.
    quant_df["_quant_loss_pp"] = (
        -quant_df["_raw_delta"]
    ).clip(lower=0)

else:
    quant_df["_accuracy"] = pd.to_numeric(
        quant_df[q_accuracy_col], errors="coerce"
    )
    quant_df["_quant_loss_pp"] = (
        BASELINE_ACCURACY - quant_df["_accuracy"]
    ).clip(lower=0)

quant_df = quant_df.dropna(subset=["_quant_loss_pp"])

# Use the mean if duplicate layer-format records exist.
quant_profile = (
    quant_df.groupby(["_layer", "_format"], as_index=False)
    ["_quant_loss_pp"].mean()
)

quant_layers = sorted(quant_profile["_layer"].unique())

print("\nQuantization profile:")
print("Layers:", len(quant_layers))
print("Formats:", sorted(quant_profile["_format"].unique()))

# =====================================================================
# 5. FAULT PROFILE
# =====================================================================

f_layer_col = find_column(
    fault_df,
    ["layer", "layer_name", "module"],
    description="fault layer column"
)

f_mechanism_col = find_column(
    fault_df,
    ["mechanism", "fault_type", "fault_mechanism"],
    description="fault mechanism column"
)

f_degradation_col = find_column(
    fault_df,
    [
        "mean_layer_degradation",
        "mean_degradation_pp",
        "mean_accuracy_degradation_pp",
        "mean_accuracy_drop_pp",
        "mean_drop_pp",
        "mean_degradation",
        "mean_accuracy_degradation",
    ],
    description="mean fault degradation column"
)

fault_df["_layer"] = fault_df[f_layer_col].astype(str)

def normalize_fault_type(value):
    s = normalize_name(value)

    if "bitflip" in s or "bitflip" == s:
        return "bitflip"

    if (
        "randomcorruption" in s
        or "randomweightreplacement" in s
        or "randomreplacement" in s
    ):
        return "random_corruption"

    return s

fault_df["_fault"] = fault_df[f_mechanism_col].apply(
    normalize_fault_type
)

fault_df["_degradation"] = pd.to_numeric(
    fault_df[f_degradation_col],
    errors="coerce"
).fillna(0).clip(lower=0)

fault_profile = (
    fault_df.groupby(["_layer", "_fault"], as_index=False)
    ["_degradation"].mean()
)

# Require both mechanisms because the configured fault mixture uses both.
available_faults = set(fault_profile["_fault"].unique())

for required_fault in FAULT_WEIGHTS:
    if required_fault not in available_faults:
        raise ValueError(
            f"Fault profile does not contain {required_fault}. "
            f"Available mechanisms: {available_faults}"
        )

print("\nFault profile:")
print("Mechanisms:", sorted(available_faults))

# =====================================================================
# 6. PROTECTION PROFILE
# =====================================================================

p_layer_col = find_column(
    protection_df,
    ["layer", "layer_name", "module"],
    description="protection layer column"
)

p_fault_col = find_column(
    protection_df,
    ["fault_type", "mechanism", "fault_mechanism"],
    description="protection fault type column"
)

p_strategy_col = find_column(
    protection_df,
    ["strategy", "protection", "protection_strategy"],
    description="protection strategy column"
)

p_residual_col = find_column(
    protection_df,
    [
        "residual_error_rate_pct",
        "residual_errors_pct",
        "residual_error_pct",
    ],
    description="protection residual error column"
)

protection_df["_layer"] = protection_df[p_layer_col].astype(str)
protection_df["_fault"] = protection_df[p_fault_col].apply(
    normalize_fault_type
)
protection_df["_strategy"] = (
    protection_df[p_strategy_col]
    .astype(str).str.lower().str.strip()
)

protection_df["_residual"] = (
    pd.to_numeric(protection_df[p_residual_col], errors="coerce")
    .fillna(100)
    .clip(0, 100) / 100.0
)

protection_profile = (
    protection_df.groupby(
        ["_layer", "_fault", "_strategy"], as_index=False
    )["_residual"].mean()
)

# Validate all required strategies and mechanisms exist.
for strategy in PROTECTIONS:
    if strategy not in set(protection_profile["_strategy"]):
        raise ValueError(
            f"Protection strategy '{strategy}' missing from profile."
        )

# =====================================================================
# 7. BUILD LOOKUP TABLES
# =====================================================================

quant_lookup = {
    (row["_layer"], row["_format"]): float(row["_quant_loss_pp"])
    for _, row in quant_profile.iterrows()
}

fault_lookup = {
    (row["_layer"], row["_fault"]): float(row["_degradation"])
    for _, row in fault_profile.iterrows()
}

protection_lookup = {
    (row["_layer"], row["_fault"], row["_strategy"]):
        float(row["_residual"])
    for _, row in protection_profile.iterrows()
}

# The FP32 baseline is explicitly added as a zero-loss candidate.
for layer in quant_layers:
    quant_lookup[(layer, "FP32")] = 0.0

# Validate layer consistency.
fault_layers = set(fault_df["_layer"].unique())
protection_layers = set(protection_df["_layer"].unique())
quant_layer_set = set(quant_layers)

missing_fault_layers = quant_layer_set - fault_layers
missing_protection_layers = quant_layer_set - protection_layers

if missing_fault_layers or missing_protection_layers:
    raise ValueError(
        "Layer mismatch between profiling datasets.\n"
        f"Missing fault layers: {missing_fault_layers}\n"
        f"Missing protection layers: {missing_protection_layers}"
    )

# =====================================================================
# 8. STORAGE MODEL
# =====================================================================

def secded_parity_bits(data_bits):
    """
    Minimum Hamming parity bits r satisfying:
        2^r >= data_bits + r + 1
    Plus one overall parity bit.
    """
    r = 0
    while (2 ** r) < (data_bits + r + 1):
        r += 1
    return r + 1

def stored_bits_per_weight(data_bits, protection):
    if protection == "none":
        return data_bits

    if protection == "parity":
        return data_bits + 1

    if protection == "secded":
        return data_bits + secded_parity_bits(data_bits)

    if protection == "dmr":
        return 2 * data_bits

    raise ValueError(protection)

# Count weights per layer from quantization inventory.
# The layer-wise quantization profile has one row per layer-format.
# The weight count is obtained from the canonical checkpoint.
CHECKPOINT = (
    ROOT / "retrain_to150/checkpoints/resnet18_cifar100_best_150.pth"
)

assert CHECKPOINT.exists()

checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

def get_state_dict(obj):
    if isinstance(obj, dict):
        for key in [
            "state_dict", "model_state_dict", "model", "net", "weights"
        ]:
            candidate = obj.get(key)
            if isinstance(candidate, dict) and candidate:
                if any(torch.is_tensor(v) for v in candidate.values()):
                    return candidate

        if any(torch.is_tensor(v) for v in obj.values()):
            return obj

    raise RuntimeError("Unable to extract checkpoint state_dict.")

state = get_state_dict(checkpoint)

weight_counts = {}

for key, tensor in state.items():
    name = key[7:] if key.startswith("module.") else key

    if (
        name.endswith(".weight")
        and torch.is_tensor(tensor)
        and tensor.ndim in (2, 4)
    ):
        layer_name = name.rsplit(".weight", 1)[0]
        weight_counts[layer_name] = tensor.numel()

missing_counts = quant_layer_set - set(weight_counts)

if missing_counts:
    raise ValueError(
        f"Could not find parameter counts for layers: {missing_counts}"
    )

TOTAL_FP32_BITS = sum(
    weight_counts[layer] * 32 for layer in quant_layers
)

STORAGE_BUDGET_BITS = (
    STORAGE_BUDGET_FRACTION * TOTAL_FP32_BITS
)

# =====================================================================
# 9. BUILD LAYER-DATATYPE-PROTECTION CANDIDATES
# =====================================================================

candidates = []

for layer in quant_layers:

    for datatype, bits in DATATYPES.items():

        q_key = (layer, datatype)

        if q_key not in quant_lookup:
            continue

        quant_loss = max(0.0, quant_lookup[q_key])

        for protection in PROTECTIONS:

            # Layer-specific fault risk, using measured FP32 fault
            # sensitivity and measured protection residual-error rates.
            residual_risk = 0.0

            for fault_type, fault_weight in FAULT_WEIGHTS.items():

                fault_key = (layer, fault_type)

                if fault_key not in fault_lookup:
                    raise ValueError(
                        f"Missing fault sensitivity for {fault_key}"
                    )

                protection_key = (
                    layer, fault_type, protection
                )

                if protection_key not in protection_lookup:
                    raise ValueError(
                        f"Missing protection profile for "
                        f"{protection_key}"
                    )

                baseline_fault_degradation = fault_lookup[fault_key]

                residual_fraction = protection_lookup[
                    protection_key
                ]

                residual_risk += (
                    fault_weight
                    * baseline_fault_degradation
                    * residual_fraction
                )

            storage_per_weight = stored_bits_per_weight(
                bits, protection
            )

            total_storage = (
                weight_counts[layer] * storage_per_weight
            )

            candidates.append({
                "layer": layer,
                "datatype": datatype,
                "datatype_bits": bits,
                "protection": protection,
                "quantization_loss_pp": quant_loss,
                "residual_fault_risk_pp": residual_risk,
                "storage_bits_per_weight": storage_per_weight,
                "total_storage_bits": total_storage,
                "objective_cost": (
                    quant_loss + LAMBDA_FAULT * residual_risk
                )
            })

candidate_df = pd.DataFrame(candidates)

assert not candidate_df.empty

print("\nCandidate construction:")
print("Layers:", len(quant_layers))
print("Candidates:", len(candidate_df))
print("Candidate combinations per layer:")
print(candidate_df.groupby("layer").size().value_counts().to_dict())

# =====================================================================
# 10. BUILD MILP
# =====================================================================

# Binary variable x_i:
# 1 = select candidate i
# 0 = do not select candidate i
#
# Objective:
# min sum_i x_i * (Q_i + lambda * R_i)
#
# Constraints:
# 1. Exactly one candidate per layer.
# 2. Total storage <= storage budget.
# 3. Sum of isolated-layer quantization losses <= loss budget.

n_variables = len(candidate_df)
layers = quant_layers

layer_to_indices = {
    layer: candidate_df.index[
        candidate_df["layer"] == layer
    ].tolist()
    for layer in layers
}

# Constraint rows:
# first N rows: one assignment per layer
# next row: storage budget
# last row: quantization-loss budget

n_constraints = len(layers) + 2

A = lil_matrix((n_constraints, n_variables), dtype=float)

lower = np.full(n_constraints, -np.inf)
upper = np.full(n_constraints, np.inf)

# Exactly one choice per layer.
for row, layer in enumerate(layers):
    for idx in layer_to_indices[layer]:
        A[row, idx] = 1.0

    lower[row] = 1.0
    upper[row] = 1.0

storage_row = len(layers)
quant_row = len(layers) + 1

A[storage_row, :] = candidate_df[
    "total_storage_bits"
].to_numpy(dtype=float)

lower[storage_row] = -np.inf
upper[storage_row] = STORAGE_BUDGET_BITS

A[quant_row, :] = candidate_df[
    "quantization_loss_pp"
].to_numpy(dtype=float)

lower[quant_row] = -np.inf
upper[quant_row] = MAX_QUANTIZATION_LOSS_PP

objective = candidate_df[
    "objective_cost"
].to_numpy(dtype=float)

integrality = np.ones(n_variables, dtype=int)

bounds = Bounds(
    lb=np.zeros(n_variables),
    ub=np.ones(n_variables)
)

constraints = LinearConstraint(
    A.tocsr(),
    lb=lower,
    ub=upper
)

# =====================================================================
# 11. SOLVE
# =====================================================================

print("\n" + "=" * 76)
print("RUNNING MILP OPTIMIZATION")
print("=" * 76)

print("Variables:", n_variables)
print("Constraints:", n_constraints)
print("Storage budget (bits):", int(STORAGE_BUDGET_BITS))
print("Storage budget fraction:", STORAGE_BUDGET_FRACTION)
print("Quantization-loss budget (pp):", MAX_QUANTIZATION_LOSS_PP)
print("Fault-risk weight lambda:", LAMBDA_FAULT)

start_time = time.time()

result = milp(
    c=objective,
    integrality=integrality,
    bounds=bounds,
    constraints=constraints,
    options={
        "time_limit": 120,
        "mip_rel_gap": 0.001,
        "disp": False
    }
)

solver_time = time.time() - start_time

print("\nSolver message:", result.message)
print("Solver success:", result.success)
print("Solver status:", result.status)
print("Solver time:", round(solver_time, 3), "seconds")

if result.x is None:
    raise RuntimeError(
        "No feasible optimization solution was found.\n"
        "Do not silently relax the constraints. Inspect candidate "
        "coverage and consider changing the storage or accuracy budget."
    )

# =====================================================================
# 12. EXTRACT SELECTED CONFIGURATION
# =====================================================================

selected = candidate_df.loc[
    result.x > 0.5
].copy().reset_index(drop=True)

assert len(selected) == len(layers), (
    "Expected exactly one selected configuration per layer."
)

assert selected["layer"].nunique() == len(layers)

selected = selected.sort_values("layer").reset_index(drop=True)

total_selected_storage = selected["total_storage_bits"].sum()
total_selected_quant_loss = selected["quantization_loss_pp"].sum()
total_selected_fault_risk = selected["residual_fault_risk_pp"].sum()

storage_fraction = (
    total_selected_storage / TOTAL_FP32_BITS
)

storage_reduction = 100 * (1 - storage_fraction)

# =====================================================================
# 13. VALIDATION OF OPTIMIZATION CONSTRAINTS
# =====================================================================

assert total_selected_storage <= STORAGE_BUDGET_BITS + 1e-4
assert total_selected_quant_loss <= MAX_QUANTIZATION_LOSS_PP + 1e-4

for layer in layers:
    assert (selected["layer"] == layer).sum() == 1

# =====================================================================
# 14. GENERATE REPORTS
# =====================================================================

selected["storage_fraction_of_fp32"] = (
    selected["total_storage_bits"] / TOTAL_FP32_BITS
)

selected["estimated_storage_reduction_pct"] = (
    100 * (
        1 - selected["storage_bits_per_weight"] / 32
    )
)

summary_by_datatype = (
    selected.groupby("datatype", as_index=False)
    .agg(
        layers=("layer", "count"),
        total_weights=("layer", lambda x: sum(
            weight_counts[layer] for layer in x
        )),
        quantization_loss_proxy_pp=(
            "quantization_loss_pp", "sum"
        ),
        residual_fault_risk_proxy_pp=(
            "residual_fault_risk_pp", "sum"
        ),
        storage_bits=("total_storage_bits", "sum")
    )
)

summary_by_protection = (
    selected.groupby("protection", as_index=False)
    .agg(
        layers=("layer", "count"),
        quantization_loss_proxy_pp=(
            "quantization_loss_pp", "sum"
        ),
        residual_fault_risk_proxy_pp=(
            "residual_fault_risk_pp", "sum"
        ),
        storage_bits=("total_storage_bits", "sum")
    )
)

# Save all candidate combinations for auditability.
candidate_path = OUTPUT_DIR / "all_optimization_candidates.csv"
selected_path = OUTPUT_DIR / "optimized_layer_configuration.csv"
datatype_path = OUTPUT_DIR / "optimization_datatype_summary.csv"
protection_path = OUTPUT_DIR / "optimization_protection_summary.csv"
report_path = OUTPUT_DIR / "optimization_report.json"

candidate_df.to_csv(candidate_path, index=False)
selected.to_csv(selected_path, index=False)
summary_by_datatype.to_csv(datatype_path, index=False)
summary_by_protection.to_csv(protection_path, index=False)

report = {
    "experiment": "JFAMP Experiment 7 - Joint Optimization",
    "solver": "scipy.optimize.milp",
    "solver_message": str(result.message),
    "solver_success": bool(result.success),
    "solver_status": int(result.status),
    "solver_time_seconds": float(solver_time),
    "layers_optimized": int(len(layers)),
    "candidate_count": int(n_variables),
    "baseline_accuracy_pct": BASELINE_ACCURACY,
    "storage_budget_fraction": STORAGE_BUDGET_FRACTION,
    "storage_budget_bits": float(STORAGE_BUDGET_BITS),
    "selected_storage_bits": float(total_selected_storage),
    "selected_storage_fraction_of_fp32": float(storage_fraction),
    "estimated_storage_reduction_pct": float(storage_reduction),
    "quantization_loss_budget_pp": MAX_QUANTIZATION_LOSS_PP,
    "selected_quantization_loss_proxy_pp": float(
        total_selected_quant_loss
    ),
    "selected_residual_fault_risk_proxy_pp": float(
        total_selected_fault_risk
    ),
    "lambda_fault": LAMBDA_FAULT,
    "fault_weights": FAULT_WEIGHTS,
    "objective_value": float(result.fun),
    "methodological_assumptions": [
        "Quantization loss is derived from isolated-layer validation experiments.",
        "The sum of isolated-layer losses is a screening proxy, not a prediction of final joint model accuracy.",
        "FP32 layer-wise fault sensitivity is used as a proxy across datatypes.",
        "Protection residual rates are taken from the FP32 codeword-level protection experiment.",
        "Parity and DMR detection are not treated as correction.",
        "SECDED correction is credited only where the measured protection profile reports zero residual data error.",
        "Datatype-specific fault behavior has not yet been independently measured.",
        "Storage calculations cover weights and protection bits only.",
        "Alignment, scales, metadata, activation storage, controller logic and physical hardware overhead are excluded.",
        "The selected configuration requires end-to-end validation."
    ],
    "output_files": [
        str(candidate_path),
        str(selected_path),
        str(datatype_path),
        str(protection_path),
        str(report_path)
    ]
}

with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

# =====================================================================
# 15. DISPLAY FINAL RESULTS
# =====================================================================

print("\n" + "=" * 76)
print("OPTIMIZATION RESULT")
print("=" * 76)

display(
    selected[
        [
            "layer",
            "datatype",
            "protection",
            "datatype_bits",
            "quantization_loss_pp",
            "residual_fault_risk_pp",
            "storage_bits_per_weight"
        ]
    ].round(5)
)

print("\nDATATYPE DISTRIBUTION")
display(summary_by_datatype.round(4))

print("\nPROTECTION DISTRIBUTION")
display(summary_by_protection.round(4))

print("\n" + "=" * 76)
print("FINAL OPTIMIZATION METRICS")
print("=" * 76)

print(f"Layers optimized: {len(layers)}")
print(f"Candidate combinations: {n_variables}")
print(f"FP32 weight storage: {TOTAL_FP32_BITS / 8 / 1024 / 1024:.3f} MiB")
print(
    f"Selected theoretical storage: "
    f"{total_selected_storage / 8 / 1024 / 1024:.3f} MiB"
)
print(f"Storage reduction: {storage_reduction:.2f}%")
print(
    f"Storage budget used: "
    f"{100 * storage_fraction / STORAGE_BUDGET_FRACTION:.2f}%"
    if STORAGE_BUDGET_FRACTION > 0 else "N/A"
)
print(
    f"Quantization loss proxy: "
    f"{total_selected_quant_loss:.5f} pp"
)
print(
    f"Residual fault risk proxy: "
    f"{total_selected_fault_risk:.5f} pp"
)
print(f"Objective value: {result.fun:.6f}")

print("\nSaved outputs:")
for path in [
    candidate_path,
    selected_path,
    datatype_path,
    protection_path,
    report_path
]:
    print(f"  {path} ({path.stat().st_size / 1024:.2f} KB)")

print("\n" + "=" * 76)
print("EXPERIMENT 7 COMPLETED")
print("=" * 76)
print(
    "IMPORTANT: This is the optimizer's selected candidate configuration. "
    "It is not yet a validated end-to-end ResNet-18 result."
)
print(
    "The next experiment must apply the selected datatypes and protection "
    "schemes to the complete model and measure actual validation accuracy."
)


JFAMP EXPERIMENT 7: JOINT OPTIMIZATION
Found: /kaggle/working/JFAMP/experiments/layerwise_quantization/layerwise_quantization.csv
Found: /kaggle/working/JFAMP/experiments/fault_profiling/fault_profiling_layer_summary.csv
Found: /kaggle/working/JFAMP/experiments/protection_profiling/protection_layer_summary.csv
Found: /kaggle/working/JFAMP/retrain_to150/fp32_baseline.json

Baseline validation accuracy: 77.78

Input dimensions:
Quantization: (147, 22)
Fault profiling: (42, 13)
Protection profiling: (168, 14)

Quantization profile:
Layers: 21
Formats: ['BF16', 'E4M11_SUBNORMAL', 'FP16', 'FP8_E4M3FN', 'FP8_E5M2', 'INT4', 'INT8']

Fault profile:
Mechanisms: ['bitflip', 'random_corruption']

Candidate construction:
Layers: 21
Candidates: 672
Candidate combinations per layer:
{32: 21}

RUNNING MILP OPTIMIZATION
Variables: 672
Constraints: 23
Storage budget (bits): 215240294
Storage budget fraction: 0.6
Quantization-loss budget (pp): 1.0
Fault-risk weight lambda: 1.0

Solver message: Optimizat

,layer,datatype,protection,datatype_bits,quantization_loss_pp,residual_fault_risk_pp,storage_bits_per_weight
0,conv1,INT8,secded,8,0.0,0.0282,13
1,fc,FP16,secded,16,0.0,0.0006,22
2,layer1.0.conv1,INT8,none,8,0.0,0.0000,8
3,layer1.0.conv2,FP8_E4M3FN,secded,8,0.0,0.0003,13
4,layer1.1.conv1,FP8_E5M2,secded,8,0.0,0.0000,13
5,layer1.1.conv2,INT8,secded,8,0.0,0.0006,13
6,layer2.0.conv1,FP16,secded,16,0.0,0.0000,22
7,layer2.0.conv2,FP8_E4M3FN,secded,8,0.0,0.0008,13
8,layer2.0.downsample.0,FP16,secded,16,0.0,0.0000,22
9,layer2.1.conv1,INT8,secded,8,0.0,0.0000,13



DATATYPE DISTRIBUTION


,datatype,layers,total_weights,quantization_loss_proxy_pp,residual_fault_risk_proxy_pp,storage_bits
0,FP16,6,2197504,0.0,0.0007,48345088
1,FP8_E4M3FN,3,2543616,0.0,0.0011,33067008
2,FP8_E5M2,3,659456,0.0,0.0000,8572928
3,INT4,1,131072,0.0,0.0002,1048576
4,INT8,8,5678784,0.0,0.0289,73639872



PROTECTION DISTRIBUTION


,protection,layers,quantization_loss_proxy_pp,residual_fault_risk_proxy_pp,storage_bits
0,none,1,0.0,0.0000,294912
1,secded,20,0.0,0.0309,164378560



FINAL OPTIMIZATION METRICS
Layers optimized: 21
Candidate combinations: 672
FP32 weight storage: 42.764 MiB
Selected theoretical storage: 19.631 MiB
Storage reduction: 54.10%
Storage budget used: 76.51%
Quantization loss proxy: 0.00000 pp
Residual fault risk proxy: 0.03090 pp
Objective value: 0.030900

Saved outputs:
  /kaggle/working/JFAMP/experiments/joint_optimization/all_optimization_candidates.csv (52.46 KB)
  /kaggle/working/JFAMP/experiments/joint_optimization/optimized_layer_configuration.csv (2.10 KB)
  /kaggle/working/JFAMP/experiments/joint_optimization/optimization_datatype_summary.csv (0.31 KB)
  /kaggle/working/JFAMP/experiments/joint_optimization/optimization_protection_summary.csv (0.15 KB)
  /kaggle/working/JFAMP/experiments/joint_optimization/optimization_report.json (2.22 KB)

EXPERIMENT 7 COMPLETED
IMPORTANT: This is the optimizer's selected candidate configuration. It is not yet a validated end-to-end ResNet-18 result.
The next experiment must apply the selected d

In [19]:

# =====================================================================
# JFAMP EXPERIMENT 7B: CATASTROPHIC-FAULT-AWARE OPTIMIZATION
#
# Refinement of the original MILP solution.
# Does NOT rerun quantization, fault injection, or protection experiments.
# Does NOT overwrite the original Experiment 7 outputs.
#
# New term:
#   Catastrophic risk = P(non-finite fault) * 100 pp
#                      * measured residual-error fraction
#
# Objective:
#   min sum x * (Q + lambda_fault*R + lambda_catastrophic*C)
#
# Constraints:
#   - Exactly one candidate per layer
#   - Same storage budget as Experiment 7
#   - Same quantization-loss budget as Experiment 7
# =====================================================================

import json
import time
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import lil_matrix

# -------------------- 1. CONFIGURATION -----------------------------

ROOT = Path("/kaggle/working/JFAMP")

OLD_DIR = ROOT / "experiments/joint_optimization"
FAULT_FILE = (
    ROOT / "experiments/fault_profiling/"
    "fault_profiling_layer_summary.csv"
)
PROTECTION_FILE = (
    ROOT / "experiments/protection_profiling/"
    "protection_layer_summary.csv"
)

CANDIDATE_FILE = OLD_DIR / "all_optimization_candidates.csv"
OLD_SELECTED_FILE = OLD_DIR / "optimized_layer_configuration.csv"
OLD_REPORT_FILE = OLD_DIR / "optimization_report.json"

OUT_DIR = ROOT / "experiments/joint_optimization_refined"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Preserve original Experiment 7 design constraints.
STORAGE_BUDGET_FRACTION = 0.60
MAX_QUANTIZATION_LOSS_PP = 1.0

LAMBDA_FAULT = 1.0
LAMBDA_CATASTROPHIC = 1.0

FAULT_WEIGHTS = {
    "bitflip": 0.50,
    "random_corruption": 0.50,
}

# Conservative severity assumption for a non-finite trial.
CATASTROPHIC_SEVERITY_PP = 100.0

print("=" * 76)
print("JFAMP EXPERIMENT 7B: CATASTROPHIC-FAULT-AWARE OPTIMIZATION")
print("=" * 76)

for p in [
    CANDIDATE_FILE,
    OLD_SELECTED_FILE,
    OLD_REPORT_FILE,
    FAULT_FILE,
    PROTECTION_FILE,
]:
    assert p.exists(), f"Missing input: {p}"
    print("Found:", p)

# -------------------- 2. LOAD EXISTING PROFILES -------------------

candidates = pd.read_csv(CANDIDATE_FILE)
old_selected = pd.read_csv(OLD_SELECTED_FILE)
fault_df = pd.read_csv(FAULT_FILE)
protection_df = pd.read_csv(PROTECTION_FILE)

with open(OLD_REPORT_FILE, "r") as f:
    old_report = json.load(f)

print("\nLoaded:")
print("Candidates:", candidates.shape)
print("Previous solution:", old_selected.shape)
print("Fault summary:", fault_df.shape)
print("Protection summary:", protection_df.shape)

# -------------------- 3. COLUMN HELPERS ---------------------------

def norm(value):
    return re.sub(r"[^a-z0-9]", "", str(value).lower())

def get_col(df, names, required=True):
    lookup = {norm(c): c for c in df.columns}

    for name in names:
        if norm(name) in lookup:
            return lookup[norm(name)]

    if required:
        raise ValueError(
            f"Missing expected column {names}. "
            f"Available: {list(df.columns)}"
        )
    return None

def normalize_fault(value):
    s = norm(value)

    if "bitflip" in s:
        return "bitflip"

    if (
        "randomcorruption" in s
        or "randomweightreplacement" in s
        or "randomreplacement" in s
    ):
        return "random_corruption"

    return s

# -------------------- 4. EXTRACT CATASTROPHIC RATES ----------------

f_layer = get_col(fault_df, ["layer", "layer_name"])
f_mechanism = get_col(
    fault_df, ["mechanism", "fault_type", "fault_mechanism"]
)
f_failed = get_col(
    fault_df, ["failed_trials", "catastrophic_trials"]
)

f_total = get_col(
    fault_df,
    ["trials", "total_trials"],
    required=False
)

f_success = get_col(
    fault_df,
    ["successful_trials"],
    required=False
)

fault_df["_layer"] = fault_df[f_layer].astype(str)
fault_df["_fault"] = fault_df[f_mechanism].apply(normalize_fault)

fault_df["_failed"] = pd.to_numeric(
    fault_df[f_failed], errors="coerce"
).fillna(0)

if f_total is not None:
    fault_df["_total"] = pd.to_numeric(
        fault_df[f_total], errors="coerce"
    )
elif f_success is not None:
    fault_df["_total"] = (
        pd.to_numeric(
            fault_df[f_success], errors="coerce"
        ).fillna(0)
        + fault_df["_failed"]
    )
else:
    raise ValueError(
        "Cannot determine total fault trials. "
        "Expected trials or successful_trials in fault summary."
    )

if fault_df["_total"].isna().any():
    raise ValueError("Missing total trial counts.")

if (fault_df["_failed"] > fault_df["_total"]).any():
    raise ValueError("Failed trial count exceeds total trial count.")

fault_df["_catastrophic_rate"] = (
    fault_df["_failed"] / fault_df["_total"].replace(0, np.nan)
).fillna(0).clip(0, 1)

cat_profile = (
    fault_df.groupby(["_layer", "_fault"], as_index=False)
    .agg(
        failed_trials=("_failed", "sum"),
        total_trials=("_total", "sum")
    )
)

cat_profile["catastrophic_rate"] = (
    cat_profile["failed_trials"]
    / cat_profile["total_trials"].replace(0, np.nan)
).fillna(0)

cat_lookup = {
    (row["_layer"], row["_fault"]):
        float(row["catastrophic_rate"])
    for _, row in cat_profile.iterrows()
}

print("\nCatastrophic-fault summary:")
display(cat_profile.round(5))

print(
    "\nTotal failed trials:",
    int(cat_profile["failed_trials"].sum())
)
print(
    "Total trials:",
    int(cat_profile["total_trials"].sum())
)

# -------------------- 5. EXTRACT PROTECTION RESIDUAL RATES --------

p_layer = get_col(protection_df, ["layer", "layer_name"])
p_fault = get_col(
    protection_df, ["fault_type", "mechanism", "fault_mechanism"]
)
p_strategy = get_col(
    protection_df, ["strategy", "protection", "protection_strategy"]
)
p_residual = get_col(
    protection_df,
    [
        "residual_error_rate_pct",
        "residual_errors_pct",
        "residual_error_pct"
    ]
)

protection_df["_layer"] = protection_df[p_layer].astype(str)
protection_df["_fault"] = protection_df[p_fault].apply(
    normalize_fault
)
protection_df["_strategy"] = (
    protection_df[p_strategy].astype(str).str.lower().str.strip()
)

protection_df["_residual_fraction"] = (
    pd.to_numeric(
        protection_df[p_residual], errors="coerce"
    ).fillna(100).clip(0, 100) / 100.0
)

protection_lookup = {
    (row["_layer"], row["_fault"], row["_strategy"]):
        float(row["_residual_fraction"])
    for _, row in protection_df.iterrows()
}

# -------------------- 6. VALIDATE CANDIDATE DATA -------------------

required_candidate_cols = [
    "layer",
    "datatype",
    "protection",
    "quantization_loss_pp",
    "residual_fault_risk_pp",
    "total_storage_bits",
    "objective_cost",
]

for col in required_candidate_cols:
    if col not in candidates.columns:
        raise ValueError(
            f"Candidate table is missing required column: {col}"
        )

candidates["layer"] = candidates["layer"].astype(str)
candidates["protection"] = (
    candidates["protection"].astype(str).str.lower().str.strip()
)

layers = sorted(candidates["layer"].unique())

assert len(layers) == 21, (
    f"Expected 21 layers, found {len(layers)}"
)

assert len(candidates) == 672, (
    f"Expected 672 candidates, found {len(candidates)}"
)

assert all(
    candidates.groupby("layer").size() == 32
), "Candidate count per layer is not 32."

# -------------------- 7. COMPUTE CATASTROPHIC-RISK COST -----------

cat_costs = []

for _, row in candidates.iterrows():

    layer = row["layer"]
    strategy = row["protection"]

    expected_catastrophic_loss = 0.0

    for fault_type, fault_weight in FAULT_WEIGHTS.items():

        cat_key = (layer, fault_type)

        if cat_key not in cat_lookup:
            raise ValueError(
                f"Missing catastrophic profile for {cat_key}"
            )

        protection_key = (
            layer, fault_type, strategy
        )

        if protection_key not in protection_lookup:
            raise ValueError(
                f"Missing protection residual profile: "
                f"{protection_key}"
            )

        probability = cat_lookup[cat_key]

        residual_fraction = protection_lookup[
            protection_key
        ]

        expected_catastrophic_loss += (
            fault_weight
            * probability
            * CATASTROPHIC_SEVERITY_PP
            * residual_fraction
        )

    cat_costs.append(expected_catastrophic_loss)

candidates["catastrophic_risk_pp"] = cat_costs

candidates["refined_objective_cost"] = (
    candidates["quantization_loss_pp"]
    + LAMBDA_FAULT * candidates["residual_fault_risk_pp"]
    + LAMBDA_CATASTROPHIC
      * candidates["catastrophic_risk_pp"]
)

print("\nCatastrophic-risk cost added.")
print(
    "Maximum candidate catastrophic-risk cost:",
    round(candidates["catastrophic_risk_pp"].max(), 6),
    "pp"
)

# -------------------- 8. BUILD REFINED MILP ------------------------

n = len(candidates)
n_layers = len(layers)

A = lil_matrix((n_layers + 2, n), dtype=float)

lower = np.full(n_layers + 2, -np.inf)
upper = np.full(n_layers + 2, np.inf)

# Exactly one assignment per layer.
for i, layer in enumerate(layers):
    indices = candidates.index[
        candidates["layer"] == layer
    ].tolist()

    for idx in indices:
        A[i, idx] = 1.0

    lower[i] = 1.0
    upper[i] = 1.0

# Storage constraint.
storage_row = n_layers
A[storage_row, :] = candidates[
    "total_storage_bits"
].to_numpy(dtype=float)

storage_budget = float(
    old_report["storage_budget_bits"]
)

lower[storage_row] = -np.inf
upper[storage_row] = storage_budget

# Quantization-loss constraint.
quant_row = n_layers + 1
A[quant_row, :] = candidates[
    "quantization_loss_pp"
].to_numpy(dtype=float)

lower[quant_row] = -np.inf
upper[quant_row] = MAX_QUANTIZATION_LOSS_PP

objective = candidates[
    "refined_objective_cost"
].to_numpy(dtype=float)

constraints = LinearConstraint(
    A.tocsr(),
    lower,
    upper
)

bounds = Bounds(
    np.zeros(n),
    np.ones(n)
)

integrality = np.ones(n, dtype=int)

# -------------------- 9. SOLVE REFINED OPTIMIZATION ----------------

print("\n" + "=" * 76)
print("RUNNING REFINED MILP")
print("=" * 76)

print("Variables:", n)
print("Layers:", n_layers)
print("Storage budget:", storage_budget)
print("Quantization-loss budget:", MAX_QUANTIZATION_LOSS_PP)
print("Lambda fault:", LAMBDA_FAULT)
print("Lambda catastrophic:", LAMBDA_CATASTROPHIC)
print("Catastrophic severity assumption:", CATASTROPHIC_SEVERITY_PP)

start = time.time()

result = milp(
    c=objective,
    integrality=integrality,
    bounds=bounds,
    constraints=constraints,
    options={
        "time_limit": 120,
        "mip_rel_gap": 0.001,
        "disp": False
    }
)

solver_time = time.time() - start

print("\nSolver message:", result.message)
print("Solver success:", result.success)
print("Solver status:", result.status)
print("Solver time:", round(solver_time, 4), "seconds")

if result.x is None:
    raise RuntimeError(
        "Refined optimization did not return a feasible solution."
    )

# -------------------- 10. EXTRACT REFINED SOLUTION ----------------

refined = candidates.loc[
    result.x > 0.5
].copy().reset_index(drop=True)

assert len(refined) == 21
assert refined["layer"].nunique() == 21

refined = refined.sort_values("layer").reset_index(drop=True)

# Validate constraints.
total_storage = refined["total_storage_bits"].sum()
total_quant_loss = refined["quantization_loss_pp"].sum()
total_fault_risk = refined["residual_fault_risk_pp"].sum()
total_cat_risk = refined["catastrophic_risk_pp"].sum()

assert total_storage <= storage_budget + 1e-4
assert total_quant_loss <= MAX_QUANTIZATION_LOSS_PP + 1e-4

# -------------------- 11. COMPARE WITH ORIGINAL SOLUTION -----------

comparison = old_selected[
    ["layer", "datatype", "protection"]
].merge(
    refined[
        [
            "layer",
            "datatype",
            "protection",
            "catastrophic_risk_pp",
            "refined_objective_cost"
        ]
    ],
    on="layer",
    how="outer",
    suffixes=("_original", "_refined")
)

comparison["changed"] = (
    (comparison["datatype_original"] != comparison["datatype"])
    | (comparison["protection_original"] != comparison["protection"])
)

print("\n" + "=" * 76)
print("ORIGINAL VS REFINED CONFIGURATION")
print("=" * 76)

display(comparison)

print(
    "\nChanged layer assignments:",
    int(comparison["changed"].sum()),
    "out of",
    len(comparison)
)

# -------------------- 12. SUMMARIES --------------------------------

datatype_summary = (
    refined.groupby("datatype", as_index=False)
    .agg(
        layers=("layer", "count"),
        storage_bits=("total_storage_bits", "sum"),
        quantization_loss_pp=("quantization_loss_pp", "sum"),
        fault_risk_pp=("residual_fault_risk_pp", "sum"),
        catastrophic_risk_pp=("catastrophic_risk_pp", "sum")
    )
)

protection_summary = (
    refined.groupby("protection", as_index=False)
    .agg(
        layers=("layer", "count"),
        storage_bits=("total_storage_bits", "sum"),
        quantization_loss_pp=("quantization_loss_pp", "sum"),
        fault_risk_pp=("residual_fault_risk_pp", "sum"),
        catastrophic_risk_pp=("catastrophic_risk_pp", "sum")
    )
)

# -------------------- 13. SAVE WITHOUT OVERWRITING ----------------

selected_path = OUT_DIR / "refined_layer_configuration.csv"
comparison_path = OUT_DIR / "original_vs_refined_comparison.csv"
candidates_path = OUT_DIR / "refined_all_candidates.csv"
datatype_path = OUT_DIR / "refined_datatype_summary.csv"
protection_path = OUT_DIR / "refined_protection_summary.csv"
cat_path = OUT_DIR / "catastrophic_risk_profile.csv"
report_path = OUT_DIR / "refined_optimization_report.json"

refined.to_csv(selected_path, index=False)
comparison.to_csv(comparison_path, index=False)
candidates.to_csv(candidates_path, index=False)
datatype_summary.to_csv(datatype_path, index=False)
protection_summary.to_csv(protection_path, index=False)
cat_profile.to_csv(cat_path, index=False)

report = {
    "experiment": "JFAMP Experiment 7B - Catastrophic Risk Refinement",
    "solver": "scipy.optimize.milp",
    "solver_message": str(result.message),
    "solver_success": bool(result.success),
    "solver_status": int(result.status),
    "solver_time_seconds": float(solver_time),
    "layers": int(len(layers)),
    "candidate_count": int(n),
    "changed_assignments": int(comparison["changed"].sum()),
    "storage_budget_bits": float(storage_budget),
    "selected_storage_bits": float(total_storage),
    "quantization_loss_budget_pp": float(MAX_QUANTIZATION_LOSS_PP),
    "selected_quantization_loss_proxy_pp": float(total_quant_loss),
    "selected_fault_risk_proxy_pp": float(total_fault_risk),
    "selected_catastrophic_risk_proxy_pp": float(total_cat_risk),
    "lambda_fault": float(LAMBDA_FAULT),
    "lambda_catastrophic": float(LAMBDA_CATASTROPHIC),
    "catastrophic_severity_assumption_pp": float(
        CATASTROPHIC_SEVERITY_PP
    ),
    "fault_weights": FAULT_WEIGHTS,
    "objective_value": float(result.fun),
    "methodological_notes": [
        "Uses previously measured fault and protection profiles.",
        "Does not rerun fault injection or quantization experiments.",
        "Catastrophic rate is failed/non-finite trials divided by total trials.",
        "A non-finite trial is assigned a conservative 100 percentage-point severity.",
        "Catastrophic risk is weighted by the measured protection residual-error fraction.",
        "This is a risk proxy, not measured post-protection network accuracy.",
        "The original Experiment 7 output directory remains unchanged.",
        "End-to-end validation is still required."
    ],
    "output_files": [
        str(selected_path),
        str(comparison_path),
        str(candidates_path),
        str(datatype_path),
        str(protection_path),
        str(cat_path),
        str(report_path)
    ]
}

with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

# -------------------- 14. DISPLAY FINAL RESULTS -------------------

print("\n" + "=" * 76)
print("REFINED LAYER CONFIGURATION")
print("=" * 76)

display(
    refined[
        [
            "layer",
            "datatype",
            "protection",
            "quantization_loss_pp",
            "residual_fault_risk_pp",
            "catastrophic_risk_pp",
            "total_storage_bits"
        ]
    ].round(6)
)

print("\nDATATYPE DISTRIBUTION")
display(datatype_summary.round(6))

print("\nPROTECTION DISTRIBUTION")
display(protection_summary.round(6))

print("\n" + "=" * 76)
print("REFINED OPTIMIZATION METRICS")
print("=" * 76)

print("Solver:", result.message)
print("Changed layer assignments:", int(comparison["changed"].sum()))
print("Storage budget satisfied:", total_storage <= storage_budget + 1e-4)
print("Quantization budget satisfied:", total_quant_loss <= MAX_QUANTIZATION_LOSS_PP + 1e-4)
print("Selected storage bits:", int(total_storage))
print("Quantization loss proxy:", round(total_quant_loss, 6), "pp")
print("Residual fault-risk proxy:", round(total_fault_risk, 6), "pp")
print("Catastrophic-risk proxy:", round(total_cat_risk, 6), "pp")
print("Refined objective:", round(float(result.fun), 8))

print("\nSaved files:")
for p in [
    selected_path,
    comparison_path,
    candidates_path,
    datatype_path,
    protection_path,
    cat_path,
    report_path
]:
    print(f"  {p} ({p.stat().st_size / 1024:.2f} KB)")

print("\n" + "=" * 76)
print("EXPERIMENT 7B COMPLETED")
print("=" * 76)
print(
    "This is a refined candidate configuration. "
    "It must still pass end-to-end model validation."
)


JFAMP EXPERIMENT 7B: CATASTROPHIC-FAULT-AWARE OPTIMIZATION
Found: /kaggle/working/JFAMP/experiments/joint_optimization/all_optimization_candidates.csv
Found: /kaggle/working/JFAMP/experiments/joint_optimization/optimized_layer_configuration.csv
Found: /kaggle/working/JFAMP/experiments/joint_optimization/optimization_report.json
Found: /kaggle/working/JFAMP/experiments/fault_profiling/fault_profiling_layer_summary.csv
Found: /kaggle/working/JFAMP/experiments/protection_profiling/protection_layer_summary.csv

Loaded:
Candidates: (672, 9)
Previous solution: (21, 11)
Fault summary: (42, 13)
Protection summary: (168, 14)

Catastrophic-fault summary:


,_layer,_fault,failed_trials,total_trials,catastrophic_rate
0,conv1,bitflip,1,100,0.01
1,conv1,random_corruption,0,100,0.00
2,fc,bitflip,0,100,0.00
3,fc,random_corruption,0,100,0.00
4,layer1.0.conv1,bitflip,2,100,0.02
5,layer1.0.conv1,random_corruption,0,100,0.00
6,layer1.0.conv2,bitflip,1,100,0.01
7,layer1.0.conv2,random_corruption,0,100,0.00
8,layer1.1.conv1,bitflip,1,100,0.01
9,layer1.1.conv1,random_corruption,0,100,0.00



Total failed trials: 19
Total trials: 4200

Catastrophic-risk cost added.
Maximum candidate catastrophic-risk cost: 3.5 pp

RUNNING REFINED MILP
Variables: 672
Layers: 21
Storage budget: 215240294.4
Quantization-loss budget: 1.0
Lambda fault: 1.0
Lambda catastrophic: 1.0
Catastrophic severity assumption: 100.0

Solver message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Solver success: True
Solver status: 0
Solver time: 0.012 seconds


KeyError: 'datatype'

In [20]:

# =====================================================================
# JFAMP EXPERIMENT 7B: RESUME AFTER MILP
# Fixes Pandas merge suffix issue. Does not rerun optimization.
# =====================================================================

import json
import pandas as pd
from pathlib import Path

ROOT = Path("/kaggle/working/JFAMP")
OLD_DIR = ROOT / "experiments/joint_optimization"
OUT_DIR = ROOT / "experiments/joint_optimization_refined"
OUT_DIR.mkdir(parents=True, exist_ok=True)

old_selected = pd.read_csv(
    OLD_DIR / "optimized_layer_configuration.csv"
)

# Use the already-computed refined solution from the previous cell.
assert "refined" in globals(), (
    "The refined solution is not in this kernel. "
    "If the kernel was restarted, rerun the complete optimization cell."
)

# -------------------- 1. ORIGINAL VS REFINED ----------------------

comparison = old_selected[
    ["layer", "datatype", "protection"]
].merge(
    refined[
        [
            "layer",
            "datatype",
            "protection",
            "catastrophic_risk_pp",
            "refined_objective_cost"
        ]
    ],
    on="layer",
    how="outer",
    suffixes=("_original", "_refined")
)

comparison["changed"] = (
    (
        comparison["datatype_original"]
        != comparison["datatype_refined"]
    )
    |
    (
        comparison["protection_original"]
        != comparison["protection_refined"]
    )
)

print("=" * 75)
print("ORIGINAL VS REFINED CONFIGURATION")
print("=" * 75)
display(comparison)

print(
    "Changed assignments:",
    int(comparison["changed"].sum()),
    "out of",
    len(comparison)
)

# -------------------- 2. SUMMARY TABLES ----------------------------

datatype_summary = (
    refined.groupby("datatype", as_index=False)
    .agg(
        layers=("layer", "count"),
        storage_bits=("total_storage_bits", "sum"),
        quantization_loss_pp=("quantization_loss_pp", "sum"),
        fault_risk_pp=("residual_fault_risk_pp", "sum"),
        catastrophic_risk_pp=("catastrophic_risk_pp", "sum")
    )
)

protection_summary = (
    refined.groupby("protection", as_index=False)
    .agg(
        layers=("layer", "count"),
        storage_bits=("total_storage_bits", "sum"),
        quantization_loss_pp=("quantization_loss_pp", "sum"),
        fault_risk_pp=("residual_fault_risk_pp", "sum"),
        catastrophic_risk_pp=("catastrophic_risk_pp", "sum")
    )
)

print("\nDATATYPE DISTRIBUTION")
display(datatype_summary.round(6))

print("\nPROTECTION DISTRIBUTION")
display(protection_summary.round(6))

# -------------------- 3. SAVE RESULTS ------------------------------

refined.to_csv(
    OUT_DIR / "refined_layer_configuration.csv",
    index=False
)

comparison.to_csv(
    OUT_DIR / "original_vs_refined_comparison.csv",
    index=False
)

candidates.to_csv(
    OUT_DIR / "refined_all_candidates.csv",
    index=False
)

datatype_summary.to_csv(
    OUT_DIR / "refined_datatype_summary.csv",
    index=False
)

protection_summary.to_csv(
    OUT_DIR / "refined_protection_summary.csv",
    index=False
)

# -------------------- 4. FINAL METRICS -----------------------------

total_storage = float(refined["total_storage_bits"].sum())
total_quant_loss = float(refined["quantization_loss_pp"].sum())
total_fault_risk = float(refined["residual_fault_risk_pp"].sum())
total_cat_risk = float(refined["catastrophic_risk_pp"].sum())

report = {
    "experiment": "JFAMP Experiment 7B - Catastrophic Risk Refinement",
    "solver_message": str(result.message),
    "solver_success": bool(result.success),
    "solver_status": int(result.status),
    "objective_value": float(result.fun),
    "changed_assignments": int(comparison["changed"].sum()),
    "storage_budget_bits": float(storage_budget),
    "selected_storage_bits": total_storage,
    "quantization_loss_budget_pp": 1.0,
    "selected_quantization_loss_proxy_pp": total_quant_loss,
    "selected_fault_risk_proxy_pp": total_fault_risk,
    "selected_catastrophic_risk_proxy_pp": total_cat_risk,
    "lambda_fault": float(LAMBDA_FAULT),
    "lambda_catastrophic": float(LAMBDA_CATASTROPHIC),
    "catastrophic_severity_assumption_pp": float(
        CATASTROPHIC_SEVERITY_PP
    ),
    "note": (
        "Catastrophic risk is a modelled proxy using observed "
        "non-finite trial rates and protection residual-error rates. "
        "End-to-end validation remains necessary."
    )
}

with open(
    OUT_DIR / "refined_optimization_report.json", "w"
) as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 75)
print("REFINED OPTIMIZATION METRICS")
print("=" * 75)

print("Solver:", result.message)
print("Storage used (bits):", int(total_storage))
print("Storage budget (bits):", int(storage_budget))
print("Storage constraint satisfied:", total_storage <= storage_budget + 1e-4)
print("Quantization loss proxy:", round(total_quant_loss, 6), "pp")
print("Fault-risk proxy:", round(total_fault_risk, 6), "pp")
print("Catastrophic-risk proxy:", round(total_cat_risk, 6), "pp")
print("Objective:", round(float(result.fun), 8))
print("Changed assignments:", int(comparison["changed"].sum()))

print("\nSaved refined outputs to:")
print(OUT_DIR)
print("\nExperiment 7B output recovery completed.")


ORIGINAL VS REFINED CONFIGURATION


,layer,datatype_original,protection_original,datatype_refined,protection_refined,catastrophic_risk_pp,refined_objective_cost,changed
0,conv1,INT8,secded,INT8,secded,0.0,0.0282,False
1,fc,FP16,secded,FP16,secded,0.0,0.0006,False
2,layer1.0.conv1,INT8,none,INT8,secded,0.0,0.0000,True
3,layer1.0.conv2,FP8_E4M3FN,secded,FP8_E4M3FN,secded,0.0,0.0003,False
4,layer1.1.conv1,FP8_E5M2,secded,FP8_E5M2,secded,0.0,0.0000,False
5,layer1.1.conv2,INT8,secded,INT8,secded,0.0,0.0006,False
6,layer2.0.conv1,FP16,secded,FP16,secded,0.0,0.0000,False
7,layer2.0.conv2,FP8_E4M3FN,secded,FP8_E4M3FN,secded,0.0,0.0008,False
8,layer2.0.downsample.0,FP16,secded,FP16,secded,0.0,0.0000,False
9,layer2.1.conv1,INT8,secded,INT8,secded,0.0,0.0000,False


Changed assignments: 1 out of 21

DATATYPE DISTRIBUTION


,datatype,layers,storage_bits,quantization_loss_pp,fault_risk_pp,catastrophic_risk_pp
0,FP16,6,48345088,0.0,0.0007,0.0
1,FP8_E4M3FN,3,33067008,0.0,0.0011,0.0
2,FP8_E5M2,3,8572928,0.0,0.0000,0.0
3,INT4,1,1048576,0.0,0.0002,0.0
4,INT8,8,73824192,0.0,0.0289,0.0



PROTECTION DISTRIBUTION


,protection,layers,storage_bits,quantization_loss_pp,fault_risk_pp,catastrophic_risk_pp
0,secded,21,164857792,0.0,0.0309,0.0



REFINED OPTIMIZATION METRICS
Solver: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Storage used (bits): 164857792
Storage budget (bits): 215240294
Storage constraint satisfied: True
Quantization loss proxy: 0.0 pp
Fault-risk proxy: 0.0309 pp
Catastrophic-risk proxy: 0.0 pp
Objective: 0.0309
Changed assignments: 1

Saved refined outputs to:
/kaggle/working/JFAMP/experiments/joint_optimization_refined

Experiment 7B output recovery completed.


In [21]:

# =====================================================================
# JFAMP EXPERIMENT 8
# END-TO-END VALIDATION OF THE REFINED MIXED-PRECISION CONFIGURATION
#
# Model: ResNet-18
# Dataset: CIFAR-100
# Validation: Original 5,000-example validation split
# Checkpoint: Canonical epoch-139 FP32 checkpoint
#
# Weight-only quantization:
#   FP32, FP16, BF16, INT8, INT4,
#   FP8_E4M3FN, FP8_E5M2, E4M11_SUBNORMAL
#
# Inference:
#   Quantized weights reconstructed to FP32.
#   Biases, BatchNorm, activations remain FP32.
#
# Protection:
#   SECDED overhead is included in theoretical storage only.
#   No fault injection or ECC decoding in this experiment.
#
# Outputs:
#   /kaggle/working/JFAMP/experiments/end_to_end_validation/
# =====================================================================

import os
import sys
import json
import time
import inspect
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader, Subset
from torchvision.models import resnet18

warnings.filterwarnings("ignore")

# -------------------------- 1. CONFIGURATION -------------------------

ROOT = Path("/kaggle/working/JFAMP")

CHECKPOINT = (
    ROOT / "retrain_to150/checkpoints/"
    "resnet18_cifar100_best_150.pth"
)

SPLIT_FILE = (
    Path("/kaggle/input/datasets/crahulnaidu/"
         "jfamp-resnet18-cifar100-checkpoint/")
    / "data_split_indices.json"
)

CONFIG_FILE = (
    ROOT / "experiments/joint_optimization_refined/"
    "refined_layer_configuration.csv"
)

CODEC_DIR = ROOT / "codec"

OUT_DIR = ROOT / "experiments/end_to_end_validation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

BATCH_SIZE = 128
NUM_WORKERS = 2

EXPECTED_BASELINE = 77.78

MEAN = (0.5071, 0.4867, 0.4408)
STD = (0.2675, 0.2565, 0.2761)

print("=" * 76)
print("JFAMP EXPERIMENT 8: END-TO-END VALIDATION")
print("=" * 76)

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

for path in [CHECKPOINT, SPLIT_FILE, CONFIG_FILE]:
    assert path.exists(), f"Required file not found: {path}"
    print("Found:", path)

# -------------------------- 2. LOAD REFINED CONFIG -------------------

config = pd.read_csv(CONFIG_FILE)

required_columns = [
    "layer",
    "datatype",
    "protection"
]

for col in required_columns:
    assert col in config.columns, (
        f"Missing configuration column: {col}"
    )

config["layer"] = config["layer"].astype(str)
config["datatype"] = config["datatype"].astype(str)

assert len(config) == 21, (
    f"Expected 21 selected layers, found {len(config)}"
)

assert config["layer"].nunique() == 21

print("\nRefined configuration loaded:")
display(config[["layer", "datatype", "protection"]])

# -------------------------- 3. LOAD E4M11 CODEC ----------------------

sys.path.insert(0, str(CODEC_DIR))

import e4m11_codec

assert hasattr(e4m11_codec, "encode_tensor")
assert hasattr(e4m11_codec, "decode_to_tensor")

print("\nTeammate E4M11 codec loaded.")
print("Codec path:", CODEC_DIR / "e4m11_codec.py")

# -------------------------- 4. FIND CIFAR-100 DATA ------------------

def find_cifar100_root():
    candidates = [
        ROOT / "data",
        ROOT / "data/cifar100",
        ROOT / "data/cifar-100-python",
        Path("/kaggle/input/cifar100"),
        Path("/kaggle/input/cifar-100"),
        Path("/kaggle/input/cifar-100-python"),
    ]

    for candidate in candidates:
        if candidate.exists():
            if (
                (candidate / "cifar-100-python").exists()
                or (candidate / "train").exists()
            ):
                return candidate

    search_roots = [
        Path("/kaggle/input"),
        Path("/kaggle/working/JFAMP/data"),
    ]

    for search_root in search_roots:
        if not search_root.exists():
            continue

        for folder, dirs, files in os.walk(search_root):
            if "test" in files and "train" in files:
                if (
                    "cifar-100-python" in folder
                    or "cifar100" in folder.lower()
                ):
                    return Path(folder).parent

    raise FileNotFoundError(
        "Could not locate extracted CIFAR-100 data. "
        "Expected the CIFAR-100 train and test files. "
        "No download was attempted."
    )

DATA_ROOT = find_cifar100_root()

print("\nCIFAR-100 root:", DATA_ROOT)

# -------------------------- 5. RECREATE ORIGINAL VALIDATION SPLIT ---

with open(SPLIT_FILE, "r") as f:
    split_info = json.load(f)

validation_indices = split_info["validation_indices"]

assert len(validation_indices) == 5000

split_hash = hashlib.sha256(
    json.dumps(
        validation_indices,
        separators=(",", ":")
    ).encode()
).hexdigest()

print("Validation examples:", len(validation_indices))
print("Validation split hash:", split_hash)

expected_hash = (
    "b632b03ccc8166092cbf5c9e02a41c754248533ef1f60292c0404e3a8d80d0e7"
)

# The canonical split hash is checked against the original saved indices.
assert split_hash == expected_hash, (
    "Validation split hash mismatch. "
    "Stopping to avoid evaluating on the wrong split."
)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

validation_dataset_full = torchvision.datasets.CIFAR100(
    root=str(DATA_ROOT),
    train=True,
    download=False,
    transform=transform
)

validation_dataset = Subset(
    validation_dataset_full,
    validation_indices
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0)
)

print("Validation loader created.")
print("Validation batches:", len(validation_loader))

# -------------------------- 6. BUILD RESNET-18 -----------------------

def build_model():
    model = resnet18(weights=None)

    model.conv1 = nn.Conv2d(
        3, 64,
        kernel_size=3,
        stride=1,
        padding=1,
        bias=False
    )

    model.maxpool = nn.Identity()
    model.fc = nn.Linear(512, 100)

    return model

# -------------------------- 7. LOAD CHECKPOINT ----------------------

checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

def extract_state_dict(obj):
    if isinstance(obj, dict):
        for key in [
            "state_dict",
            "model_state_dict",
            "model",
            "net",
            "weights"
        ]:
            if key in obj and isinstance(obj[key], dict):
                try:
                    if any(
                        torch.is_tensor(v)
                        for v in obj[key].values()
                    ):
                        return obj[key]
                except Exception:
                    pass

        if any(torch.is_tensor(v) for v in obj.values()):
            return obj

    raise ValueError(
        "Could not identify model state_dict in checkpoint."
    )

state_dict = extract_state_dict(checkpoint)

clean_state_dict = {}

for key, value in state_dict.items():
    new_key = key

    while new_key.startswith("module."):
        new_key = new_key[len("module."):]

    clean_state_dict[new_key] = value

model_fp32 = build_model()

load_result = model_fp32.load_state_dict(
    clean_state_dict,
    strict=True
)

model_fp32 = model_fp32.to(DEVICE)
model_fp32.eval()

print("\nCanonical checkpoint loaded.")
print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)

# -------------------------- 8. EVALUATION FUNCTION ------------------

@torch.inference_mode()
def evaluate(model, loader):
    model.eval()

    correct = 0
    total = 0
    total_loss = 0.0

    criterion = nn.CrossEntropyLoss(reduction="sum")

    for images, labels in loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        outputs = model(images)

        loss = criterion(outputs, labels)

        total_loss += float(loss.item())

        predictions = outputs.argmax(dim=1)

        correct += int(
            (predictions == labels).sum().item()
        )

        total += labels.size(0)

    accuracy = 100.0 * correct / total
    average_loss = total_loss / total

    return {
        "accuracy": accuracy,
        "loss": average_loss,
        "correct": correct,
        "total": total
    }

# -------------------------- 9. FP32 BASELINE ------------------------

print("\n" + "=" * 76)
print("EVALUATING FP32 BASELINE")
print("=" * 76)

start = time.time()

baseline_metrics = evaluate(
    model_fp32,
    validation_loader
)

baseline_time = time.time() - start

print("FP32 accuracy:", round(
    baseline_metrics["accuracy"], 4
), "%")

print("FP32 loss:", round(
    baseline_metrics["loss"], 8
))

print("Evaluation time:", round(
    baseline_time, 2
), "seconds")

print(
    "Previously recorded FP32 baseline:",
    EXPECTED_BASELINE,
    "%"
)

# -------------------------- 10. QUANTIZATION HELPERS ----------------

def quantize_fp16(weight):
    return weight.to(torch.float16).to(torch.float32)

def quantize_bf16(weight):
    return weight.to(torch.bfloat16).to(torch.float32)

def quantize_fp8(weight, fmt):
    # Perform conversion on CPU to avoid unsupported GPU operations
    # on hardware without native FP8 execution support.
    cpu_weight = weight.detach().cpu().contiguous()

    if fmt == "FP8_E4M3FN":
        dtype = torch.float8_e4m3fn
    elif fmt == "FP8_E5M2":
        dtype = torch.float8_e5m2
    else:
        raise ValueError(fmt)

    return (
        cpu_weight.to(dtype)
        .to(torch.float32)
        .to(weight.device)
    )

def quantize_int8_per_output_channel(weight):
    qmax = 127

    if weight.ndim < 2:
        raise ValueError("Expected Conv2d/Linear weight.")

    flat = weight.reshape(weight.shape[0], -1)

    scale = (
        flat.abs().amax(dim=1)
        .clamp(min=1e-12)
        / qmax
    )

    scale_shape = [weight.shape[0]] + [1] * (
        weight.ndim - 1
    )

    scale = scale.reshape(scale_shape)

    quantized = torch.round(weight / scale)
    quantized = torch.clamp(quantized, -127, 127)

    return quantized * scale

def quantize_int4_per_tensor(weight):
    qmax = 7

    scale = weight.abs().max().clamp(min=1e-12) / qmax

    quantized = torch.round(weight / scale)
    quantized = torch.clamp(quantized, -7, 7)

    return quantized * scale

def quantize_e4m11_subnormal(weight):
    # Use the exact teammate-provided codec.
    # No replacement or approximation of its encoding logic.

    codec = e4m11_codec

    encoded = codec.encode_tensor(
        weight.detach(),
        mode="subnormal",
        all_ones="clamp"
    )

    decoded = codec.decode_to_tensor(encoded)

    if isinstance(decoded, np.ndarray):
        decoded = torch.from_numpy(decoded)

    if not torch.is_tensor(decoded):
        decoded = torch.as_tensor(decoded)

    return decoded.to(
        device=weight.device,
        dtype=torch.float32
    )

def apply_quantization(weight, datatype):
    if datatype == "FP16":
        return quantize_fp16(weight)

    if datatype == "BF16":
        return quantize_bf16(weight)

    if datatype == "INT8":
        return quantize_int8_per_output_channel(weight)

    if datatype == "INT4":
        return quantize_int4_per_tensor(weight)

    if datatype == "FP8_E4M3FN":
        return quantize_fp8(weight, datatype)

    if datatype == "FP8_E5M2":
        return quantize_fp8(weight, datatype)

    if datatype == "E4M11_SUBNORMAL":
        return quantize_e4m11_subnormal(weight)

    if datatype == "FP32":
        return weight.detach().clone()

    raise ValueError(
        f"Unsupported datatype: {datatype}"
    )

# -------------------------- 11. APPLY LAYER-WISE CONFIGURATION ------

model_quantized = build_model()

model_quantized.load_state_dict(
    model_fp32.state_dict(),
    strict=True
)

model_quantized = model_quantized.to(DEVICE)
model_quantized.eval()

module_dict = dict(model_quantized.named_modules())

layer_records = []

print("\n" + "=" * 76)
print("APPLYING REFINED MIXED-PRECISION CONFIGURATION")
print("=" * 76)

for _, row in config.iterrows():

    layer_name = row["layer"]
    datatype = row["datatype"]
    protection = str(row["protection"]).lower()

    if layer_name not in module_dict:
        raise KeyError(
            f"Configured layer not found in model: {layer_name}"
        )

    module = module_dict[layer_name]

    if not isinstance(module, (nn.Conv2d, nn.Linear)):
        raise TypeError(
            f"{layer_name} is not Conv2d or Linear."
        )

    original_weight = module.weight.detach().clone()

    quantized_weight = apply_quantization(
        original_weight,
        datatype
    )

    if quantized_weight.shape != original_weight.shape:
        raise ValueError(
            f"Shape mismatch in {layer_name}"
        )

    if not torch.isfinite(quantized_weight).all():
        raise ValueError(
            f"Non-finite quantized weight in {layer_name}"
        )

    module.weight.data.copy_(quantized_weight)

    error = (
        original_weight.float()
        - quantized_weight.float()
    )

    signal_power = (
        original_weight.float().pow(2).sum().item()
    )

    noise_power = error.pow(2).sum().item()

    if noise_power == 0:
        sqnr_db = float("inf")
    elif signal_power == 0:
        sqnr_db = float("-inf")
    else:
        sqnr_db = 10 * np.log10(
            signal_power / noise_power
        )

    layer_records.append({
        "layer": layer_name,
        "datatype": datatype,
        "protection": protection,
        "parameter_count": original_weight.numel(),
        "weight_mse": float(error.pow(2).mean().item()),
        "weight_sqnr_db": float(sqnr_db),
        "finite_weights": bool(
            torch.isfinite(quantized_weight).all().item()
        )
    })

    print(
        f"{layer_name:30s} "
        f"{datatype:18s} "
        f"{protection}"
    )

assert len(layer_records) == 21

# -------------------------- 12. END-TO-END EVALUATION ----------------

print("\n" + "=" * 76)
print("EVALUATING REFINED MIXED-PRECISION MODEL")
print("=" * 76)

start = time.time()

quantized_metrics = evaluate(
    model_quantized,
    validation_loader
)

quantized_time = time.time() - start

accuracy_delta = (
    quantized_metrics["accuracy"]
    - baseline_metrics["accuracy"]
)

loss_delta = (
    quantized_metrics["loss"]
    - baseline_metrics["loss"]
)

print("\nFP32 accuracy:",
      round(baseline_metrics["accuracy"], 4), "%")

print("Refined mixed-precision accuracy:",
      round(quantized_metrics["accuracy"], 4), "%")

print("Accuracy change:",
      round(accuracy_delta, 4), "percentage points")

print("FP32 loss:",
      round(baseline_metrics["loss"], 8))

print("Mixed-precision loss:",
      round(quantized_metrics["loss"], 8))

print("Loss change:",
      round(loss_delta, 8))

print("Mixed-precision evaluation time:",
      round(quantized_time, 2), "seconds")

# -------------------------- 13. THEORETICAL STORAGE -----------------

def datatype_bits(datatype):
    mapping = {
        "FP32": 32,
        "FP16": 16,
        "BF16": 16,
        "INT8": 8,
        "INT4": 4,
        "FP8_E4M3FN": 8,
        "FP8_E5M2": 8,
        "E4M11_SUBNORMAL": 16
    }

    if datatype not in mapping:
        raise ValueError(datatype)

    return mapping[datatype]

def secded_total_bits(data_bits):
    # Minimum Hamming parity bits r satisfying:
    # 2^r >= data_bits + r + 1
    r = 0

    while 2**r < data_bits + r + 1:
        r += 1

    # Add one overall parity bit.
    return data_bits + r + 1

storage_records = []

for _, row in config.iterrows():

    layer_name = row["layer"]
    datatype = row["datatype"]
    protection = str(row["protection"]).lower()

    module = module_dict[layer_name]

    count = module.weight.numel()
    bits = datatype_bits(datatype)

    if protection == "none":
        protected_bits = bits

    elif protection == "parity":
        protected_bits = bits + 1

    elif protection == "secded":
        protected_bits = secded_total_bits(bits)

    elif protection == "dmr":
        protected_bits = bits * 2

    else:
        raise ValueError(
            f"Unknown protection: {protection}"
        )

    storage_records.append({
        "layer": layer_name,
        "datatype": datatype,
        "protection": protection,
        "weights": count,
        "datatype_bits_per_weight": bits,
        "protected_bits_per_weight": protected_bits,
        "data_storage_bits": count * bits,
        "protected_storage_bits": count * protected_bits
    })

storage_df = pd.DataFrame(storage_records)

total_fp32_bits = int(
    storage_df["weights"].sum() * 32
)

total_data_bits = int(
    storage_df["data_storage_bits"].sum()
)

total_protected_bits = int(
    storage_df["protected_storage_bits"].sum()
)

storage_reduction = (
    1 - total_protected_bits / total_fp32_bits
) * 100

print("\n" + "=" * 76)
print("THEORETICAL STORAGE ANALYSIS")
print("=" * 76)

print("FP32 storage:",
      round(total_fp32_bits / 8 / 1024**2, 4), "MiB")

print("Quantized data storage:",
      round(total_data_bits / 8 / 1024**2, 4), "MiB")

print("Quantized + protection storage:",
      round(total_protected_bits / 8 / 1024**2, 4), "MiB")

print("Storage reduction after protection:",
      round(storage_reduction, 4), "%")

display(storage_df)

# -------------------------- 14. SAVE QUANTIZED MODEL ----------------

model_path = (
    OUT_DIR / "resnet18_jfamp_refined_dequantized_fp32.pth"
)

torch.save(
    {
        "model_state_dict": model_quantized.state_dict(),
        "architecture": "CIFAR100_ResNet18",
        "experiment": "JFAMP_Experiment_8",
        "configuration": config.to_dict(orient="records"),
        "inference_mode": "FP32 inference with quantized/dequantized weights"
    },
    model_path
)

# -------------------------- 15. SAVE REPORTS ------------------------

layer_df = pd.DataFrame(layer_records)

layer_df.to_csv(
    OUT_DIR / "end_to_end_layer_quantization.csv",
    index=False
)

storage_df.to_csv(
    OUT_DIR / "end_to_end_storage_report.csv",
    index=False
)

config.to_csv(
    OUT_DIR / "configuration_used.csv",
    index=False
)

results = {
    "experiment": "JFAMP Experiment 8 - End-to-End Validation",
    "architecture": "CIFAR100 ResNet18",
    "checkpoint": str(CHECKPOINT),
    "checkpoint_epoch": 139,
    "validation_examples": int(baseline_metrics["total"]),
    "validation_split_sha256": split_hash,
    "device": str(DEVICE),
    "inference_mode": "FP32 with quantized/dequantized weights",
    "weight_only_quantization": True,
    "bias_fp32": True,
    "batchnorm_fp32": True,
    "activation_fp32": True,
    "protection_execution": False,
    "protection_storage_model": "Theoretical SECDED/parity/DMR overhead",
    "fp32_baseline": {
        "accuracy": float(baseline_metrics["accuracy"]),
        "loss": float(baseline_metrics["loss"]),
        "correct": int(baseline_metrics["correct"])
    },
    "refined_model": {
        "accuracy": float(quantized_metrics["accuracy"]),
        "loss": float(quantized_metrics["loss"]),
        "correct": int(quantized_metrics["correct"])
    },
    "accuracy_delta_percentage_points": float(accuracy_delta),
    "loss_delta": float(loss_delta),
    "storage": {
        "fp32_bits": total_fp32_bits,
        "quantized_data_bits": total_data_bits,
        "quantized_protected_bits": total_protected_bits,
        "fp32_mib": total_fp32_bits / 8 / 1024**2,
        "quantized_data_mib": total_data_bits / 8 / 1024**2,
        "quantized_protected_mib": total_protected_bits / 8 / 1024**2,
        "storage_reduction_percentage": float(storage_reduction)
    },
    "timing_seconds": {
        "fp32_evaluation": float(baseline_time),
        "refined_model_evaluation": float(quantized_time)
    },
    "outputs": {
        "model": str(model_path),
        "layer_report": str(
            OUT_DIR / "end_to_end_layer_quantization.csv"
        ),
        "storage_report": str(
            OUT_DIR / "end_to_end_storage_report.csv"
        ),
        "configuration": str(
            OUT_DIR / "configuration_used.csv"
        )
    }
}

with open(
    OUT_DIR / "end_to_end_validation_results.json",
    "w"
) as f:
    json.dump(results, f, indent=2)

# -------------------------- 16. FINAL VALIDATION CHECKS -------------

assert quantized_metrics["total"] == 5000
assert len(layer_df) == 21
assert layer_df["finite_weights"].all()
assert np.isfinite(quantized_metrics["accuracy"])
assert np.isfinite(quantized_metrics["loss"])

print("\n" + "=" * 76)
print("JFAMP EXPERIMENT 8 COMPLETED")
print("=" * 76)

print("FP32 baseline accuracy:",
      round(baseline_metrics["accuracy"], 4), "%")

print("Refined model accuracy:",
      round(quantized_metrics["accuracy"], 4), "%")

print("Accuracy delta:",
      round(accuracy_delta, 4), "pp")

print("Storage reduction:",
      round(storage_reduction, 4), "%")

print("\nOutput directory:")
print(OUT_DIR)

print("\nGenerated files:")
for path in sorted(OUT_DIR.iterdir()):
    if path.is_file():
        print(
            f"{path.name:55s} "
            f"{path.stat().st_size / 1024:.2f} KB"
        )

print("\nExperiment 8 validation finished.")


JFAMP EXPERIMENT 8: END-TO-END VALIDATION
PyTorch: 2.11.0+cu130
Torchvision: 0.26.0+cu130
Device: cuda
GPU: Tesla T4
Found: /kaggle/working/JFAMP/retrain_to150/checkpoints/resnet18_cifar100_best_150.pth
Found: /kaggle/input/datasets/crahulnaidu/jfamp-resnet18-cifar100-checkpoint/data_split_indices.json
Found: /kaggle/working/JFAMP/experiments/joint_optimization_refined/refined_layer_configuration.csv

Refined configuration loaded:


,layer,datatype,protection
0,conv1,INT8,secded
1,fc,FP16,secded
2,layer1.0.conv1,INT8,secded
3,layer1.0.conv2,FP8_E4M3FN,secded
4,layer1.1.conv1,FP8_E5M2,secded
5,layer1.1.conv2,INT8,secded
6,layer2.0.conv1,FP16,secded
7,layer2.0.conv2,FP8_E4M3FN,secded
8,layer2.0.downsample.0,FP16,secded
9,layer2.1.conv1,INT8,secded



Teammate E4M11 codec loaded.
Codec path: /kaggle/working/JFAMP/codec/e4m11_codec.py

CIFAR-100 root: /kaggle/working/JFAMP/data
Validation examples: 5000
Validation split hash: 96b2f881b3019c0c941dd7d0421af3e76c505687f67643104a670fefd03dca97


AssertionError: Validation split hash mismatch. Stopping to avoid evaluating on the wrong split.

In [22]:

import hashlib
import json
from pathlib import Path

SPLIT_FILE = Path(
    "/kaggle/input/datasets/crahulnaidu/"
    "jfamp-resnet18-cifar100-checkpoint/"
    "data_split_indices.json"
)

EXPECTED_HASH = (
    "b632b03ccc8166092cbf5c9e02a41c754248533ef1f60292c0404e3a8d80d0e7"
)

# Hash of the original JSON file bytes
raw_file_hash = hashlib.sha256(
    SPLIT_FILE.read_bytes()
).hexdigest()

# Hash of the re-serialized validation index list
with open(SPLIT_FILE, "r") as f:
    split_info = json.load(f)

indices = split_info["validation_indices"]

serialized_indices_hash = hashlib.sha256(
    json.dumps(
        indices,
        separators=(",", ":")
    ).encode()
).hexdigest()

print("Original expected hash:      ", EXPECTED_HASH)
print("Raw JSON file SHA256:        ", raw_file_hash)
print("Serialized indices SHA256:  ", serialized_indices_hash)
print("Number of validation samples:", len(indices))

print(
    "\nRaw JSON matches canonical hash:",
    raw_file_hash == EXPECTED_HASH
)

print(
    "Serialized indices match canonical hash:",
    serialized_indices_hash == EXPECTED_HASH
)


Original expected hash:       b632b03ccc8166092cbf5c9e02a41c754248533ef1f60292c0404e3a8d80d0e7
Raw JSON file SHA256:         b632b03ccc8166092cbf5c9e02a41c754248533ef1f60292c0404e3a8d80d0e7
Serialized indices SHA256:   96b2f881b3019c0c941dd7d0421af3e76c505687f67643104a670fefd03dca97
Number of validation samples: 5000

Raw JSON matches canonical hash: True
Serialized indices match canonical hash: False


In [24]:

# =====================================================================
# JFAMP EXPERIMENT 8
# END-TO-END VALIDATION OF REFINED MIXED-PRECISION CONFIGURATION
#
# Model       : CIFAR-100 ResNet-18
# Checkpoint  : Canonical epoch-139 best checkpoint
# Validation  : Original 5,000-example validation split
#
# Quantization:
#   FP16, BF16, INT8, INT4,
#   FP8_E4M3FN, FP8_E5M2, E4M11_SUBNORMAL
#
# Inference:
#   Weight-only quantization
#   Quantized weights reconstructed to FP32
#   Bias, BatchNorm and activations remain FP32
#
# Protection:
#   Theoretical SECDED storage overhead
#   No ECC decoding or fault injection in this experiment
#
# Output:
#   /kaggle/working/JFAMP/experiments/end_to_end_validation/
# =====================================================================

import os
import sys
import json
import time
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader, Subset
from torchvision.models import resnet18

warnings.filterwarnings("ignore")

# =====================================================================
# 1. CONFIGURATION
# =====================================================================

ROOT = Path("/kaggle/working/JFAMP")

CHECKPOINT = (
    ROOT / "retrain_to150/checkpoints/"
    "resnet18_cifar100_best_150.pth"
)

SPLIT_FILE = Path(
    "/kaggle/input/datasets/crahulnaidu/"
    "jfamp-resnet18-cifar100-checkpoint/"
    "data_split_indices.json"
)

CONFIG_FILE = (
    ROOT / "experiments/joint_optimization_refined/"
    "refined_layer_configuration.csv"
)

CODEC_DIR = ROOT / "codec"

OUT_DIR = (
    ROOT / "experiments/end_to_end_validation"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

BATCH_SIZE = 128

# Avoid multiprocessing worker failures.
NUM_WORKERS = 0

EXPECTED_BASELINE = 77.78

EXPECTED_SPLIT_HASH = (
    "b632b03ccc8166092cbf5c9e02a41c754248533ef1f60292c0404e3a8d80d0e7"
)

MEAN = (0.5071, 0.4867, 0.4408)
STD = (0.2675, 0.2565, 0.2761)

print("=" * 76)
print("JFAMP EXPERIMENT 8: END-TO-END VALIDATION")
print("=" * 76)

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

for path in [
    CHECKPOINT,
    SPLIT_FILE,
    CONFIG_FILE,
    CODEC_DIR / "e4m11_codec.py"
]:
    assert path.exists(), f"Missing required file: {path}"
    print("Verified:", path)

# =====================================================================
# 2. LOAD REFINED OPTIMIZATION CONFIGURATION
# =====================================================================

config = pd.read_csv(CONFIG_FILE)

required_columns = [
    "layer",
    "datatype",
    "protection"
]

for col in required_columns:
    assert col in config.columns, (
        f"Missing configuration column: {col}"
    )

config["layer"] = config["layer"].astype(str)
config["datatype"] = config["datatype"].astype(str)
config["protection"] = (
    config["protection"].astype(str).str.lower()
)

assert len(config) == 21
assert config["layer"].nunique() == 21

SUPPORTED_DTYPES = {
    "FP32",
    "FP16",
    "BF16",
    "INT8",
    "INT4",
    "FP8_E4M3FN",
    "FP8_E5M2",
    "E4M11_SUBNORMAL"
}

unsupported = (
    set(config["datatype"].unique())
    - SUPPORTED_DTYPES
)

assert not unsupported, (
    f"Unsupported datatypes in configuration: {unsupported}"
)

print("\nRefined configuration loaded:")
display(config[["layer", "datatype", "protection"]])

# =====================================================================
# 3. LOAD TEAMMATE E4M11 CODEC
# =====================================================================

sys.path.insert(0, str(CODEC_DIR))

import e4m11_codec

assert hasattr(e4m11_codec, "encode_tensor")
assert hasattr(e4m11_codec, "decode_to_tensor")

print("\nE4M11 codec loaded:")
print(CODEC_DIR / "e4m11_codec.py")

# =====================================================================
# 4. LOCATE CIFAR-100 DATASET
# =====================================================================

def find_cifar100_root():

    candidates = [
        ROOT / "data",
        ROOT / "data/cifar100",
        ROOT / "data/cifar-100-python",
        Path("/kaggle/input/cifar100"),
        Path("/kaggle/input/cifar-100"),
        Path("/kaggle/input/cifar-100-python")
    ]

    for candidate in candidates:

        if not candidate.exists():
            continue

        if (candidate / "cifar-100-python").exists():
            return candidate

        if (
            (candidate / "train").exists()
            and (candidate / "test").exists()
        ):
            return candidate

        if (
            candidate.name == "cifar-100-python"
            and (candidate / "train").exists()
        ):
            return candidate.parent

    search_roots = [
        Path("/kaggle/input"),
        ROOT / "data"
    ]

    for search_root in search_roots:

        if not search_root.exists():
            continue

        for folder, dirs, files in os.walk(search_root):

            if (
                "train" in files
                and "test" in files
                and "cifar-100-python" in folder
            ):
                return Path(folder).parent

    raise FileNotFoundError(
        "CIFAR-100 extracted dataset not found. "
        "No dataset download was attempted."
    )


DATA_ROOT = find_cifar100_root()

print("\nCIFAR-100 root:", DATA_ROOT)

# =====================================================================
# 5. VERIFY ORIGINAL VALIDATION SPLIT
# =====================================================================

with open(SPLIT_FILE, "r") as f:
    split_info = json.load(f)

validation_indices = split_info["validation_indices"]

assert len(validation_indices) == 5000

# IMPORTANT:
# The canonical hash refers to the original JSON FILE BYTES.
# Do not hash a re-serialized list of indices here.

split_hash = hashlib.sha256(
    SPLIT_FILE.read_bytes()
).hexdigest()

assert split_hash == EXPECTED_SPLIT_HASH, (
    "Original validation split JSON SHA256 mismatch."
)

print("\nOriginal validation split verified.")
print("Validation examples:", len(validation_indices))
print("Original JSON SHA256:", split_hash)


# =====================================================================
# 6. CREATE ORIGINAL 5,000-IMAGE VALIDATION DATALOADER
# =====================================================================

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

# CIFAR-100 training archive contains 50,000 images.
# The original split JSON defines:
#   45,000 training indices
#    5,000 validation indices

validation_dataset_full = torchvision.datasets.CIFAR100(
    root=str(DATA_ROOT),
    train=True,
    download=False,
    transform=transform
)

assert len(validation_dataset_full) == 50000, (
    f"Unexpected CIFAR-100 dataset size: "
    f"{len(validation_dataset_full)}"
)

# Verify that the original split indices are valid.
assert min(validation_indices) >= 0
assert max(validation_indices) < len(validation_dataset_full)
assert len(set(validation_indices)) == 5000

validation_dataset = Subset(
    validation_dataset_full,
    validation_indices
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

assert len(validation_dataset) == 5000

print("CIFAR-100 full training archive:", len(validation_dataset_full))
print("Original training samples:", 45000)
print("Original validation samples:", len(validation_dataset))
print("Validation batches:", len(validation_loader))
print("Original split SHA256:", split_hash)

print("\nValidation dataloader created successfully.")


# =====================================================================
# 7. BUILD CIFAR-100 RESNET-18
# =====================================================================

def build_model():

    model = resnet18(weights=None)

    model.conv1 = nn.Conv2d(
        3,
        64,
        kernel_size=3,
        stride=1,
        padding=1,
        bias=False
    )

    model.maxpool = nn.Identity()

    model.fc = nn.Linear(
        512,
        100
    )

    return model


# =====================================================================
# 8. LOAD CANONICAL EPOCH-139 CHECKPOINT
# =====================================================================

checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False
)


def extract_state_dict(obj):

    if isinstance(obj, dict):

        possible_keys = [
            "state_dict",
            "model_state_dict",
            "model",
            "net",
            "weights"
        ]

        for key in possible_keys:

            if key in obj and isinstance(obj[key], dict):

                if any(
                    torch.is_tensor(v)
                    for v in obj[key].values()
                ):
                    return obj[key]

        if any(
            torch.is_tensor(v)
            for v in obj.values()
        ):
            return obj

    raise ValueError(
        "Unable to extract model state_dict."
    )


state_dict = extract_state_dict(checkpoint)

clean_state_dict = {}

for key, value in state_dict.items():

    new_key = key

    while new_key.startswith("module."):
        new_key = new_key[len("module."):]

    clean_state_dict[new_key] = value


model_fp32 = build_model()

load_result = model_fp32.load_state_dict(
    clean_state_dict,
    strict=True
)

model_fp32 = model_fp32.to(DEVICE)
model_fp32.eval()

assert len(load_result.missing_keys) == 0
assert len(load_result.unexpected_keys) == 0

print("\nCanonical checkpoint loaded successfully.")
print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)

# =====================================================================
# 9. EVALUATION FUNCTION
# =====================================================================

@torch.inference_mode()
def evaluate(model, loader):

    model.eval()

    correct = 0
    total = 0
    total_loss = 0.0

    criterion = nn.CrossEntropyLoss(
        reduction="sum"
    )

    for images, labels in loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        outputs = model(images)

        loss = criterion(outputs, labels)

        predictions = outputs.argmax(dim=1)

        total_loss += float(loss.item())

        correct += int(
            (predictions == labels).sum().item()
        )

        total += labels.size(0)

    return {
        "accuracy": 100.0 * correct / total,
        "loss": total_loss / total,
        "correct": correct,
        "total": total
    }


# =====================================================================
# 10. EVALUATE FP32 BASELINE
# =====================================================================

print("\n" + "=" * 76)
print("FP32 BASELINE EVALUATION")
print("=" * 76)

start = time.time()

baseline_metrics = evaluate(
    model_fp32,
    validation_loader
)

baseline_time = time.time() - start

print(
    "FP32 accuracy:",
    round(baseline_metrics["accuracy"], 4),
    "%"
)

print(
    "FP32 loss:",
    round(baseline_metrics["loss"], 8)
)

print(
    "Correct predictions:",
    baseline_metrics["correct"],
    "/",
    baseline_metrics["total"]
)

print(
    "Previously recorded baseline:",
    EXPECTED_BASELINE,
    "%"
)

print(
    "Evaluation time:",
    round(baseline_time, 2),
    "seconds"
)

# =====================================================================
# 11. QUANTIZATION IMPLEMENTATIONS
# =====================================================================

def quantize_fp16(weight):

    return (
        weight.to(torch.float16)
        .to(torch.float32)
    )


def quantize_bf16(weight):

    return (
        weight.to(torch.bfloat16)
        .to(torch.float32)
    )


def quantize_fp8(weight, fmt):

    cpu_weight = (
        weight.detach()
        .cpu()
        .contiguous()
    )

    if fmt == "FP8_E4M3FN":
        dtype = torch.float8_e4m3fn

    elif fmt == "FP8_E5M2":
        dtype = torch.float8_e5m2

    else:
        raise ValueError(fmt)

    return (
        cpu_weight.to(dtype)
        .to(torch.float32)
        .to(weight.device)
    )


def quantize_int8_per_output_channel(weight):

    qmax = 127

    flat = weight.reshape(
        weight.shape[0],
        -1
    )

    scale = (
        flat.abs().amax(dim=1)
        .clamp(min=1e-12)
        / qmax
    )

    scale_shape = (
        [weight.shape[0]]
        + [1] * (weight.ndim - 1)
    )

    scale = scale.reshape(scale_shape)

    quantized = torch.round(
        weight / scale
    )

    quantized = torch.clamp(
        quantized,
        -127,
        127
    )

    return quantized * scale


def quantize_int4_per_tensor(weight):

    qmax = 7

    scale = (
        weight.abs().max().clamp(min=1e-12)
        / qmax
    )

    quantized = torch.round(
        weight / scale
    )

    quantized = torch.clamp(
        quantized,
        -7,
        7
    )

    return quantized * scale


def quantize_e4m11_subnormal(weight):

    encoded = e4m11_codec.encode_tensor(
        weight.detach(),
        mode="subnormal",
        all_ones="clamp"
    )

    decoded = e4m11_codec.decode_to_tensor(
        encoded
    )

    if isinstance(decoded, np.ndarray):
        decoded = torch.from_numpy(decoded)

    if not torch.is_tensor(decoded):
        decoded = torch.as_tensor(decoded)

    return decoded.to(
        device=weight.device,
        dtype=torch.float32
    )


def apply_quantization(weight, datatype):

    if datatype == "FP32":
        return weight.detach().clone()

    elif datatype == "FP16":
        return quantize_fp16(weight)

    elif datatype == "BF16":
        return quantize_bf16(weight)

    elif datatype == "INT8":
        return quantize_int8_per_output_channel(weight)

    elif datatype == "INT4":
        return quantize_int4_per_tensor(weight)

    elif datatype == "FP8_E4M3FN":
        return quantize_fp8(
            weight,
            datatype
        )

    elif datatype == "FP8_E5M2":
        return quantize_fp8(
            weight,
            datatype
        )

    elif datatype == "E4M11_SUBNORMAL":
        return quantize_e4m11_subnormal(weight)

    else:
        raise ValueError(
            f"Unknown datatype: {datatype}"
        )


# =====================================================================
# 12. APPLY LAYER-WISE REFINED CONFIGURATION
# =====================================================================

model_quantized = build_model()

model_quantized.load_state_dict(
    model_fp32.state_dict(),
    strict=True
)

model_quantized = model_quantized.to(DEVICE)
model_quantized.eval()

module_dict = dict(
    model_quantized.named_modules()
)

layer_records = []

print("\n" + "=" * 76)
print("APPLYING REFINED MIXED-PRECISION CONFIGURATION")
print("=" * 76)

for _, row in config.iterrows():

    layer_name = row["layer"]
    datatype = row["datatype"]
    protection = row["protection"]

    if layer_name not in module_dict:
        raise KeyError(
            f"Layer not found: {layer_name}"
        )

    module = module_dict[layer_name]

    if not isinstance(
        module,
        (nn.Conv2d, nn.Linear)
    ):
        raise TypeError(
            f"Unexpected module type: {layer_name}"
        )

    original_weight = (
        module.weight.detach().clone()
    )

    quantized_weight = apply_quantization(
        original_weight,
        datatype
    )

    assert quantized_weight.shape == original_weight.shape

    assert torch.isfinite(
        quantized_weight
    ).all(), (
        f"Non-finite weights in {layer_name}"
    )

    # Copy reconstructed quantized weights into FP32 model.
    module.weight.data.copy_(
        quantized_weight
    )

    error = (
        original_weight.float()
        - quantized_weight.float()
    )

    signal_power = (
        original_weight.float()
        .pow(2)
        .sum()
        .item()
    )

    noise_power = (
        error.pow(2)
        .sum()
        .item()
    )

    if noise_power == 0:
        sqnr_db = float("inf")

    elif signal_power == 0:
        sqnr_db = float("-inf")

    else:
        sqnr_db = 10.0 * np.log10(
            signal_power / noise_power
        )

    layer_records.append({
        "layer": layer_name,
        "datatype": datatype,
        "protection": protection,
        "parameter_count": original_weight.numel(),
        "weight_mse": float(
            error.pow(2).mean().item()
        ),
        "weight_sqnr_db": float(sqnr_db),
        "finite_weights": True
    })

    print(
        f"{layer_name:30s} "
        f"{datatype:18s} "
        f"{protection}"
    )

assert len(layer_records) == 21

# =====================================================================
# 13. END-TO-END REFINED MODEL EVALUATION
# =====================================================================

print("\n" + "=" * 76)
print("REFINED MIXED-PRECISION MODEL EVALUATION")
print("=" * 76)

start = time.time()

quantized_metrics = evaluate(
    model_quantized,
    validation_loader
)

quantized_time = time.time() - start

accuracy_delta = (
    quantized_metrics["accuracy"]
    - baseline_metrics["accuracy"]
)

loss_delta = (
    quantized_metrics["loss"]
    - baseline_metrics["loss"]
)

print("\nFP32 accuracy:",
      round(baseline_metrics["accuracy"], 4), "%")

print("Refined model accuracy:",
      round(quantized_metrics["accuracy"], 4), "%")

print("Accuracy change:",
      round(accuracy_delta, 4), "pp")

print("FP32 loss:",
      round(baseline_metrics["loss"], 8))

print("Refined model loss:",
      round(quantized_metrics["loss"], 8))

print("Loss change:",
      round(loss_delta, 8))

print("Correct refined predictions:",
      quantized_metrics["correct"],
      "/",
      quantized_metrics["total"])

print("Evaluation time:",
      round(quantized_time, 2),
      "seconds")

# =====================================================================
# 14. THEORETICAL STORAGE CALCULATION
# =====================================================================

def datatype_bits(datatype):

    mapping = {
        "FP32": 32,
        "FP16": 16,
        "BF16": 16,
        "INT8": 8,
        "INT4": 4,
        "FP8_E4M3FN": 8,
        "FP8_E5M2": 8,
        "E4M11_SUBNORMAL": 16
    }

    return mapping[datatype]


def secded_total_bits(data_bits):

    r = 0

    while (
        2 ** r
        < data_bits + r + 1
    ):
        r += 1

    # Hamming parity bits + overall parity bit.
    return data_bits + r + 1


storage_records = []

for _, row in config.iterrows():

    layer_name = row["layer"]
    datatype = row["datatype"]
    protection = row["protection"]

    module = module_dict[layer_name]

    count = module.weight.numel()

    bits = datatype_bits(datatype)

    if protection == "none":

        protected_bits = bits

    elif protection == "parity":

        protected_bits = bits + 1

    elif protection == "secded":

        protected_bits = secded_total_bits(bits)

    elif protection == "dmr":

        protected_bits = bits * 2

    else:
        raise ValueError(
            f"Unknown protection strategy: {protection}"
        )

    storage_records.append({

        "layer": layer_name,

        "datatype": datatype,

        "protection": protection,

        "weights": count,

        "datatype_bits_per_weight": bits,

        "protected_bits_per_weight": protected_bits,

        "data_storage_bits": count * bits,

        "protected_storage_bits": (
            count * protected_bits
        )
    })


storage_df = pd.DataFrame(storage_records)

total_weights = int(
    storage_df["weights"].sum()
)

total_fp32_bits = total_weights * 32

total_data_bits = int(
    storage_df["data_storage_bits"].sum()
)

total_protected_bits = int(
    storage_df["protected_storage_bits"].sum()
)

storage_reduction = (
    1.0
    - total_protected_bits / total_fp32_bits
) * 100.0

print("\n" + "=" * 76)
print("THEORETICAL STORAGE ANALYSIS")
print("=" * 76)

print(
    "Total weights:",
    f"{total_weights:,}"
)

print(
    "FP32 storage:",
    round(total_fp32_bits / 8 / 1024**2, 4),
    "MiB"
)

print(
    "Quantized data storage:",
    round(total_data_bits / 8 / 1024**2, 4),
    "MiB"
)

print(
    "Quantized + protection storage:",
    round(total_protected_bits / 8 / 1024**2, 4),
    "MiB"
)

print(
    "Storage reduction after protection:",
    round(storage_reduction, 4),
    "%"
)

display(storage_df)

# =====================================================================
# 15. SAVE QUANTIZED MODEL
# =====================================================================

model_path = (
    OUT_DIR
    / "resnet18_jfamp_refined_dequantized_fp32.pth"
)

torch.save(
    {
        "model_state_dict": model_quantized.state_dict(),

        "architecture": "CIFAR100_ResNet18",

        "experiment": "JFAMP_Experiment_8",

        "configuration": config.to_dict(
            orient="records"
        ),

        "inference_mode": (
            "FP32 inference with quantized/dequantized weights"
        ),

        "checkpoint_epoch": 139
    },
    model_path
)

print("\nQuantized model saved:", model_path)

# =====================================================================
# 16. SAVE LAYER-WISE AND STORAGE REPORTS
# =====================================================================

layer_df = pd.DataFrame(layer_records)

layer_df.to_csv(
    OUT_DIR / "end_to_end_layer_quantization.csv",
    index=False
)

storage_df.to_csv(
    OUT_DIR / "end_to_end_storage_report.csv",
    index=False
)

config.to_csv(
    OUT_DIR / "configuration_used.csv",
    index=False
)

# =====================================================================
# 17. SAVE EXPERIMENT RESULTS
# =====================================================================

results = {

    "experiment": (
        "JFAMP Experiment 8 - End-to-End Validation"
    ),

    "architecture": "CIFAR100 ResNet18",

    "checkpoint": str(CHECKPOINT),

    "checkpoint_epoch": 139,

    "validation_examples": int(
        baseline_metrics["total"]
    ),

    "validation_split_json_sha256": split_hash,

    "device": str(DEVICE),

    "inference_mode": (
        "FP32 inference with quantized/dequantized weights"
    ),

    "weight_only_quantization": True,

    "bias_fp32": True,

    "batchnorm_fp32": True,

    "activation_fp32": True,

    "protection_execution": False,

    "protection_storage_model": (
        "Theoretical SECDED/parity/DMR overhead"
    ),

    "fp32_baseline": {

        "accuracy": float(
            baseline_metrics["accuracy"]
        ),

        "loss": float(
            baseline_metrics["loss"]
        ),

        "correct": int(
            baseline_metrics["correct"]
        )
    },

    "refined_model": {

        "accuracy": float(
            quantized_metrics["accuracy"]
        ),

        "loss": float(
            quantized_metrics["loss"]
        ),

        "correct": int(
            quantized_metrics["correct"]
        )
    },

    "accuracy_delta_percentage_points": float(
        accuracy_delta
    ),

    "loss_delta": float(
        loss_delta
    ),

    "storage": {

        "total_weights": total_weights,

        "fp32_bits": total_fp32_bits,

        "quantized_data_bits": total_data_bits,

        "quantized_protected_bits": (
            total_protected_bits
        ),

        "fp32_mib": (
            total_fp32_bits / 8 / 1024**2
        ),

        "quantized_data_mib": (
            total_data_bits / 8 / 1024**2
        ),

        "quantized_protected_mib": (
            total_protected_bits / 8 / 1024**2
        ),

        "storage_reduction_percentage": float(
            storage_reduction
        )
    },

    "timing_seconds": {

        "fp32_evaluation": float(
            baseline_time
        ),

        "refined_model_evaluation": float(
            quantized_time
        )
    },

    "outputs": {

        "model": str(model_path),

        "layer_report": str(
            OUT_DIR / "end_to_end_layer_quantization.csv"
        ),

        "storage_report": str(
            OUT_DIR / "end_to_end_storage_report.csv"
        ),

        "configuration": str(
            OUT_DIR / "configuration_used.csv"
        )
    }
}

results_path = (
    OUT_DIR / "end_to_end_validation_results.json"
)

with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

# =====================================================================
# 18. FINAL VALIDATION CHECKS
# =====================================================================

assert quantized_metrics["total"] == 5000

assert len(layer_df) == 21

assert layer_df["finite_weights"].all()

assert np.isfinite(
    quantized_metrics["accuracy"]
)

assert np.isfinite(
    quantized_metrics["loss"]
)

assert total_weights == 11210432, (
    f"Unexpected total weight count: {total_weights}"
)

# =====================================================================
# 19. FINAL SUMMARY
# =====================================================================

print("\n" + "=" * 76)
print("JFAMP EXPERIMENT 8 COMPLETED")
print("=" * 76)

print(
    "FP32 baseline accuracy:",
    round(baseline_metrics["accuracy"], 4),
    "%"
)

print(
    "Refined model accuracy:",
    round(quantized_metrics["accuracy"], 4),
    "%"
)

print(
    "Accuracy delta:",
    round(accuracy_delta, 4),
    "pp"
)

print(
    "FP32 loss:",
    round(baseline_metrics["loss"], 8)
)

print(
    "Refined model loss:",
    round(quantized_metrics["loss"], 8)
)

print(
    "Storage reduction:",
    round(storage_reduction, 4),
    "%"
)

print("\nOutput directory:")
print(OUT_DIR)

print("\nGenerated files:")

for path in sorted(OUT_DIR.iterdir()):

    if path.is_file():

        print(
            f"{path.name:55s}"
            f"{path.stat().st_size / 1024:.2f} KB"
        )

print("\nExperiment 8 finished successfully.")


JFAMP EXPERIMENT 8: END-TO-END VALIDATION
PyTorch: 2.11.0+cu130
Torchvision: 0.26.0+cu130
Device: cuda
GPU: Tesla T4
Verified: /kaggle/working/JFAMP/retrain_to150/checkpoints/resnet18_cifar100_best_150.pth
Verified: /kaggle/input/datasets/crahulnaidu/jfamp-resnet18-cifar100-checkpoint/data_split_indices.json
Verified: /kaggle/working/JFAMP/experiments/joint_optimization_refined/refined_layer_configuration.csv
Verified: /kaggle/working/JFAMP/codec/e4m11_codec.py

Refined configuration loaded:


,layer,datatype,protection
0,conv1,INT8,secded
1,fc,FP16,secded
2,layer1.0.conv1,INT8,secded
3,layer1.0.conv2,FP8_E4M3FN,secded
4,layer1.1.conv1,FP8_E5M2,secded
5,layer1.1.conv2,INT8,secded
6,layer2.0.conv1,FP16,secded
7,layer2.0.conv2,FP8_E4M3FN,secded
8,layer2.0.downsample.0,FP16,secded
9,layer2.1.conv1,INT8,secded



E4M11 codec loaded:
/kaggle/working/JFAMP/codec/e4m11_codec.py

CIFAR-100 root: /kaggle/working/JFAMP/data

Original validation split verified.
Validation examples: 5000
Original JSON SHA256: b632b03ccc8166092cbf5c9e02a41c754248533ef1f60292c0404e3a8d80d0e7
CIFAR-100 full training archive: 50000
Original training samples: 45000
Original validation samples: 5000
Validation batches: 40
Original split SHA256: b632b03ccc8166092cbf5c9e02a41c754248533ef1f60292c0404e3a8d80d0e7

Validation dataloader created successfully.

Canonical checkpoint loaded successfully.
Missing keys: []
Unexpected keys: []

FP32 BASELINE EVALUATION
FP32 accuracy: 77.78 %
FP32 loss: 0.91432507
Correct predictions: 3889 / 5000
Previously recorded baseline: 77.78 %
Evaluation time: 2.39 seconds

APPLYING REFINED MIXED-PRECISION CONFIGURATION
conv1                          INT8               secded
fc                             FP16               secded
layer1.0.conv1                 INT8               secded
layer1.0.

,layer,datatype,protection,weights,datatype_bits_per_weight,protected_bits_per_weight,data_storage_bits,protected_storage_bits
0,conv1,INT8,secded,1728,8,13,13824,22464
1,fc,FP16,secded,51200,16,22,819200,1126400
2,layer1.0.conv1,INT8,secded,36864,8,13,294912,479232
3,layer1.0.conv2,FP8_E4M3FN,secded,36864,8,13,294912,479232
4,layer1.1.conv1,FP8_E5M2,secded,36864,8,13,294912,479232
5,layer1.1.conv2,INT8,secded,36864,8,13,294912,479232
6,layer2.0.conv1,FP16,secded,73728,16,22,1179648,1622016
7,layer2.0.conv2,FP8_E4M3FN,secded,147456,8,13,1179648,1916928
8,layer2.0.downsample.0,FP16,secded,8192,16,22,131072,180224
9,layer2.1.conv1,INT8,secded,147456,8,13,1179648,1916928



Quantized model saved: /kaggle/working/JFAMP/experiments/end_to_end_validation/resnet18_jfamp_refined_dequantized_fp32.pth

JFAMP EXPERIMENT 8 COMPLETED
FP32 baseline accuracy: 77.78 %
Refined model accuracy: 77.88 %
Accuracy delta: 0.1 pp
FP32 loss: 0.91432507
Refined model loss: 0.91851938
Storage reduction: 54.0445 %

Output directory:
/kaggle/working/JFAMP/experiments/end_to_end_validation

Generated files:
configuration_used.csv                                 1.76 KB
end_to_end_layer_quantization.csv                      1.74 KB
end_to_end_storage_report.csv                          1.27 KB
end_to_end_validation_results.json                     1.85 KB
resnet18_jfamp_refined_dequantized_fp32.pth            43913.95 KB

Experiment 8 finished successfully.


In [28]:

# =====================================================================
# JFAMP EXPERIMENT 9A
# ACTUAL MIXED-PRECISION ENCODING + SECDED VALIDATION
#
# Purpose:
#   Validate ECC on the actual selected weight representations.
#
# Datatypes:
#   INT8, INT4, FP16, FP8_E4M3FN, FP8_E5M2
#
# ECC:
#   SECDED: single-error correction, double-error detection
#
# This stage does not yet perform full-network fault injection.
# =====================================================================

import os
import json
import time
import hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from torchvision.models import resnet18

# ---------------------------------------------------------------------
# 1. PATHS AND CONFIGURATION
# ---------------------------------------------------------------------

ROOT = Path("/kaggle/working/JFAMP")

CHECKPOINT = (
    ROOT / "retrain_to150/checkpoints/"
    "resnet18_cifar100_best_150.pth"
)

CONFIG_FILE = (
    ROOT / "experiments/joint_optimization_refined/"
    "refined_layer_configuration.csv"
)

OUT_DIR = (
    ROOT / "experiments/experiment_9A_ecc_validation"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 75)
print("JFAMP EXPERIMENT 9A")
print("ACTUAL MIXED-PRECISION ENCODING + SECDED VALIDATION")
print("=" * 75)

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

# ---------------------------------------------------------------------
# 2. VERIFY INPUT FILES
# ---------------------------------------------------------------------

assert CHECKPOINT.exists(), CHECKPOINT
assert CONFIG_FILE.exists(), CONFIG_FILE

config = pd.read_csv(CONFIG_FILE)

assert len(config) == 21
assert config["layer"].nunique() == 21

print("\nConfiguration loaded:", len(config), "layers")

# ---------------------------------------------------------------------
# 3. BUILD AND LOAD CANONICAL MODEL
# ---------------------------------------------------------------------

def build_model():

    model = resnet18(weights=None)

    model.conv1 = nn.Conv2d(
        3, 64,
        kernel_size=3,
        stride=1,
        padding=1,
        bias=False
    )

    model.maxpool = nn.Identity()
    model.fc = nn.Linear(512, 100)

    return model


checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

def extract_state_dict(obj):

    if isinstance(obj, dict):

        for key in [
            "state_dict",
            "model_state_dict",
            "model",
            "net",
            "weights"
        ]:

            if key in obj and isinstance(obj[key], dict):

                if any(torch.is_tensor(v) for v in obj[key].values()):
                    return obj[key]

        if any(torch.is_tensor(v) for v in obj.values()):
            return obj

    raise ValueError("Could not find model state_dict.")


state_dict = extract_state_dict(checkpoint)

state_dict = {
    k.removeprefix("module."): v
    for k, v in state_dict.items()
}

model = build_model()
model.load_state_dict(state_dict, strict=True)
model.eval()

modules = dict(model.named_modules())

print("Canonical checkpoint loaded.")

# ---------------------------------------------------------------------
# 4. DATATYPE ENCODING
# ---------------------------------------------------------------------

def encode_weight(weight, datatype):

    w = weight.detach().cpu().float().contiguous()

    if datatype == "FP16":

        q = w.to(torch.float16)

        symbols = (
            q.contiguous()
            .view(torch.uint16)
            .numpy()
            .astype(np.uint32)
        )

        return symbols, {
            "format": "FP16",
            "bits": 16
        }

    elif datatype == "FP8_E4M3FN":

        q = w.to(torch.float8_e4m3fn)

        symbols = (
            q.contiguous()
            .view(torch.uint8)
            .numpy()
            .astype(np.uint32)
        )

        return symbols, {
            "format": datatype,
            "bits": 8
        }

    elif datatype == "FP8_E5M2":

        q = w.to(torch.float8_e5m2)

        symbols = (
            q.contiguous()
            .view(torch.uint8)
            .numpy()
            .astype(np.uint32)
        )

        return symbols, {
            "format": datatype,
            "bits": 8
        }

    elif datatype == "INT8":

        flat = w.reshape(w.shape[0], -1)

        scale = (
            flat.abs().amax(dim=1).clamp(min=1e-12)
            / 127.0
        )

        scale_shape = (
            [w.shape[0]]
            + [1] * (w.ndim - 1)
        )

        scale_view = scale.reshape(scale_shape)

        q = torch.round(w / scale_view)
        q = torch.clamp(q, -127, 127).to(torch.int8)

        symbols = (
            q.contiguous()
            .view(torch.uint8)
            .numpy()
            .astype(np.uint32)
        )

        return symbols, {
            "format": datatype,
            "bits": 8,
            "scale": scale.numpy().tolist(),
            "scale_type": "per_output_channel_symmetric"
        }

    elif datatype == "INT4":

        scale = (
            w.abs().max().clamp(min=1e-12)
            / 7.0
        )

        q = torch.round(w / scale)
        q = torch.clamp(q, -7, 7).to(torch.int8)

        # Store each signed INT4 value in its low four bits.
        symbols = (
            q.contiguous()
            .view(torch.uint8)
            .numpy()
            .astype(np.uint32)
            & 0x0F
        )

        return symbols, {
            "format": datatype,
            "bits": 4,
            "scale": float(scale.item()),
            "scale_type": "per_tensor_symmetric"
        }

    else:
        raise ValueError(
            f"Unsupported datatype: {datatype}"
        )


# ---------------------------------------------------------------------
# 5. VECTORISED SECDED ENCODER
# ---------------------------------------------------------------------

def get_ecc_layout(data_bits):

    r = 0

    while 2**r < data_bits + r + 1:
        r += 1

    parity_positions = [
        2**i for i in range(r)
    ]

    total_hamming_bits = data_bits + r

    data_positions = [
        p for p in range(1, total_hamming_bits + 1)
        if p not in parity_positions
    ]

    overall_position = total_hamming_bits + 1

    return (
        r,
        parity_positions,
        data_positions,
        overall_position
    )


def encode_secded(symbols, data_bits):

    symbols = np.asarray(
        symbols,
        dtype=np.uint32
    ).reshape(-1)

    r, parity_positions, data_positions, overall_position = (
        get_ecc_layout(data_bits)
    )

    code = np.zeros(
        symbols.shape,
        dtype=np.uint32
    )

    # Place data bits.
    for bit_index, position in enumerate(data_positions):

        bit = (
            (symbols >> bit_index) & 1
        )

        code |= (
            bit << (position - 1)
        )

    # Generate Hamming parity bits.
    for parity_position in parity_positions:

        parity = np.zeros(
            symbols.shape,
            dtype=np.uint32
        )

        for position in range(1, data_bits + r + 1):

            if position & parity_position:

                parity ^= (
                    (code >> (position - 1)) & 1
                )

        code |= (
            parity << (parity_position - 1)
        )

    # Generate overall parity.
    overall = np.zeros(
        symbols.shape,
        dtype=np.uint32
    )

    for position in range(1, data_bits + r + 1):

        overall ^= (
            (code >> (position - 1)) & 1
        )

    code |= (
        overall << (overall_position - 1)
    )

    return code


# ---------------------------------------------------------------------
# 6. VECTORISED SECDED DECODER
# ---------------------------------------------------------------------

def decode_secded(code, data_bits):

    code = np.asarray(
        code,
        dtype=np.uint32
    ).copy().reshape(-1)

    r, parity_positions, data_positions, overall_position = (
        get_ecc_layout(data_bits)
    )

    syndrome = np.zeros(
        code.shape,
        dtype=np.uint32
    )

    # Calculate syndrome.
    for parity_position in parity_positions:

        parity = np.zeros(
            code.shape,
            dtype=np.uint32
        )

        for position in range(1, data_bits + r + 1):

            if position & parity_position:

                parity ^= (
                    (code >> (position - 1)) & 1
                )

        syndrome |= (
            parity * parity_position
        )

    # Calculate overall parity.
    overall = np.zeros(
        code.shape,
        dtype=np.uint32
    )

    for position in range(1, overall_position + 1):

        overall ^= (
            (code >> (position - 1)) & 1
        )

    # Single-bit error: non-zero syndrome and odd overall parity.
    correctable = (
        (syndrome != 0)
        & (overall == 1)
        & (syndrome <= data_bits + r)
    )

    # Overall parity-bit error: zero syndrome and odd overall parity.
    overall_bit_error = (
        (syndrome == 0)
        & (overall == 1)
    )

    # Double-bit error: non-zero syndrome and even overall parity.
    double_error = (
        (syndrome != 0)
        & (overall == 0)
    )

    # Correct the indicated Hamming codeword bit.
    indices = np.flatnonzero(correctable)

    if len(indices) > 0:

        bit_positions = syndrome[indices] - 1

        code[indices] ^= (
            np.uint32(1) << bit_positions
        )

    # Correct the overall parity bit.
    indices = np.flatnonzero(overall_bit_error)

    if len(indices) > 0:

        code[indices] ^= (
            np.uint32(1) << (
                overall_position - 1
            )
        )

    # Extract data.
    decoded = np.zeros(
        code.shape,
        dtype=np.uint32
    )

    for bit_index, position in enumerate(data_positions):

        bit = (
            (code >> (position - 1)) & 1
        )

        decoded |= (
            bit << bit_index
        )

    status = {
        "corrected": int(
            correctable.sum()
            + overall_bit_error.sum()
        ),
        "double_detected": int(
            double_error.sum()
        ),
        "uncorrectable": int(
            double_error.sum()
        )
    }

    return decoded, status


# ---------------------------------------------------------------------
# 7. VALIDATE REPRESENTATIVE SINGLE AND DOUBLE BIT ERRORS
# ---------------------------------------------------------------------

def test_ecc(data_bits):

    sample_values = np.array(
        [
            0,
            1,
            3,
            5,
            7,
            15,
            (1 << data_bits) - 1,
            (1 << data_bits) // 2
        ],
        dtype=np.uint32
    )

    encoded = encode_secded(
        sample_values,
        data_bits
    )

    # No-fault roundtrip.
    decoded, status = decode_secded(
        encoded,
        data_bits
    )

    assert np.array_equal(
        decoded,
        sample_values
    ), "No-fault SECDED roundtrip failed."

    r, parity_positions, data_positions, overall_position = (
        get_ecc_layout(data_bits)
    )

    total_bits = overall_position

    # Test every single-bit position on a representative codeword.
    for bit_position in range(total_bits):

        corrupted = encoded[:1].copy()

        corrupted[0] ^= (
            np.uint32(1) << bit_position
        )

        recovered, result = decode_secded(
            corrupted,
            data_bits
        )

        assert recovered[0] == sample_values[0], (
            f"Single-bit correction failed at bit {bit_position}"
        )

    # Test a representative double-bit error.
    corrupted = encoded[:1].copy()

    corrupted[0] ^= (
        np.uint32(1) << 0
    )

    corrupted[0] ^= (
        np.uint32(1) << 1
    )

    _, double_result = decode_secded(
        corrupted,
        data_bits
    )

    assert double_result["double_detected"] == 1, (
        "Representative double-bit detection failed."
    )

    return {
        "data_bits": data_bits,
        "codeword_bits": total_bits,
        "single_bit_positions_tested": total_bits,
        "single_bit_correction": "PASS",
        "double_bit_detection": "PASS",
        "no_fault_roundtrip": "PASS"
    }


# ---------------------------------------------------------------------
# 8. RUN ECC UNIT TESTS
# ---------------------------------------------------------------------

print("\n" + "=" * 75)
print("SECDED UNIT TESTS")
print("=" * 75)

ecc_tests = []

for bits in [4, 8, 16]:

    result = test_ecc(bits)

    ecc_tests.append(result)

    print(
        f"{bits}-bit data -> "
        f"{result['codeword_bits']}-bit SECDED codeword: "
        "PASS"
    )

# ---------------------------------------------------------------------
# 9. ENCODE ALL 21 ACTUAL MIXED-PRECISION LAYERS
# ---------------------------------------------------------------------

print("\n" + "=" * 75)
print("ACTUAL MIXED-PRECISION LAYER ENCODING")
print("=" * 75)

layer_results = []

total_weights = 0
total_data_bits = 0
total_ecc_bits = 0

start_time = time.time()

for _, row in config.iterrows():

    layer_name = str(row["layer"])
    datatype = str(row["datatype"])
    protection = str(row["protection"]).lower()

    assert protection == "secded", (
        f"Unexpected protection for {layer_name}: {protection}"
    )

    assert layer_name in modules, (
        f"Layer not found: {layer_name}"
    )

    module = modules[layer_name]

    assert isinstance(
        module,
        (nn.Conv2d, nn.Linear)
    )

    original_weight = module.weight.detach()

    symbols, metadata = encode_weight(
    original_weight,
    datatype
)

    data_bits = metadata["bits"]

    # Verify the total number of encoded weight symbols.
    assert symbols.size == original_weight.numel(), (
        f"Encoded symbol count mismatch in {layer_name}: "
        f"symbols={symbols.size}, "
        f"weights={original_weight.numel()}")

    # Flatten before ECC processing.
    symbols = symbols.reshape(-1)

    # Actual SECDED codeword generation.
    codewords = encode_secded(
        symbols,
        data_bits
    )

    # Full no-fault decoding.
    recovered_symbols, decode_status = decode_secded(
        codewords,
        data_bits
    )

    exact_recovery = np.array_equal(
        symbols,
        recovered_symbols
    )

    assert exact_recovery, (
        f"No-fault ECC recovery failed: {layer_name}"
    )

    r, _, _, overall_position = get_ecc_layout(
        data_bits
    )

    codeword_bits = overall_position

    weight_count = len(symbols)

    data_storage = (
        weight_count * data_bits
    )

    ecc_storage = (
        weight_count * codeword_bits
    )

    total_weights += weight_count
    total_data_bits += data_storage
    total_ecc_bits += ecc_storage

    layer_results.append({

        "layer": layer_name,

        "datatype": datatype,

        "protection": protection,

        "weights": weight_count,

        "data_bits_per_weight": data_bits,

        "secded_bits_per_weight": codeword_bits,

        "data_storage_bits": data_storage,

        "secded_storage_bits": ecc_storage,

        "no_fault_exact_recovery": exact_recovery,

        "no_fault_decoder_corrected": (
            decode_status["corrected"]
        ),

        "no_fault_double_errors": (
            decode_status["double_detected"]
        )
    })

    print(
        f"{layer_name:30s} "
        f"{datatype:15s} "
        f"weights={weight_count:>9,} "
        f"codeword={codeword_bits:>2} bits "
        f"PASS"
    )

    # Release temporary arrays before the next layer.
    del symbols
    del codewords
    del recovered_symbols


# ---------------------------------------------------------------------
# 10. SUMMARY
# ---------------------------------------------------------------------

layer_df = pd.DataFrame(layer_results)

assert len(layer_df) == 21

assert layer_df[
    "no_fault_exact_recovery"
].all()

assert layer_df[
    "no_fault_decoder_corrected"
].sum() == 0

assert layer_df[
    "no_fault_double_errors"
].sum() == 0

fp32_storage_bits = total_weights * 32

storage_reduction = (
    1.0 - total_ecc_bits / fp32_storage_bits
) * 100.0

summary = {

    "experiment": "JFAMP Experiment 9A",

    "description": (
        "Actual datatype encoding and SECDED verification"
    ),

    "layers_processed": len(layer_df),

    "total_weights": int(total_weights),

    "total_data_bits": int(total_data_bits),

    "total_secded_bits": int(total_ecc_bits),

    "fp32_reference_bits": int(fp32_storage_bits),

    "fp32_reference_mib": (
        fp32_storage_bits / 8 / 1024**2
    ),

    "mixed_precision_secded_mib": (
        total_ecc_bits / 8 / 1024**2
    ),

    "storage_reduction_percentage": float(
        storage_reduction
    ),

    "all_layers_exact_no_fault_recovery": bool(
        layer_df["no_fault_exact_recovery"].all()
    ),

    "ecc_unit_tests": ecc_tests,

    "double_bit_detection_scope": (
        "Representative double-bit test per datatype width"
    ),

    "single_bit_correction_scope": (
        "Every codeword bit position tested per datatype width"
    ),

    "full_network_fault_injection_performed": False
}

layer_df.to_csv(
    OUT_DIR / "experiment_9A_layer_encoding.csv",
    index=False
)

with open(
    OUT_DIR / "experiment_9A_summary.json",
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

with open(
    OUT_DIR / "experiment_9A_ecc_unit_tests.json",
    "w"
) as f:

    json.dump(
        ecc_tests,
        f,
        indent=2
    )

# ---------------------------------------------------------------------
# 11. FINAL OUTPUT
# ---------------------------------------------------------------------

print("\n" + "=" * 75)
print("EXPERIMENT 9A COMPLETED")
print("=" * 75)

print("Layers processed:", len(layer_df))
print("Total weights:", f"{total_weights:,}")

print(
    "Mixed-precision SECDED storage:",
    round(total_ecc_bits / 8 / 1024**2, 4),
    "MiB"
)

print(
    "FP32 reference storage:",
    round(fp32_storage_bits / 8 / 1024**2, 4),
    "MiB"
)

print(
    "Storage reduction:",
    round(storage_reduction, 4),
    "%"
)

print(
    "All layers exact no-fault recovery:",
    summary["all_layers_exact_no_fault_recovery"]
)

print(
    "Elapsed time:",
    round(time.time() - start_time, 2),
    "seconds"
)

print("\nSaved outputs:")

for path in sorted(OUT_DIR.iterdir()):
    print(path.name)

print("\nAll Experiment 9A checks passed.")


JFAMP EXPERIMENT 9A
ACTUAL MIXED-PRECISION ENCODING + SECDED VALIDATION
PyTorch: 2.11.0+cu130
Device: cuda

Configuration loaded: 21 layers
Canonical checkpoint loaded.

SECDED UNIT TESTS
4-bit data -> 8-bit SECDED codeword: PASS
8-bit data -> 13-bit SECDED codeword: PASS
16-bit data -> 22-bit SECDED codeword: PASS

ACTUAL MIXED-PRECISION LAYER ENCODING
conv1                          INT8            weights=    1,728 codeword=13 bits PASS
fc                             FP16            weights=   51,200 codeword=22 bits PASS
layer1.0.conv1                 INT8            weights=   36,864 codeword=13 bits PASS
layer1.0.conv2                 FP8_E4M3FN      weights=   36,864 codeword=13 bits PASS
layer1.1.conv1                 FP8_E5M2        weights=   36,864 codeword=13 bits PASS
layer1.1.conv2                 INT8            weights=   36,864 codeword=13 bits PASS
layer2.0.conv1                 FP16            weights=   73,728 codeword=22 bits PASS
layer2.0.conv2                 FP8_

In [29]:

# =============================================================================
# JFAMP EXPERIMENT 9B
# END-TO-END MIXED-PRECISION FAULT INJECTION AND ECC RECOVERY
#
# Model: ResNet-18 / CIFAR-100
# Checkpoint: canonical epoch-139 best checkpoint
# Configuration: Experiment 7B refined joint optimization
# Validation: original 5,000-image validation split
#
# Fault scenarios:
#   1. Single-bit payload corruption
#   2. Double-bit payload corruption
#
# Protection strategies:
#   1. None
#   2. Parity
#   3. SECDED
#   4. DMR
#
# Faults: 100 randomly selected weights per layer per scenario.
# All strategies receive matched fault locations and bit positions.
# =============================================================================

import os
import json
import csv
import time
import hashlib
import random
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

warnings.filterwarnings("ignore")

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ROOT = Path("/kaggle/working/JFAMP")
INPUT_ROOT = Path(
    "/kaggle/input/datasets/crahulnaidu/"
    "jfamp-resnet18-cifar100-checkpoint"
)

CHECKPOINT = INPUT_ROOT / "resnet18_cifar100_best.pth"
SPLIT_FILE = INPUT_ROOT / "data_split_indices.json"

CONFIG_FILE = (
    ROOT / "experiments/joint_optimization_refined/"
    "refined_layer_configuration.csv"
)

OUTPUT = ROOT / "experiments/experiment_9B_fault_recovery"
OUTPUT.mkdir(parents=True, exist_ok=True)

N_FAULTS_PER_LAYER = 100
BATCH_SIZE = 128
NUM_WORKERS = 2

print("=" * 79)
print("JFAMP EXPERIMENT 9B")
print("ACTUAL MIXED-PRECISION FAULT INJECTION + ECC RECOVERY")
print("=" * 79)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# =============================================================================
# 1. LOAD CANONICAL CHECKPOINT AND OPTIMIZED CONFIGURATION
# =============================================================================

assert CHECKPOINT.exists(), f"Missing checkpoint: {CHECKPOINT}"
assert SPLIT_FILE.exists(), f"Missing split file: {SPLIT_FILE}"
assert CONFIG_FILE.exists(), f"Missing refined configuration: {CONFIG_FILE}"

with open(SPLIT_FILE, "r") as f:
    split_json = json.load(f)

validation_indices = split_json["validation_indices"]

raw_split_hash = hashlib.sha256(SPLIT_FILE.read_bytes()).hexdigest()
EXPECTED_HASH = (
    "b632b03ccc8166092cbf5c9e02a41c754248533ef1f60292c0404e3a8d80d0e7"
)
assert raw_split_hash == EXPECTED_HASH, (
    f"Canonical split hash mismatch: {raw_split_hash}"
)

config_df = pd.read_csv(CONFIG_FILE)

def find_column(df, candidates):
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
    raise ValueError(
        f"Could not find any of {candidates}. "
        f"Available columns: {list(df.columns)}"
    )

layer_col = find_column(
    config_df,
    ["layer", "layer_name", "module", "module_name"]
)

dtype_col = find_column(
    config_df,
    ["datatype", "data_type", "precision", "selected_datatype"]
)

layer_config = {
    str(row[layer_col]): str(row[dtype_col]).upper()
    for _, row in config_df.iterrows()
}

print(f"\nConfiguration loaded: {len(layer_config)} layers")

checkpoint_obj = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

if isinstance(checkpoint_obj, dict):
    for key in [
        "model_state_dict",
        "state_dict",
        "model",
        "net"
    ]:
        if key in checkpoint_obj and isinstance(checkpoint_obj[key], dict):
            checkpoint_obj = checkpoint_obj[key]
            break

# Remove DataParallel prefixes if present.
state_dict = {
    k.replace("module.", "", 1) if k.startswith("module.") else k: v
    for k, v in checkpoint_obj.items()
    if isinstance(v, torch.Tensor)
}

model = models.resnet18(weights=None)
model.conv1 = nn.Conv2d(
    3, 64, kernel_size=3, stride=1, padding=1, bias=False
)
model.maxpool = nn.Identity()
model.fc = nn.Linear(model.fc.in_features, 100)

model.load_state_dict(state_dict, strict=True)
model = model.to(DEVICE).eval()

print("Canonical checkpoint loaded.")


# =============================================================================
# 2. LOAD ORIGINAL CIFAR-100 VALIDATION SPLIT
# =============================================================================

possible_roots = [
    ROOT / "data",
    Path("/kaggle/input"),
    Path("/kaggle/working")
]

dataset_root = None

for search_root in possible_roots:
    if not search_root.exists():
        continue

    for candidate in search_root.rglob("cifar-100-python"):
        if (candidate / "train").exists() and (candidate / "test").exists():
            dataset_root = candidate.parent
            break

    if dataset_root is not None:
        break

assert dataset_root is not None, (
    "Could not locate extracted CIFAR-100 data containing "
    "'cifar-100-python/train' and 'test'."
)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

full_train_dataset = datasets.CIFAR100(
    root=str(dataset_root),
    train=True,
    download=False,
    transform=transform
)

assert len(full_train_dataset) == 50000
assert len(validation_indices) == 5000

val_dataset = Subset(full_train_dataset, validation_indices)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0)
)

print("Dataset root:", dataset_root)
print("Validation samples:", len(val_dataset))
print("Raw split SHA256:", raw_split_hash)


# =============================================================================
# 3. MIXED-PRECISION ENCODING AND DECODING
# =============================================================================

FLOAT8_AVAILABLE = (
    hasattr(torch, "float8_e4m3fn")
    and hasattr(torch, "float8_e5m2")
)

def encode_layer_weight(weight, datatype):
    """
    Return:
        symbols: flattened unsigned integer payload symbols
        metadata: information required to reconstruct FP32 weights
        data_width: payload width per weight
    """

    w = weight.detach().cpu().float().contiguous()
    shape = tuple(w.shape)

    if datatype == "FP16":
        q = w.to(torch.float16)
        symbols = q.view(torch.uint16).numpy().astype(np.uint32).reshape(-1)
        return symbols, {"shape": shape}, 16

    if datatype in ("FP8_E4M3FN", "FP8_E5M2"):
        if not FLOAT8_AVAILABLE:
            raise RuntimeError("PyTorch float8 support is unavailable.")

        dtype = (
            torch.float8_e4m3fn
            if datatype == "FP8_E4M3FN"
            else torch.float8_e5m2
        )

        q = w.to(dtype)
        symbols = q.view(torch.uint8).numpy().astype(np.uint32).reshape(-1)

        if not torch.isfinite(q.float()).all():
            raise RuntimeError(f"Non-finite values generated in {datatype}")

        return symbols, {"shape": shape, "dtype": datatype}, 8

    if datatype == "INT8":
        # Symmetric per-output-channel quantization.
        reduce_dims = tuple(range(1, w.ndim))

        max_abs = w.abs().amax(dim=reduce_dims, keepdim=True)
        scale = torch.clamp(max_abs / 127.0, min=1e-12)

        q = torch.round(w / scale).clamp(-127, 127).to(torch.int8)

        symbols = (
            q.contiguous()
            .view(torch.uint8)
            .numpy()
            .astype(np.uint32)
            .reshape(-1)
        )

        return symbols, {
            "shape": shape,
            "scale": scale.numpy()
        }, 8

    if datatype == "INT4":
        # Symmetric per-tensor quantization.
        max_abs = w.abs().max()
        scale = max(float(max_abs.item()) / 7.0, 1e-12)

        q = torch.round(w / scale).clamp(-7, 7).to(torch.int8)

        # Four-bit two's-complement representation.
        symbols = (
            q.numpy().astype(np.int16).reshape(-1) & 0x0F
        ).astype(np.uint32)

        return symbols, {
            "shape": shape,
            "scale": scale
        }, 4

    raise ValueError(f"Unsupported datatype: {datatype}")


def decode_layer_weight(symbols, metadata, datatype):
    """Reconstruct a FP32 weight tensor from payload symbols."""

    shape = metadata["shape"]
    symbols = np.asarray(symbols, dtype=np.uint32).reshape(-1)

    if datatype == "FP16":
        raw = symbols.astype(np.uint16).reshape(shape)
        return torch.from_numpy(raw.copy()).view(torch.float16).float()

    if datatype in ("FP8_E4M3FN", "FP8_E5M2"):
        raw = symbols.astype(np.uint8).reshape(shape)
        q = torch.from_numpy(raw.copy()).view(
            torch.float8_e4m3fn
            if datatype == "FP8_E4M3FN"
            else torch.float8_e5m2
        )
        return q.float()

    if datatype == "INT8":
        raw = symbols.astype(np.uint8).reshape(shape)
        q = torch.from_numpy(raw.copy()).view(torch.int8).float()
        scale = torch.from_numpy(metadata["scale"]).float()
        return q * scale

    if datatype == "INT4":
        signed = symbols.astype(np.int16)
        signed = np.where(signed >= 8, signed - 16, signed)
        q = torch.from_numpy(signed.astype(np.float32).reshape(shape))
        return q * float(metadata["scale"])

    raise ValueError(datatype)


# =============================================================================
# 4. SECDED IMPLEMENTATION
# =============================================================================

def get_hamming_parity_count(data_width):
    r = 0
    while 2 ** r < data_width + r + 1:
        r += 1
    return r


def data_positions(data_width):
    r = get_hamming_parity_count(data_width)
    n = data_width + r

    return [
        pos for pos in range(1, n + 1)
        if (pos & (pos - 1)) != 0
    ]


def secded_encode(value, data_width):
    """Encode one payload symbol into a systematic SECDED codeword."""

    value = int(value)
    r = get_hamming_parity_count(data_width)
    n = data_width + r

    code = 0
    positions = data_positions(data_width)

    for bit_index, pos in enumerate(positions):
        if (value >> bit_index) & 1:
            code |= (1 << (pos - 1))

    # Hamming parity bits.
    for p in range(r):
        parity_pos = 1 << p
        parity = 0

        for pos in range(1, n + 1):
            if pos & parity_pos:
                parity ^= (code >> (pos - 1)) & 1

        if parity:
            code |= (1 << (parity_pos - 1))

    # Overall parity bit at position n+1.
    overall = code.bit_count() & 1

    if overall:
        code |= (1 << n)

    return code


def secded_decode(code, data_width):
    """
    Returns:
        decoded_data
        status: clean / corrected / detected_double
    """

    r = get_hamming_parity_count(data_width)
    n = data_width + r

    syndrome = 0

    for p in range(r):
        parity_pos = 1 << p
        parity = 0

        for pos in range(1, n + 1):
            if pos & parity_pos:
                parity ^= (code >> (pos - 1)) & 1

        if parity:
            syndrome |= parity_pos

    overall_parity = code.bit_count() & 1

    status = "clean"

    if syndrome != 0 and overall_parity == 1:
        # Single error in Hamming portion.
        if syndrome <= n:
            code ^= (1 << (syndrome - 1))
            status = "corrected"
        else:
            status = "detected_double"

    elif syndrome == 0 and overall_parity == 1:
        # Single error in overall parity bit.
        code ^= (1 << n)
        status = "corrected"

    elif syndrome != 0 and overall_parity == 0:
        # Double error: detected but not corrected.
        status = "detected_double"

    decoded = 0
    positions = data_positions(data_width)

    for bit_index, pos in enumerate(positions):
        decoded |= (
            ((code >> (pos - 1)) & 1) << bit_index
        )

    return decoded, status


# =============================================================================
# 5. PARITY AND DMR IMPLEMENTATIONS
# =============================================================================

def parity_encode(value, data_width):
    value = int(value)
    parity = value.bit_count() & 1
    return value | (parity << data_width)


def parity_decode(code, data_width):
    value = code & ((1 << data_width) - 1)
    stored_parity = (code >> data_width) & 1
    calculated_parity = value.bit_count() & 1

    detected = stored_parity != calculated_parity

    return value, detected


def map_payload_bit_to_secded_position(bit_index, data_width):
    """Map a payload bit index to its one-based Hamming position."""
    return data_positions(data_width)[bit_index]


# =============================================================================
# 6. CREATE MATCHED FAULT LOCATIONS FOR EVERY LAYER
# =============================================================================

layer_modules = dict(model.named_modules())

quantized_clean_weights = {}
layer_metadata = {}
layer_symbols = {}
layer_widths = {}

fault_plan = {}

fault_scenarios = {
    "single_bit": 1,
    "double_bit": 2
}

for layer_name, datatype in layer_config.items():

    assert layer_name in layer_modules, (
        f"Layer not found in model: {layer_name}"
    )

    module = layer_modules[layer_name]

    assert isinstance(module, (nn.Conv2d, nn.Linear)), (
        f"Unexpected module type: {layer_name}"
    )

    symbols, metadata, data_width = encode_layer_weight(
        module.weight, datatype
    )

    assert symbols.size == module.weight.numel()

    # Verify clean roundtrip before injecting faults.
    recovered = decode_layer_weight(
        symbols, metadata, datatype
    )

    assert torch.isfinite(recovered).all()

    quantized_clean_weights[layer_name] = recovered
    layer_metadata[layer_name] = metadata
    layer_symbols[layer_name] = symbols.copy()
    layer_widths[layer_name] = data_width

    rng = np.random.default_rng(
        SEED + sum(ord(c) for c in layer_name)
    )

    n_weights = symbols.size
    n_faults = min(N_FAULTS_PER_LAYER, n_weights)

    selected_indices = rng.choice(
        n_weights,
        size=n_faults,
        replace=False
    )

    plan = {}

    for scenario, n_bits in fault_scenarios.items():

        bit_positions = np.zeros(
            (n_faults, n_bits),
            dtype=np.int32
        )

        for i in range(n_faults):
            bit_positions[i] = rng.choice(
                data_width,
                size=n_bits,
                replace=False
            )

        plan[scenario] = {
            "indices": selected_indices.copy(),
            "bit_positions": bit_positions
        }

    fault_plan[layer_name] = plan

print("\nAll 21 layers converted to their selected formats.")
print("Clean mixed-precision reconstruction completed.")


# =============================================================================
# 7. BUILD CLEAN MIXED-PRECISION MODEL
# =============================================================================

def install_weights(target_model, weights_by_layer):
    modules = dict(target_model.named_modules())

    with torch.no_grad():
        for name, weight in weights_by_layer.items():
            modules[name].weight.copy_(
                weight.to(
                    device=modules[name].weight.device,
                    dtype=modules[name].weight.dtype
                )
            )


clean_model = models.resnet18(weights=None)
clean_model.conv1 = nn.Conv2d(
    3, 64, kernel_size=3, stride=1, padding=1, bias=False
)
clean_model.maxpool = nn.Identity()
clean_model.fc = nn.Linear(clean_model.fc.in_features, 100)

clean_model.load_state_dict(state_dict, strict=True)
clean_model = clean_model.to(DEVICE).eval()

install_weights(clean_model, quantized_clean_weights)


# =============================================================================
# 8. EVALUATION FUNCTION
# =============================================================================

@torch.no_grad()
def evaluate_model(eval_model, loader):
    eval_model.eval()

    correct = 0
    total = 0
    loss_sum = 0.0

    criterion = nn.CrossEntropyLoss(reduction="sum")

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        logits = eval_model(images)

        loss_sum += criterion(logits, labels).item()
        predictions = logits.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return {
        "accuracy": 100.0 * correct / total,
        "correct": correct,
        "total": total,
        "loss": loss_sum / total
    }


print("\nEvaluating clean baselines...")

fp32_metrics = evaluate_model(model, val_loader)
clean_mp_metrics = evaluate_model(clean_model, val_loader)

print(
    f"FP32 baseline:              "
    f"{fp32_metrics['accuracy']:.4f}% "
    f"({fp32_metrics['correct']}/{fp32_metrics['total']})"
)

print(
    f"Clean mixed precision:      "
    f"{clean_mp_metrics['accuracy']:.4f}% "
    f"({clean_mp_metrics['correct']}/{clean_mp_metrics['total']})"
)


# =============================================================================
# 9. APPLY MATCHED FAULTS AND PROTECTION
# =============================================================================

STRATEGIES = ["none", "parity", "secded", "dmr"]

fault_results = []
mechanism_results = []
network_results = []

fault_status_totals = defaultdict(lambda: defaultdict(int))

# Build the original quantized weights for each evaluation.
for scenario in fault_scenarios:

    print("\n" + "=" * 79)
    print("FAULT SCENARIO:", scenario.upper())
    print("=" * 79)

    for strategy in STRATEGIES:

        print(f"\nRunning strategy: {strategy.upper()}")

        corrupted_weights = {}

        strategy_counts = defaultdict(int)
        total_injected = 0
        total_detected = 0
        total_corrected = 0
        total_residual = 0

        for layer_name, datatype in layer_config.items():

            original_symbols = layer_symbols[layer_name]
            symbols = original_symbols.copy()

            data_width = layer_widths[layer_name]
            plan = fault_plan[layer_name][scenario]

            indices = plan["indices"]
            bit_positions = plan["bit_positions"]

            for trial, symbol_index in enumerate(indices):

                original_value = int(original_symbols[symbol_index])
                positions = bit_positions[trial].tolist()

                total_injected += len(positions)

                if strategy == "none":

                    corrupted_value = original_value

                    for bit in positions:
                        corrupted_value ^= (1 << bit)

                    symbols[symbol_index] = corrupted_value
                    total_residual += 1

                    status = "unprotected"

                elif strategy == "parity":

                    code = parity_encode(
                        original_value, data_width
                    )

                    for bit in positions:
                        code ^= (1 << bit)

                    decoded, detected = parity_decode(
                        code, data_width
                    )

                    symbols[symbol_index] = decoded

                    if detected:
                        total_detected += 1
                        status = "detected_not_corrected"
                    else:
                        status = "undetected"

                    # Parity detection alone does not recover the value.
                    if decoded != original_value:
                        total_residual += 1

                elif strategy == "secded":

                    code = secded_encode(
                        original_value, data_width
                    )

                    # Inject into payload locations in the Hamming codeword.
                    for bit in positions:
                        code_position = (
                            map_payload_bit_to_secded_position(
                                bit, data_width
                            )
                        )
                        code ^= (1 << (code_position - 1))

                    decoded, status = secded_decode(
                        code, data_width
                    )

                    symbols[symbol_index] = decoded

                    if status == "corrected":
                        total_detected += 1
                        total_corrected += 1

                    elif status == "detected_double":
                        total_detected += 1

                    if decoded != original_value:
                        total_residual += 1

                elif strategy == "dmr":

                    # Two identical copies are stored.
                    copy_a = original_value
                    copy_b = original_value

                    # Fault affects the first copy.
                    for bit in positions:
                        copy_a ^= (1 << bit)

                    mismatch = copy_a != copy_b

                    # DMR detects mismatch but cannot identify which copy
                    # is correct. The policy here selects copy A, so no
                    # correction is claimed.
                    decoded = copy_a
                    symbols[symbol_index] = decoded

                    if mismatch:
                        total_detected += 1
                        status = "detected_not_corrected"
                    else:
                        status = "undetected"

                    if decoded != original_value:
                        total_residual += 1

                else:
                    raise ValueError(strategy)

                strategy_counts[status] += 1

                fault_results.append({
                    "scenario": scenario,
                    "strategy": strategy,
                    "layer": layer_name,
                    "datatype": datatype,
                    "weight_index": int(symbol_index),
                    "payload_bit_positions": ",".join(
                        map(str, positions)
                    ),
                    "original_symbol": original_value,
                    "recovered_symbol": int(symbols[symbol_index]),
                    "status": status,
                    "symbol_recovered_exactly": (
                        int(symbols[symbol_index]) == original_value
                    )
                })

            # Reconstruct this layer from the modified payload.
            corrupted_weights[layer_name] = decode_layer_weight(
                symbols,
                layer_metadata[layer_name],
                datatype
            )

        # Construct a fresh model for each strategy/scenario.
        fault_model = models.resnet18(weights=None)
        fault_model.conv1 = nn.Conv2d(
            3, 64, kernel_size=3, stride=1, padding=1, bias=False
        )
        fault_model.maxpool = nn.Identity()
        fault_model.fc = nn.Linear(
            fault_model.fc.in_features, 100
        )

        fault_model.load_state_dict(state_dict, strict=True)
        fault_model = fault_model.to(DEVICE).eval()

        install_weights(fault_model, corrupted_weights)

        metrics = evaluate_model(fault_model, val_loader)

        delta_vs_clean = (
            metrics["accuracy"] - clean_mp_metrics["accuracy"]
        )

        detection_rate = (
            100.0 * total_detected / total_injected
            if total_injected else 0.0
        )

        correction_rate = (
            100.0 * total_corrected / total_injected
            if total_injected else 0.0
        )

        residual_rate = (
            100.0 * total_residual / (
                total_injected // (
                    1 if scenario == "single_bit" else 2
                )
            )
            if total_injected else 0.0
        )

        network_results.append({
            "scenario": scenario,
            "strategy": strategy,
            "accuracy": metrics["accuracy"],
            "correct": metrics["correct"],
            "loss": metrics["loss"],
            "delta_vs_fp32_pp": (
                metrics["accuracy"] - fp32_metrics["accuracy"]
            ),
            "delta_vs_clean_mp_pp": delta_vs_clean,
            "accuracy_degradation_vs_clean_mp_pp": (
                clean_mp_metrics["accuracy"] - metrics["accuracy"]
            ),
            "detection_rate_percent": detection_rate,
            "correction_rate_percent": correction_rate,
            "residual_corrupted_symbols": total_residual,
            "total_corrupted_symbols": (
                total_injected // (
                    1 if scenario == "single_bit" else 2
                )
            )
        })

        mechanism_results.append({
            "scenario": scenario,
            "strategy": strategy,
            "total_injected_bit_flips": total_injected,
            "detected_events": total_detected,
            "corrected_events": total_corrected,
            "residual_corrupted_symbols": total_residual,
            "status_counts": dict(strategy_counts)
        })

        print(
            f"Accuracy: {metrics['accuracy']:.4f}% | "
            f"Delta vs clean MP: {delta_vs_clean:+.4f} pp"
        )
        print(
            f"Detected events: {total_detected} | "
            f"Corrected events: {total_corrected} | "
            f"Residual corrupted symbols: {total_residual}"
        )

        del fault_model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()


# =============================================================================
# 10. STORAGE OVERHEAD ACCOUNTING
# =============================================================================

storage_rows = []

for layer_name, datatype in layer_config.items():

    n = layer_symbols[layer_name].size
    width = layer_widths[layer_name]

    if width == 4:
        secded_width = 8
    elif width == 8:
        secded_width = 13
    elif width == 16:
        secded_width = 22
    else:
        raise ValueError(width)

    strategy_widths = {
        "none": width,
        "parity": width + 1,
        "secded": secded_width,
        "dmr": width * 2
    }

    for strategy, bits_per_symbol in strategy_widths.items():

        storage_rows.append({
            "layer": layer_name,
            "datatype": datatype,
            "weights": n,
            "strategy": strategy,
            "bits_per_weight": bits_per_symbol,
            "storage_bits": n * bits_per_symbol
        })

storage_df = pd.DataFrame(storage_rows)

storage_summary = (
    storage_df.groupby("strategy", as_index=False)["storage_bits"]
    .sum()
)

fp32_storage_bits = sum(
    layer_symbols[name].size * 32
    for name in layer_config
)

storage_summary["storage_MiB"] = (
    storage_summary["storage_bits"] / 8 / (1024 ** 2)
)

storage_summary["reduction_vs_fp32_percent"] = (
    100.0 * (
        1 - storage_summary["storage_bits"] / fp32_storage_bits
    )
)


# =============================================================================
# 11. SAVE RESULTS
# =============================================================================

fault_df = pd.DataFrame(fault_results)
network_df = pd.DataFrame(network_results)
mechanism_df = pd.DataFrame(mechanism_results)

fault_df.to_csv(
    OUTPUT / "experiment_9B_fault_records.csv",
    index=False
)

network_df.to_csv(
    OUTPUT / "experiment_9B_network_results.csv",
    index=False
)

mechanism_df.to_csv(
    OUTPUT / "experiment_9B_ecc_mechanism_results.csv",
    index=False
)

storage_df.to_csv(
    OUTPUT / "experiment_9B_storage_by_layer.csv",
    index=False
)

storage_summary.to_csv(
    OUTPUT / "experiment_9B_storage_summary.csv",
    index=False
)

summary = {
    "experiment": "JFAMP Experiment 9B",
    "seed": SEED,
    "checkpoint": str(CHECKPOINT),
    "configuration": str(CONFIG_FILE),
    "validation_split_sha256": raw_split_hash,
    "validation_samples": len(val_dataset),
    "layers": len(layer_config),
    "total_weights": int(sum(
        layer_symbols[name].size for name in layer_config
    )),
    "faults_per_layer_per_scenario": N_FAULTS_PER_LAYER,
    "fault_scenarios": list(fault_scenarios.keys()),
    "strategies": STRATEGIES,
    "fp32_baseline": fp32_metrics,
    "clean_mixed_precision": clean_mp_metrics,
    "network_results": network_results,
    "mechanism_results": mechanism_results,
    "storage_summary": storage_summary.to_dict(orient="records"),
    "important_scope_note": (
        "Faults are injected into 100 sampled payload symbols per layer "
        "per scenario. This is a controlled end-to-end experiment, not "
        "an exhaustive bit-level test. DMR detects mismatches but the "
        "evaluation policy deliberately selects the potentially corrupted "
        "first copy; it does not claim correction."
    )
}

with open(
    OUTPUT / "experiment_9B_summary.json",
    "w"
) as f:
    json.dump(summary, f, indent=2)


# =============================================================================
# 12. FINAL REPORT
# =============================================================================

print("\n" + "=" * 79)
print("EXPERIMENT 9B COMPLETED")
print("=" * 79)

print("\nBASELINES")
print(
    f"FP32:                 {fp32_metrics['accuracy']:.4f}%"
)
print(
    f"Clean mixed precision:{clean_mp_metrics['accuracy']:.4f}%"
)

print("\nEND-TO-END FAULT RESULTS")

display_cols = [
    "scenario",
    "strategy",
    "accuracy",
    "delta_vs_clean_mp_pp",
    "detection_rate_percent",
    "correction_rate_percent",
    "residual_corrupted_symbols"
]

print(network_df[display_cols].to_string(index=False))

print("\nSTORAGE SUMMARY")
print(storage_summary.to_string(index=False))

print("\nOUTPUT FILES")
for p in sorted(OUTPUT.glob("*")):
    print(p.name, f"({p.stat().st_size / 1024:.1f} KB)")

print("\nOutput directory:", OUTPUT)
print("\nAll Experiment 9B results saved.")


JFAMP EXPERIMENT 9B
ACTUAL MIXED-PRECISION FAULT INJECTION + ECC RECOVERY
PyTorch: 2.11.0+cu130
Device: cuda
GPU: Tesla T4

Configuration loaded: 21 layers
Canonical checkpoint loaded.
Dataset root: /kaggle/working/JFAMP/data
Validation samples: 5000
Raw split SHA256: b632b03ccc8166092cbf5c9e02a41c754248533ef1f60292c0404e3a8d80d0e7

All 21 layers converted to their selected formats.
Clean mixed-precision reconstruction completed.

Evaluating clean baselines...
FP32 baseline:              72.4800% (3624/5000)
Clean mixed precision:      72.1400% (3607/5000)

FAULT SCENARIO: SINGLE_BIT

Running strategy: NONE
Accuracy: 0.9200% | Delta vs clean MP: -71.2200 pp
Detected events: 0 | Corrected events: 0 | Residual corrupted symbols: 2100

Running strategy: PARITY
Accuracy: 0.9200% | Delta vs clean MP: -71.2200 pp
Detected events: 2100 | Corrected events: 0 | Residual corrupted symbols: 2100

Running strategy: SECDED
Accuracy: 72.1400% | Delta vs clean MP: +0.0000 pp
Detected events: 2100 | C

In [30]:

# =============================================================================
# JFAMP EXPERIMENT 9B — VERIFIED RERUN
# Correct canonical checkpoint + corrected ECC event denominators
#
# IMPORTANT:
# Run in the SAME Kaggle notebook/kernel after the original 9B cell.
# Reuses its tested helper functions and matched fault plan.
# =============================================================================

import json
import time
import hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from torchvision import models

print("=" * 79)
print("JFAMP EXPERIMENT 9B — VERIFIED RERUN")
print("=" * 79)

# ---------------------------------------------------------------------------
# 1. DISCOVER AND VERIFY THE CANONICAL CHECKPOINT
# ---------------------------------------------------------------------------

canonical_candidates = [
    Path("/kaggle/working/JFAMP/retrain_to150/checkpoints/"
         "resnet18_cifar100_best_150.pth"),
    Path("/kaggle/working/JFAMP/retrain_to150/checkpoints/"
         "resnet18_cifar100_best.pth"),
]

# Search the JFAMP workspace if the expected path is unavailable.
if not any(p.exists() for p in canonical_candidates):
    workspace = Path("/kaggle/working/JFAMP")
    if workspace.exists():
        canonical_candidates += list(
            workspace.rglob("resnet18_cifar100_best_150.pth")
        )

canonical_matches = [p for p in canonical_candidates if p.exists()]

if not canonical_matches:
    raise FileNotFoundError(
        "Canonical epoch-150 checkpoint not found. "
        "Expected resnet18_cifar100_best_150.pth under "
        "/kaggle/working/JFAMP/retrain_to150/checkpoints/. "
        "Do not fall back to the original Kaggle input checkpoint."
    )

CHECKPOINT_9B = canonical_matches[0]
print("Selected canonical checkpoint:", CHECKPOINT_9B)

# Load canonical checkpoint.
ckpt_obj = torch.load(
    CHECKPOINT_9B,
    map_location="cpu",
    weights_only=False
)

if isinstance(ckpt_obj, dict):
    for key in ["model_state_dict", "state_dict", "model", "net"]:
        if key in ckpt_obj and isinstance(ckpt_obj[key], dict):
            ckpt_obj = ckpt_obj[key]
            break

canonical_state = {
    k.replace("module.", "", 1) if k.startswith("module.") else k: v
    for k, v in ckpt_obj.items()
    if isinstance(v, torch.Tensor)
}

# Reconstruct canonical FP32 model.
canonical_fp32 = models.resnet18(weights=None)
canonical_fp32.conv1 = nn.Conv2d(
    3, 64, kernel_size=3, stride=1, padding=1, bias=False
)
canonical_fp32.maxpool = nn.Identity()
canonical_fp32.fc = nn.Linear(canonical_fp32.fc.in_features, 100)

canonical_fp32.load_state_dict(canonical_state, strict=True)
canonical_fp32 = canonical_fp32.to(DEVICE).eval()

checkpoint_sha256 = hashlib.sha256(
    CHECKPOINT_9B.read_bytes()
).hexdigest()

print("Checkpoint SHA256:", checkpoint_sha256)

# ---------------------------------------------------------------------------
# 2. VERIFY THE CANONICAL FP32 BASELINE
# ---------------------------------------------------------------------------

canonical_fp32_metrics = evaluate_model(
    canonical_fp32, val_loader
)

print("\nCanonical FP32 baseline:")
print(canonical_fp32_metrics)

EXPECTED_ACCURACY = 77.78
actual_accuracy = canonical_fp32_metrics["accuracy"]

if abs(actual_accuracy - EXPECTED_ACCURACY) > 0.001:
    raise RuntimeError(
        f"STOP: Expected canonical FP32 accuracy "
        f"{EXPECTED_ACCURACY:.2f}%, got {actual_accuracy:.4f}%. "
        "Do not run 9B until checkpoint, validation data and "
        "preprocessing are reconciled."
    )

print("Canonical baseline verification: PASS")

# ---------------------------------------------------------------------------
# 3. RE-ENCODE ALL 21 LAYERS FROM THE CANONICAL CHECKPOINT
# ---------------------------------------------------------------------------

canonical_modules = dict(canonical_fp32.named_modules())

canonical_symbols = {}
canonical_metadata = {}
canonical_widths = {}
canonical_clean_weights = {}

for layer_name, datatype in layer_config.items():

    assert layer_name in canonical_modules

    module = canonical_modules[layer_name]

    symbols, metadata, width = encode_layer_weight(
        module.weight, datatype
    )

    recovered = decode_layer_weight(
        symbols, metadata, datatype
    )

    assert symbols.size == module.weight.numel()
    assert torch.isfinite(recovered).all()

    canonical_symbols[layer_name] = symbols.copy()
    canonical_metadata[layer_name] = metadata
    canonical_widths[layer_name] = width
    canonical_clean_weights[layer_name] = recovered

assert len(canonical_symbols) == 21

print("\nCanonical mixed-precision encoding:")
print("Layers:", len(canonical_symbols))
print("Weights:", sum(x.size for x in canonical_symbols.values()))

# Rebuild clean mixed-precision model.
clean_mp_model = models.resnet18(weights=None)
clean_mp_model.conv1 = nn.Conv2d(
    3, 64, kernel_size=3, stride=1, padding=1, bias=False
)
clean_mp_model.maxpool = nn.Identity()
clean_mp_model.fc = nn.Linear(clean_mp_model.fc.in_features, 100)

clean_mp_model.load_state_dict(canonical_state, strict=True)
clean_mp_model = clean_mp_model.to(DEVICE).eval()

install_weights(clean_mp_model, canonical_clean_weights)

clean_mp_metrics = evaluate_model(clean_mp_model, val_loader)

print("\nCanonical clean mixed-precision baseline:")
print(clean_mp_metrics)

# ---------------------------------------------------------------------------
# 4. VERIFY THAT THE ORIGINAL MATCHED FAULT PLAN IS AVAILABLE
# ---------------------------------------------------------------------------

assert "fault_plan" in globals(), (
    "Original fault_plan is unavailable. Run the original 9B setup "
    "cell first in this notebook session."
)

for layer_name in layer_config:
    assert layer_name in fault_plan
    for scenario in ["single_bit", "double_bit"]:
        assert scenario in fault_plan[layer_name]

print("\nMatched fault plan: available")

# ---------------------------------------------------------------------------
# 5. RUN FAULT INJECTION WITH CORRECTED EVENT DENOMINATORS
# ---------------------------------------------------------------------------

STRATEGIES_9B = ["none", "parity", "secded", "dmr"]
SCENARIOS_9B = ["single_bit", "double_bit"]

verified_fault_records = []
verified_network_results = []
verified_mechanism_results = []

for scenario in SCENARIOS_9B:

    print("\n" + "=" * 79)
    print("SCENARIO:", scenario.upper())
    print("=" * 79)

    for strategy in STRATEGIES_9B:

        print("\nStrategy:", strategy.upper())

        corrupted_weights = {}

        # Counts are explicitly per corrupted SYMBOL/event.
        total_symbols = 0
        total_bit_flips = 0
        detected_symbols = 0
        corrected_symbols = 0
        residual_symbols = 0

        status_counts = {}

        for layer_name, datatype in layer_config.items():

            original_symbols = canonical_symbols[layer_name]
            modified_symbols = original_symbols.copy()

            width = canonical_widths[layer_name]
            plan = fault_plan[layer_name][scenario]

            indices = plan["indices"]
            bit_positions = plan["bit_positions"]

            for trial, symbol_index in enumerate(indices):

                original_value = int(
                    original_symbols[symbol_index]
                )

                positions = [
                    int(x) for x in bit_positions[trial]
                ]

                total_symbols += 1
                total_bit_flips += len(positions)

                status = ""
                detected = False
                corrected = False

                if strategy == "none":

                    decoded = original_value

                    for bit in positions:
                        decoded ^= (1 << bit)

                    status = "unprotected"

                elif strategy == "parity":

                    code = parity_encode(original_value, width)

                    for bit in positions:
                        code ^= (1 << bit)

                    decoded, detected = parity_decode(code, width)

                    status = (
                        "detected_not_corrected"
                        if detected else "undetected"
                    )

                elif strategy == "secded":

                    code = secded_encode(original_value, width)

                    for bit in positions:
                        code_position = (
                            map_payload_bit_to_secded_position(
                                bit, width
                            )
                        )
                        code ^= (1 << (code_position - 1))

                    decoded, secded_status = secded_decode(
                        code, width
                    )

                    detected = secded_status in [
                        "corrected", "detected_double"
                    ]

                    corrected = (
                        secded_status == "corrected"
                        and decoded == original_value
                    )

                    status = secded_status

                elif strategy == "dmr":

                    copy_a = original_value
                    copy_b = original_value

                    for bit in positions:
                        copy_a ^= (1 << bit)

                    detected = copy_a != copy_b

                    # Deliberately no correction policy:
                    # select copy A, which contains the injected fault.
                    decoded = copy_a

                    status = (
                        "detected_not_corrected"
                        if detected else "undetected"
                    )

                else:
                    raise ValueError(strategy)

                modified_symbols[symbol_index] = decoded

                if detected:
                    detected_symbols += 1

                if corrected:
                    corrected_symbols += 1

                if decoded != original_value:
                    residual_symbols += 1

                status_counts[status] = (
                    status_counts.get(status, 0) + 1
                )

                verified_fault_records.append({
                    "scenario": scenario,
                    "strategy": strategy,
                    "layer": layer_name,
                    "datatype": datatype,
                    "weight_index": int(symbol_index),
                    "payload_bit_positions": ",".join(
                        map(str, positions)
                    ),
                    "original_symbol": original_value,
                    "decoded_symbol": int(decoded),
                    "status": status,
                    "detected": bool(detected),
                    "corrected_exactly": bool(corrected),
                    "residual_error": bool(
                        decoded != original_value
                    )
                })

            corrupted_weights[layer_name] = decode_layer_weight(
                modified_symbols,
                canonical_metadata[layer_name],
                datatype
            )

        # Build the network from the canonical checkpoint.
        fault_model = models.resnet18(weights=None)
        fault_model.conv1 = nn.Conv2d(
            3, 64, kernel_size=3, stride=1, padding=1, bias=False
        )
        fault_model.maxpool = nn.Identity()
        fault_model.fc = nn.Linear(
            fault_model.fc.in_features, 100
        )

        fault_model.load_state_dict(canonical_state, strict=True)
        fault_model = fault_model.to(DEVICE).eval()

        install_weights(fault_model, corrupted_weights)

        metrics = evaluate_model(fault_model, val_loader)

        # Correct denominators: corrupted symbols, not flipped bits.
        detection_rate = (
            100.0 * detected_symbols / total_symbols
        )

        correction_rate = (
            100.0 * corrected_symbols / total_symbols
        )

        residual_rate = (
            100.0 * residual_symbols / total_symbols
        )

        result = {
            "scenario": scenario,
            "strategy": strategy,
            "accuracy": metrics["accuracy"],
            "correct_predictions": metrics["correct"],
            "loss": metrics["loss"],
            "delta_vs_clean_mp_pp": (
                metrics["accuracy"] - clean_mp_metrics["accuracy"]
            ),
            "delta_vs_fp32_pp": (
                metrics["accuracy"] -
                canonical_fp32_metrics["accuracy"]
            ),
            "total_corrupted_symbols": total_symbols,
            "total_bit_flips": total_bit_flips,
            "detected_symbols": detected_symbols,
            "corrected_symbols": corrected_symbols,
            "residual_corrupted_symbols": residual_symbols,
            "detection_rate_percent": detection_rate,
            "correction_rate_percent": correction_rate,
            "residual_symbol_error_rate_percent": residual_rate
        }

        verified_network_results.append(result)

        verified_mechanism_results.append({
            "scenario": scenario,
            "strategy": strategy,
            "total_corrupted_symbols": total_symbols,
            "total_bit_flips": total_bit_flips,
            "detected_symbols": detected_symbols,
            "corrected_symbols": corrected_symbols,
            "residual_corrupted_symbols": residual_symbols,
            "detection_rate_percent": detection_rate,
            "correction_rate_percent": correction_rate,
            "residual_symbol_error_rate_percent": residual_rate,
            "status_counts": status_counts
        })

        print(
            f"Accuracy: {metrics['accuracy']:.4f}% | "
            f"Detection: {detection_rate:.2f}% | "
            f"Correction: {correction_rate:.2f}% | "
            f"Residual symbols: {residual_symbols}/{total_symbols}"
        )

        del fault_model

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

# ---------------------------------------------------------------------------
# 6. STORAGE ACCOUNTING
# ---------------------------------------------------------------------------

storage_rows = []

for layer_name, datatype in layer_config.items():

    n = canonical_symbols[layer_name].size
    width = canonical_widths[layer_name]

    r = get_hamming_parity_count(width)
    secded_width = width + r + 1

    strategy_widths = {
        "none": width,
        "parity": width + 1,
        "secded": secded_width,
        "dmr": width * 2
    }

    for strategy, bits_per_weight in strategy_widths.items():

        storage_rows.append({
            "layer": layer_name,
            "datatype": datatype,
            "weights": n,
            "strategy": strategy,
            "bits_per_weight": bits_per_weight,
            "storage_bits": n * bits_per_weight
        })

storage_df = pd.DataFrame(storage_rows)

storage_summary = (
    storage_df.groupby("strategy", as_index=False)["storage_bits"]
    .sum()
)

total_fp32_bits = sum(
    canonical_symbols[name].size * 32
    for name in layer_config
)

storage_summary["storage_MiB"] = (
    storage_summary["storage_bits"] / 8 / (1024 ** 2)
)

storage_summary["reduction_vs_fp32_percent"] = (
    100 * (
        1 - storage_summary["storage_bits"] / total_fp32_bits
    )
)

# ---------------------------------------------------------------------------
# 7. SAVE VERIFIED RESULTS IN A NEW DIRECTORY
# ---------------------------------------------------------------------------

OUT_9B = Path(
    "/kaggle/working/JFAMP/experiments/"
    "experiment_9B_fault_recovery_verified"
)
OUT_9B.mkdir(parents=True, exist_ok=True)

fault_df = pd.DataFrame(verified_fault_records)
network_df = pd.DataFrame(verified_network_results)
mechanism_df = pd.DataFrame(verified_mechanism_results)

fault_df.to_csv(
    OUT_9B / "experiment_9B_verified_fault_records.csv",
    index=False
)

network_df.to_csv(
    OUT_9B / "experiment_9B_verified_network_results.csv",
    index=False
)

mechanism_df.to_csv(
    OUT_9B / "experiment_9B_verified_ecc_mechanism_results.csv",
    index=False
)

storage_df.to_csv(
    OUT_9B / "experiment_9B_verified_storage_by_layer.csv",
    index=False
)

storage_summary.to_csv(
    OUT_9B / "experiment_9B_verified_storage_summary.csv",
    index=False
)

summary = {
    "experiment": "JFAMP Experiment 9B — verified rerun",
    "checkpoint_path": str(CHECKPOINT_9B),
    "checkpoint_sha256": checkpoint_sha256,
    "validation_split_sha256": raw_split_hash,
    "validation_samples": len(val_dataset),
    "layers": len(layer_config),
    "total_weights": int(sum(
        x.size for x in canonical_symbols.values()
    )),
    "faults_per_layer_per_scenario": N_FAULTS_PER_LAYER,
    "fp32_baseline": canonical_fp32_metrics,
    "clean_mixed_precision_baseline": clean_mp_metrics,
    "network_results": verified_network_results,
    "mechanism_results": verified_mechanism_results,
    "storage_summary": storage_summary.to_dict(orient="records"),
    "rate_denominator": (
        "Number of corrupted weight symbols, not number of bit flips."
    ),
    "dmr_policy": (
        "DMR detects copy mismatch but does not correct. "
        "The evaluation selects the corrupted first copy."
    ),
    "fault_scope": (
        "100 matched sampled payload symbols per layer per scenario. "
        "ECC parity metadata and quantization metadata are not faulted."
    )
}

with open(OUT_9B / "experiment_9B_verified_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# ---------------------------------------------------------------------------
# 8. FINAL REPORT
# ---------------------------------------------------------------------------

print("\n" + "=" * 79)
print("VERIFIED EXPERIMENT 9B COMPLETED")
print("=" * 79)

print("\nCHECKPOINT")
print(CHECKPOINT_9B)
print("SHA256:", checkpoint_sha256)

print("\nBASELINES")
print(
    f"FP32: {canonical_fp32_metrics['accuracy']:.4f}% "
    f"({canonical_fp32_metrics['correct']}/5000)"
)
print(
    f"Clean mixed precision: {clean_mp_metrics['accuracy']:.4f}% "
    f"({clean_mp_metrics['correct']}/5000)"
)

print("\nNETWORK RESULTS")
print(network_df[[
    "scenario",
    "strategy",
    "accuracy",
    "delta_vs_clean_mp_pp",
    "detection_rate_percent",
    "correction_rate_percent",
    "residual_symbol_error_rate_percent"
]].to_string(index=False))

print("\nSTORAGE SUMMARY")
print(storage_summary.to_string(index=False))

print("\nSAVED TO:")
print(OUT_9B)

print("\nVerified output files:")
for p in sorted(OUT_9B.glob("*")):
    print(" -", p.name)

print("\nExperiment 9B verified rerun finished.")


JFAMP EXPERIMENT 9B — VERIFIED RERUN
Selected canonical checkpoint: /kaggle/working/JFAMP/retrain_to150/checkpoints/resnet18_cifar100_best_150.pth
Checkpoint SHA256: b133efde858e1c99ac82909ad17d2b519da59d04c5e78c944576a3792c709f5d

Canonical FP32 baseline:
{'accuracy': 77.78, 'correct': 3889, 'total': 5000, 'loss': 0.9143250726699829}
Canonical baseline verification: PASS

Canonical mixed-precision encoding:
Layers: 21
Weights: 11210432

Canonical clean mixed-precision baseline:
{'accuracy': 77.88, 'correct': 3894, 'total': 5000, 'loss': 0.91851968126297}

Matched fault plan: available

SCENARIO: SINGLE_BIT

Strategy: NONE
Accuracy: 0.9000% | Detection: 0.00% | Correction: 0.00% | Residual symbols: 2100/2100

Strategy: PARITY
Accuracy: 0.9000% | Detection: 100.00% | Correction: 0.00% | Residual symbols: 2100/2100

Strategy: SECDED
Accuracy: 77.8800% | Detection: 100.00% | Correction: 100.00% | Residual symbols: 0/2100

Strategy: DMR
Accuracy: 0.9000% | Detection: 100.00% | Correction: 

In [31]:

# =============================================================================
# JFAMP EXPERIMENT 10
# FINAL CONSOLIDATED EVALUATION, RESEARCH TABLES AND REPRODUCIBILITY REPORT
#
# This cell:
#   1. Discovers and reads completed experiment outputs.
#   2. Verifies canonical Experiment 8 and verified 9B results.
#   3. Consolidates quantization, layer-wise sensitivity and fault results.
#   4. Summarizes the refined optimizer configuration.
#   5. Compares storage/protection trade-offs.
#   6. Creates research-ready plots and tables.
#   7. Generates a final JSON report and Markdown research summary.
#   8. Packages the consolidated deliverables into a ZIP.
#
# No model training, checkpoint modification or fault injection is repeated.
# All previous experiment directories remain unchanged.
# =============================================================================

import os
import json
import hashlib
import zipfile
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

ROOT = Path("/kaggle/working/JFAMP")
EXP = ROOT / "experiments"

OUT = EXP / "experiment_10_final_evaluation"
PLOTS = OUT / "plots"
TABLES = OUT / "tables"

OUT.mkdir(parents=True, exist_ok=True)
PLOTS.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

print("=" * 82)
print("JFAMP EXPERIMENT 10")
print("FINAL CONSOLIDATED EVALUATION")
print("=" * 82)


# =============================================================================
# 1. GENERAL UTILITIES
# =============================================================================

def load_json(path):
    path = Path(path)
    if not path.exists():
        return None
    with open(path, "r") as f:
        return json.load(f)


def load_csv(path):
    path = Path(path)
    if not path.exists():
        return None
    return pd.read_csv(path)


def find_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def find_col(df, candidates):
    if df is None:
        return None
    lower_map = {str(c).lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]
    return None


def numeric_value(obj, keys):
    """Find the first numeric value among candidate dictionary keys."""
    if not isinstance(obj, dict):
        return None

    for key in keys:
        if key in obj:
            try:
                value = float(obj[key])
                if np.isfinite(value):
                    return value
            except (TypeError, ValueError):
                pass

    return None


def save_plot(filename):
    path = PLOTS / filename
    plt.tight_layout()
    plt.savefig(path, dpi=220, bbox_inches="tight")
    plt.close()
    return path


# =============================================================================
# 2. LOCATE ALL COMPLETED EXPERIMENT OUTPUTS
# =============================================================================

paths = {
    "e4m11_json": EXP / "e4m11_subnormal/e4m11_subnormal_results.json",
    "precision_csv": EXP / "precision_comparison/precision_comparison.csv",
    "precision_json": EXP / "precision_comparison/precision_comparison.json",
    "layerwise_csv": EXP / "layerwise_quantization/layerwise_quantization.csv",
    "layer_inventory": EXP / "layerwise_quantization/layer_inventory.csv",
    "fault_layer_summary": EXP / "fault_profiling/fault_profiling_layer_summary.csv",
    "fault_mechanism_summary": EXP / "fault_profiling/fault_profiling_mechanism_summary.csv",
    "protection_layer_summary": EXP / "protection_profiling/protection_layer_summary.csv",
    "protection_mechanism_summary": EXP / "protection_profiling/protection_mechanism_summary.csv",
    "original_optimizer": EXP / "joint_optimization/optimization_report.json",
    "refined_optimizer": EXP / "joint_optimization_refined/refined_optimization_report.json",
    "refined_config": EXP / "joint_optimization_refined/refined_layer_configuration.csv",
    "end_to_end_json": EXP / "end_to_end_validation/end_to_end_validation_results.json",
    "end_to_end_config": EXP / "end_to_end_validation/configuration_used.csv",
    "e9a_json": EXP / "experiment_9A_ecc_validation/experiment_9A_summary.json",
    "e9a_layers": EXP / "experiment_9A_ecc_validation/experiment_9A_layer_encoding.csv",
    "e9b_json": EXP / "experiment_9B_fault_recovery_verified/experiment_9B_verified_summary.json",
    "e9b_network": EXP / "experiment_9B_fault_recovery_verified/experiment_9B_verified_network_results.csv",
    "e9b_mechanism": EXP / "experiment_9B_fault_recovery_verified/experiment_9B_verified_ecc_mechanism_results.csv",
    "e9b_storage": EXP / "experiment_9B_fault_recovery_verified/experiment_9B_verified_storage_summary.csv",
}

loaded = {}
missing = []

for name, path in paths.items():
    if path.suffix == ".json":
        loaded[name] = load_json(path)
    else:
        loaded[name] = load_csv(path)

    if loaded[name] is None:
        missing.append(str(path))

print("\nExperiment artifact discovery:")
for name, value in loaded.items():
    status = "FOUND" if value is not None else "NOT FOUND"
    print(f"{name:28s} {status}")

if missing:
    print("\nWARNING: Some optional artifacts were not found.")
    for p in missing:
        print(" -", p)


# =============================================================================
# 3. VERIFY CANONICAL EXPERIMENT 8 AND VERIFIED EXPERIMENT 9B
# =============================================================================

e8 = loaded["end_to_end_json"]
e9b = loaded["e9b_json"]
e9b_network = loaded["e9b_network"]
e9b_storage = loaded["e9b_storage"]

if e8 is None:
    raise FileNotFoundError(
        "Experiment 8 result JSON is required for final evaluation."
    )

if e9b is None or e9b_network is None:
    raise FileNotFoundError(
        "Verified Experiment 9B results are required."
    )


def extract_accuracy_from_dict(data, candidate_keys):
    return numeric_value(data, candidate_keys)


# Experiment 8 schema.
e8_fp32 = None
e8_mp = None

for key in ["fp32", "fp32_baseline", "baseline"]:
    if isinstance(e8.get(key), dict):
        e8_fp32 = e8[key]
        break

for key in [
    "mixed_precision",
    "refined",
    "jfamp",
    "quantized",
    "optimized"
]:
    if isinstance(e8.get(key), dict):
        e8_mp = e8[key]
        break

# Also support flat schemas.
if e8_fp32 is None:
    e8_fp32 = {
        "accuracy": extract_accuracy_from_dict(
            e8, ["fp32_accuracy", "baseline_accuracy"]
        ),
        "loss": extract_accuracy_from_dict(
            e8, ["fp32_loss", "baseline_loss"]
        )
    }

if e8_mp is None:
    e8_mp = {
        "accuracy": extract_accuracy_from_dict(
            e8, [
                "mixed_precision_accuracy",
                "jfamp_accuracy",
                "quantized_accuracy"
            ]
        ),
        "loss": extract_accuracy_from_dict(
            e8, [
                "mixed_precision_loss",
                "jfamp_loss",
                "quantized_loss"
            ]
        )
    }

# Verified 9B nested schema.
e9b_fp32 = e9b.get("fp32_baseline", {})
e9b_mp = e9b.get("clean_mixed_precision_baseline", {})

fp32_acc = numeric_value(e9b_fp32, ["accuracy"])
mp_acc = numeric_value(e9b_mp, ["accuracy"])

if fp32_acc is None or mp_acc is None:
    raise ValueError(
        "Could not extract canonical baseline accuracy from verified 9B."
    )

assert abs(fp32_acc - 77.78) < 0.001, (
    f"Unexpected FP32 baseline: {fp32_acc}"
)

assert abs(mp_acc - 77.88) < 0.001, (
    f"Unexpected mixed-precision baseline: {mp_acc}"
)

# Check Experiment 8 values where their schema exposes them.
e8_fp32_acc = numeric_value(e8_fp32, ["accuracy", "validation_accuracy"])
e8_mp_acc = numeric_value(e8_mp, ["accuracy", "validation_accuracy"])

if e8_fp32_acc is not None:
    assert abs(e8_fp32_acc - fp32_acc) < 0.001, (
        "Experiment 8 and verified 9B FP32 baseline mismatch."
    )

if e8_mp_acc is not None:
    assert abs(e8_mp_acc - mp_acc) < 0.001, (
        "Experiment 8 and verified 9B mixed-precision baseline mismatch."
    )

# Checkpoint provenance.
canonical_checkpoint = Path(
    "/kaggle/working/JFAMP/retrain_to150/checkpoints/"
    "resnet18_cifar100_best_150.pth"
)

canonical_checkpoint_hash = (
    sha256_file(canonical_checkpoint)
    if canonical_checkpoint.exists()
    else e9b.get("checkpoint_sha256")
)

expected_e9b_hash = e9b.get("checkpoint_sha256")

if (
    canonical_checkpoint_hash
    and expected_e9b_hash
    and canonical_checkpoint_hash != expected_e9b_hash
):
    raise RuntimeError("Canonical checkpoint hash mismatch.")

print("\nCanonical verification:")
print("FP32 accuracy:", fp32_acc)
print("Clean mixed-precision accuracy:", mp_acc)
print("Checkpoint SHA256:", canonical_checkpoint_hash)
print("Experiment 8 / 9B baseline consistency: PASS")


# =============================================================================
# 4. CONSOLIDATED EXPERIMENT 1–4 PRECISION TABLE
# =============================================================================

precision_df = loaded["precision_csv"]

if precision_df is not None:
    precision_df.to_csv(
        TABLES / "global_precision_comparison.csv",
        index=False
    )

    print("\nGlobal precision comparison:")
    print(precision_df.head(20).to_string(index=False))

    # Generate precision accuracy plot when matching columns are available.
    fmt_col = find_col(
        precision_df,
        ["format", "datatype", "precision", "data_type"]
    )
    acc_col = find_col(
        precision_df,
        ["accuracy", "test_accuracy", "validation_accuracy"]
    )

    if fmt_col and acc_col:
        temp = precision_df.copy()
        temp[acc_col] = pd.to_numeric(temp[acc_col], errors="coerce")
        temp = temp.dropna(subset=[acc_col])

        plt.figure(figsize=(10, 5))
        plt.bar(temp[fmt_col].astype(str), temp[acc_col])
        plt.axhline(
            fp32_acc,
            linestyle="--",
            label="Canonical FP32 baseline (Experiment 8/9B)"
        )
        plt.ylabel("Accuracy (%)")
        plt.xlabel("Precision format")
        plt.title(
            "Global Precision Comparison — Earlier Profiling Experiment"
        )
        plt.xticks(rotation=35, ha="right")
        plt.legend()
        save_plot("global_precision_comparison.png")


# Layer-wise quantization data.
layerwise_df = loaded["layerwise_csv"]

if layerwise_df is not None:
    layerwise_df.to_csv(
        TABLES / "layerwise_quantization_profile.csv",
        index=False
    )

    fmt_col = find_col(
        layerwise_df,
        ["format", "datatype", "precision", "data_type"]
    )
    delta_col = find_col(
        layerwise_df,
        ["accuracy_delta_pp", "accuracy_delta", "delta_pp", "delta"]
    )

    if fmt_col and delta_col:
        temp = layerwise_df.copy()
        temp[delta_col] = pd.to_numeric(
            temp[delta_col], errors="coerce"
        )

        aggregate = (
            temp.groupby(fmt_col)[delta_col]
            .agg(["mean", "min", "max", "std", "count"])
            .reset_index()
        )

        aggregate.to_csv(
            TABLES / "layerwise_quantization_aggregate.csv",
            index=False
        )

        plt.figure(figsize=(10, 5))
        plt.bar(
            aggregate[fmt_col].astype(str),
            aggregate["mean"]
        )
        plt.axhline(0, linewidth=1)
        plt.ylabel("Mean accuracy delta (percentage points)")
        plt.xlabel("Precision format")
        plt.title("Mean Isolated-Layer Quantization Sensitivity")
        plt.xticks(rotation=35, ha="right")
        save_plot("layerwise_quantization_sensitivity.png")


# =============================================================================
# 5. FAULT AND PROTECTION PROFILING TABLES
# =============================================================================

fault_layer_df = loaded["fault_layer_summary"]
fault_mechanism_df = loaded["fault_mechanism_summary"]
protection_layer_df = loaded["protection_layer_summary"]
protection_mechanism_df = loaded["protection_mechanism_summary"]

for name, df in [
    ("fault_layer_summary.csv", fault_layer_df),
    ("fault_mechanism_summary.csv", fault_mechanism_df),
    ("protection_layer_summary.csv", protection_layer_df),
    ("protection_mechanism_summary.csv", protection_mechanism_df)
]:
    if df is not None:
        df.to_csv(TABLES / name, index=False)

# Save layer-wise fault sensitivity plot if compatible columns exist.
if fault_layer_df is not None:
    layer_col = find_col(
        fault_layer_df, ["layer", "layer_name"]
    )
    impact_col = find_col(
        fault_layer_df,
        [
            "mean_accuracy_degradation_pp",
            "mean_degradation_pp",
            "mean_accuracy_drop_pp",
            "mean_delta_pp"
        ]
    )

    if layer_col and impact_col:
        temp = fault_layer_df.copy()
        temp[impact_col] = pd.to_numeric(
            temp[impact_col], errors="coerce"
        )
        temp = temp.dropna(subset=[impact_col])
        temp = temp.sort_values(impact_col, ascending=False)

        plt.figure(figsize=(12, 6))
        plt.bar(
            temp[layer_col].astype(str),
            temp[impact_col]
        )
        plt.ylabel("Mean accuracy degradation (pp)")
        plt.xlabel("Layer")
        plt.title("Layer-wise Fault Sensitivity")
        plt.xticks(rotation=75, ha="right")
        save_plot("layerwise_fault_sensitivity.png")


# =============================================================================
# 6. REFINED JOINT OPTIMIZATION SUMMARY
# =============================================================================

refined_optimizer = loaded["refined_optimizer"]
refined_config = loaded["refined_config"]

if refined_config is not None:
    refined_config.to_csv(
        TABLES / "final_optimized_layer_configuration.csv",
        index=False
    )

if refined_optimizer is not None:
    with open(TABLES / "refined_optimizer_report.json", "w") as f:
        json.dump(refined_optimizer, f, indent=2)

# Extract high-level optimizer facts without assuming a single schema.
optimizer_summary = {
    "solver_status": None,
    "selected_layers": (
        len(refined_config) if refined_config is not None else None
    ),
    "storage_reduction_percent": None,
    "quantization_loss_proxy_pp": None,
    "residual_fault_risk_proxy": None,
    "objective_value": None
}

if isinstance(refined_optimizer, dict):

    # Flatten top-level numeric fields and known nested result objects.
    search_dicts = [refined_optimizer]

    for value in refined_optimizer.values():
        if isinstance(value, dict):
            search_dicts.append(value)

    for d in search_dicts:
        for key in ["solver_status", "status", "message"]:
            if key in d and optimizer_summary["solver_status"] is None:
                optimizer_summary["solver_status"] = d[key]

        for key in [
            "storage_reduction_percent",
            "storage_reduction_pct"
        ]:
            if key in d:
                optimizer_summary["storage_reduction_percent"] = d[key]

        for key in [
            "quantization_loss_proxy_pp",
            "quantization_loss"
        ]:
            if key in d:
                optimizer_summary["quantization_loss_proxy_pp"] = d[key]

        for key in [
            "residual_fault_risk_proxy",
            "fault_risk"
        ]:
            if key in d:
                optimizer_summary["residual_fault_risk_proxy"] = d[key]

        for key in ["objective_value", "objective"]:
            if key in d:
                optimizer_summary["objective_value"] = d[key]


# =============================================================================
# 7. END-TO-END AND ECC VALIDATION TABLE
# =============================================================================

e8_summary = {
    "fp32_accuracy_percent": fp32_acc,
    "mixed_precision_accuracy_percent": mp_acc,
    "accuracy_delta_pp": mp_acc - fp32_acc,
    "fp32_correct": int(e9b_fp32.get("correct", 3889)),
    "mixed_precision_correct": int(e9b_mp.get("correct", 3894)),
    "validation_samples": int(e9b.get("validation_samples", 5000))
}

e8_summary["fp32_correct"] = int(e8_fp32.get(
    "correct", e8_summary["fp32_correct"]
) or e8_summary["fp32_correct"])

e8_summary["mixed_precision_correct"] = int(e8_mp.get(
    "correct", e8_summary["mixed_precision_correct"]
) or e8_summary["mixed_precision_correct"])

pd.DataFrame([e8_summary]).to_csv(
    TABLES / "canonical_end_to_end_validation.csv",
    index=False
)

# Experiment 9A summary.
e9a = loaded["e9a_json"]

e9a_summary = {}
if isinstance(e9a, dict):
    e9a_summary = e9a

    with open(TABLES / "experiment_9A_summary.json", "w") as f:
        json.dump(e9a_summary, f, indent=2)

# Experiment 9B result table.
e9b_network = e9b_network.copy()

expected_network_cols = [
    "scenario",
    "strategy",
    "accuracy",
    "delta_vs_clean_mp_pp",
    "detection_rate_percent",
    "correction_rate_percent",
    "residual_symbol_error_rate_percent"
]

for col in expected_network_cols:
    if col not in e9b_network.columns:
        raise ValueError(
            f"Verified 9B network results missing column: {col}"
        )

e9b_network.to_csv(
    TABLES / "verified_fault_recovery_comparison.csv",
    index=False
)

# Ensure expected SECDED results.
single_secded = e9b_network[
    (e9b_network["scenario"] == "single_bit")
    & (e9b_network["strategy"] == "secded")
]

double_secded = e9b_network[
    (e9b_network["scenario"] == "double_bit")
    & (e9b_network["strategy"] == "secded")
]

assert len(single_secded) == 1
assert len(double_secded) == 1

assert abs(
    float(single_secded.iloc[0]["detection_rate_percent"]) - 100
) < 1e-9

assert abs(
    float(single_secded.iloc[0]["correction_rate_percent"]) - 100
) < 1e-9

assert abs(
    float(double_secded.iloc[0]["detection_rate_percent"]) - 100
) < 1e-9

assert abs(
    float(double_secded.iloc[0]["correction_rate_percent"])
) < 1e-9

assert int(
    single_secded.iloc[0]["residual_symbol_error_rate_percent"]
) == 0

# Fault-recovery plot.
plot_df = e9b_network.copy()
scenario_order = ["single_bit", "double_bit"]
strategy_order = ["none", "parity", "secded", "dmr"]

plot_df["scenario"] = pd.Categorical(
    plot_df["scenario"],
    categories=scenario_order,
    ordered=True
)

plot_df["strategy"] = pd.Categorical(
    plot_df["strategy"],
    categories=strategy_order,
    ordered=True
)

plot_df = plot_df.sort_values(["scenario", "strategy"])

for scenario in scenario_order:
    subset = plot_df[plot_df["scenario"] == scenario]

    plt.figure(figsize=(8, 5))
    plt.bar(
        subset["strategy"].astype(str),
        subset["accuracy"]
    )
    plt.ylim(0, 100)
    plt.ylabel("Validation accuracy (%)")
    plt.xlabel("Protection strategy")
    plt.title(
        f"End-to-End Accuracy After {scenario.replace('_', ' ')} Faults"
    )
    save_plot(f"fault_recovery_{scenario}.png")


# =============================================================================
# 8. STORAGE / PROTECTION COMPARISON
# =============================================================================

if e9b_storage is None:
    raise FileNotFoundError(
        "Verified Experiment 9B storage summary is missing."
    )

storage_df = e9b_storage.copy()

storage_strategy_col = find_col(
    storage_df, ["strategy", "protection"]
)

storage_mib_col = find_col(
    storage_df, ["storage_MiB", "storage_mib"]
)

reduction_col = find_col(
    storage_df,
    [
        "reduction_vs_fp32_percent",
        "storage_reduction_percent"
    ]
)

if not storage_strategy_col or not storage_mib_col:
    raise ValueError(
        "Could not identify storage strategy and MiB columns."
    )

storage_df.to_csv(
    TABLES / "verified_storage_protection_comparison.csv",
    index=False
)

# Add FP32 reference for plotting.
storage_plot = storage_df[
    [storage_strategy_col, storage_mib_col]
].copy()

storage_plot.columns = ["strategy", "storage_MiB"]

storage_plot = pd.concat([
    pd.DataFrame([{
        "strategy": "FP32 reference",
        "storage_MiB": 42.7644
    }]),
    storage_plot
], ignore_index=True)

plt.figure(figsize=(9, 5))
plt.bar(
    storage_plot["strategy"].astype(str),
    storage_plot["storage_MiB"]
)
plt.ylabel("Storage (MiB)")
plt.xlabel("Representation / protection")
plt.title("Storage Cost of JFAMP Protection Strategies")
plt.xticks(rotation=25, ha="right")
save_plot("storage_protection_tradeoff.png")


# =============================================================================
# 9. CONSOLIDATED EXPERIMENT EVIDENCE MATRIX
# =============================================================================

evidence = [
    {
        "experiment": "1",
        "name": "E4M11 subnormal weight quantization",
        "status": "Completed",
        "evidence": "Clean accuracy preservation; SQNR and quantization statistics",
        "qualification": "Weight-only quantization; dequantized FP32 inference"
    },
    {
        "experiment": "2",
        "name": "Global precision comparison",
        "status": "Completed",
        "evidence": "FP16, BF16, INT8, INT4 and FP8 comparisons",
        "qualification": "Earlier profiling checkpoint; software simulation"
    },
    {
        "experiment": "3",
        "name": "Layer-wise quantization profiling",
        "status": "Completed",
        "evidence": "21 layers across seven precision formats",
        "qualification": "One layer quantized at a time"
    },
    {
        "experiment": "4",
        "name": "Layer-wise quantization sensitivity",
        "status": "Completed",
        "evidence": "Sensitive and tolerant layer identification",
        "qualification": "Isolated-layer accuracy changes are not additive"
    },
    {
        "experiment": "5",
        "name": "Single-fault profiling",
        "status": "Completed",
        "evidence": "Bit-flip and random-corruption profiles",
        "qualification": "Single-weight fault model; finite-trial estimates"
    },
    {
        "experiment": "6",
        "name": "Protection profiling",
        "status": "Completed",
        "evidence": "Parity, SECDED and DMR codeword-level behavior",
        "qualification": "Initial protection profiling used FP32 codewords"
    },
    {
        "experiment": "7 / 7B",
        "name": "Joint optimization and refinement",
        "status": "Completed",
        "evidence": "Layer-wise datatype and protection assignments",
        "qualification": "Optimization uses measured proxies and constraints"
    },
    {
        "experiment": "8",
        "name": "Clean end-to-end validation",
        "status": "Completed",
        "evidence": "77.78% FP32 and 77.88% optimized mixed precision",
        "qualification": "Dequantized FP32 inference; ECC overhead theoretical"
    },
    {
        "experiment": "9A",
        "name": "Actual mixed-precision SECDED encoding",
        "status": "Completed",
        "evidence": "21 layers; exact no-fault recovery",
        "qualification": "Payload and codeword validation"
    },
    {
        "experiment": "9B",
        "name": "Verified end-to-end fault recovery",
        "status": "Completed",
        "evidence": "Single-bit correction and double-bit detection",
        "qualification": "Sampled payload faults; 100 symbols per layer/scenario"
    },
    {
        "experiment": "10",
        "name": "Final consolidated evaluation",
        "status": "Completed",
        "evidence": "Consolidated tables, plots, provenance and report",
        "qualification": "No new hardware measurements"
    }
]

evidence_df = pd.DataFrame(evidence)

evidence_df.to_csv(
    TABLES / "experiment_evidence_matrix.csv",
    index=False
)


# =============================================================================
# 10. CHECKPOINT AND EXPERIMENT PROVENANCE MANIFEST
# =============================================================================

manifest_rows = []

for name, path in paths.items():
    path = Path(path)

    if path.exists() and path.is_file():
        manifest_rows.append({
            "artifact_name": name,
            "path": str(path),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path)
        })

manifest_df = pd.DataFrame(manifest_rows)

manifest_df.to_csv(
    TABLES / "input_artifact_manifest.csv",
    index=False
)

provenance = {
    "canonical_checkpoint": str(canonical_checkpoint),
    "canonical_checkpoint_sha256": canonical_checkpoint_hash,
    "verified_9B_checkpoint_sha256": expected_e9b_hash,
    "canonical_validation_accuracy_fp32": fp32_acc,
    "canonical_validation_accuracy_mixed_precision": mp_acc,
    "validation_samples": 5000,
    "validation_split_sha256": e9b.get("validation_split_sha256"),
    "experiment_2_to_7_checkpoint_note": (
        "Earlier profiling and optimization experiments used the "
        "epoch-139 checkpoint according to the experiment record. "
        "Experiments 8 and verified 9B use the epoch-150 canonical "
        "checkpoint. Their results are not treated as same-checkpoint "
        "comparisons unless verified separately."
    ),
    "inference_note": (
        "Quantized weights are reconstructed/dequantized to FP32 for "
        "network evaluation. This is not native FP8, INT8 or INT4 hardware "
        "execution."
    ),
    "hardware_note": (
        "No FPGA area, clock frequency, throughput, power, or energy "
        "measurements are included in these experiments."
    )
}

with open(OUT / "provenance.json", "w") as f:
    json.dump(provenance, f, indent=2)


# =============================================================================
# 11. FINAL RESEARCH METRICS
# =============================================================================

single_secded_row = single_secded.iloc[0]
double_secded_row = double_secded.iloc[0]

storage_secded = storage_df[
    storage_df[storage_strategy_col].astype(str).str.lower() == "secded"
]

if len(storage_secded) != 1:
    raise ValueError("Expected exactly one SECDED storage summary row.")

secded_storage_mib = float(
    storage_secded.iloc[0][storage_mib_col]
)

if reduction_col:
    secded_storage_reduction = float(
        storage_secded.iloc[0][reduction_col]
    )
else:
    secded_storage_reduction = (
        100 * (1 - secded_storage_mib / 42.7644)
    )

final_metrics = {
    "canonical_fp32_accuracy_percent": fp32_acc,
    "canonical_mixed_precision_accuracy_percent": mp_acc,
    "clean_accuracy_delta_pp": mp_acc - fp32_acc,
    "validation_samples": 5000,
    "number_of_optimized_layers": (
        len(refined_config) if refined_config is not None else 21
    ),
    "total_model_weights": 11210432,
    "mixed_precision_plus_secded_storage_MiB": secded_storage_mib,
    "fp32_reference_storage_MiB": 42.7644,
    "storage_reduction_percent": secded_storage_reduction,
    "single_bit_secded_detection_percent": float(
        single_secded_row["detection_rate_percent"]
    ),
    "single_bit_secded_correction_percent": float(
        single_secded_row["correction_rate_percent"]
    ),
    "single_bit_secded_residual_error_percent": float(
        single_secded_row["residual_symbol_error_rate_percent"]
    ),
    "single_bit_secded_network_accuracy_percent": float(
        single_secded_row["accuracy"]
    ),
    "double_bit_secded_detection_percent": float(
        double_secded_row["detection_rate_percent"]
    ),
    "double_bit_secded_correction_percent": float(
        double_secded_row["correction_rate_percent"]
    ),
    "double_bit_secded_residual_error_percent": float(
        double_secded_row["residual_symbol_error_rate_percent"]
    ),
    "double_bit_secded_network_accuracy_percent": float(
        double_secded_row["accuracy"]
    )
}

with open(OUT / "final_metrics.json", "w") as f:
    json.dump(final_metrics, f, indent=2)


# =============================================================================
# 12. WRITE FINAL RESEARCH SUMMARY
# =============================================================================

report = f"""
# JFAMP — Final Experimental Evaluation

## 1. Research objective

JFAMP investigates joint fault-aware mixed-precision quantization and
selective protection for neural-network weights. The experimental pipeline
profiles quantization sensitivity and fault behavior, selects per-layer
precision/protection configurations, and evaluates the resulting
mixed-precision model with encoded ECC protection.

## 2. Canonical end-to-end results

| Metric | Result |
|---|---:|
| FP32 validation accuracy | {fp32_acc:.2f}% |
| Optimized mixed-precision accuracy | {mp_acc:.2f}% |
| Accuracy difference | {mp_acc - fp32_acc:+.2f} pp |
| Validation samples | 5,000 |
| Optimized layers | {final_metrics['number_of_optimized_layers']} |
| Model weights | 11,210,432 |

The canonical FP32 checkpoint SHA256 is:

`{canonical_checkpoint_hash}`

The FP32 and clean mixed-precision results are consistent between
Experiments 8 and verified 9B.

## 3. Storage and protection

| Metric | Result |
|---|---:|
| FP32 reference storage | 42.7644 MiB |
| Mixed precision + SECDED | {secded_storage_mib:.4f} MiB |
| Storage reduction | {secded_storage_reduction:.2f}% |

The storage estimate includes the theoretical SECDED codeword overhead
for the selected weight representations. Quantization scales, controller
logic, address metadata, alignment and other implementation overheads
are not fully included.

## 4. Verified fault-recovery results

### Single-bit payload faults

| Metric | SECDED result |
|---|---:|
| Detection rate | {final_metrics['single_bit_secded_detection_percent']:.2f}% |
| Correction rate | {final_metrics['single_bit_secded_correction_percent']:.2f}% |
| Residual symbol error rate | {final_metrics['single_bit_secded_residual_error_percent']:.2f}% |
| Network accuracy | {final_metrics['single_bit_secded_network_accuracy_percent']:.2f}% |

### Double-bit payload faults

| Metric | SECDED result |
|---|---:|
| Detection rate | {final_metrics['double_bit_secded_detection_percent']:.2f}% |
| Correction rate | {final_metrics['double_bit_secded_correction_percent']:.2f}% |
| Residual symbol error rate | {final_metrics['double_bit_secded_residual_error_percent']:.2f}% |
| Network accuracy | {final_metrics['double_bit_secded_network_accuracy_percent']:.2f}% |

SECDED corrected the tested single-bit payload errors and restored the
clean mixed-precision network accuracy. Double-bit errors were detected
but not corrected.

## 5. Main findings

1. The profiling experiments demonstrate that quantization sensitivity
   varies by layer and datatype.
2. The fault profiling experiments show that fault impact depends on
   the fault mechanism, bit position and affected layer.
3. The joint optimizer assigns precision and protection on a per-layer
   basis under the specified storage and quality constraints.
4. The canonical end-to-end mixed-precision model preserves clean
   validation accuracy relative to the FP32 reference.
5. Actual mixed-precision payload encoding and SECDED no-fault recovery
   were verified across all 21 selected layers.
6. The verified 9B experiment demonstrates single-bit SECDED correction
   and double-bit error detection under the tested payload fault model.

## 6. Experimental limitations

- Quantized weights are reconstructed/dequantized to FP32 for network
  inference. These experiments do not establish native FP8, INT8 or
  INT4 hardware execution.
- No FPGA area, power, clock frequency, throughput or energy measurements
  are included.
- The verified fault experiment samples 100 weight symbols per layer
  per scenario; it is not exhaustive.
- Fault injection targets payload bits. ECC metadata, quantization scales,
  address logic and ECC controller faults are outside this experiment.
- DMR detects mismatches but does not identify the correct copy without
  an additional recovery mechanism.
- Earlier profiling and optimization experiments used an epoch-139
  checkpoint, while canonical Experiments 8 and verified 9B use epoch 150.
  Results from different checkpoint generations should not be treated as
  direct same-model comparisons.
- Isolated-layer quantization losses and fault-risk proxies are not
  necessarily additive when multiple layer choices are combined.
- The measured validation differences are based on a fixed 5,000-image
  validation set and should not be interpreted as statistically
  established generalization improvements.

## 7. Reproducibility

The output directory contains:
- Final metric JSON
- Input artifact hash manifest
- Checkpoint and validation provenance
- Consolidated CSV tables
- Research plots
- This report

## 8. Conclusion

The completed experiments provide a software-level demonstration of a
joint layer-wise mixed-precision and protection workflow. The optimized
configuration preserves clean validation accuracy while reducing
theoretical weight storage, and the verified SECDED experiment demonstrates
single-bit correction and double-bit detection in the tested encoded
payloads.

Hardware implementation and physical performance evaluation remain
necessary before making claims about FPGA/ASIC efficiency or deployment.
"""

with open(OUT / "JFAMP_final_research_summary.md", "w") as f:
    f.write(report.strip() + "\n")


# =============================================================================
# 13. CREATE EXPERIMENT 10 SUMMARY JSON
# =============================================================================

experiment10_summary = {
    "experiment": "JFAMP Experiment 10 — Final Consolidated Evaluation",
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "status": "completed",
    "canonical_metrics": final_metrics,
    "optimizer_summary": optimizer_summary,
    "provenance": provenance,
    "evidence_matrix": evidence,
    "outputs": {
        "tables": str(TABLES),
        "plots": str(PLOTS),
        "research_summary": str(OUT / "JFAMP_final_research_summary.md")
    }
}

with open(OUT / "experiment_10_final_summary.json", "w") as f:
    json.dump(experiment10_summary, f, indent=2)


# =============================================================================
# 14. CREATE ZIP PACKAGE
# =============================================================================

ZIP_PATH = ROOT / "JFAMP_experiment_10_final_evaluation.zip"

with zipfile.ZipFile(
    ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED
) as zf:

    for path in OUT.rglob("*"):
        if path.is_file():
            zf.write(
                path,
                arcname=str(Path("experiment_10_final_evaluation") /
                            path.relative_to(OUT))
            )

# =============================================================================
# 15. FINAL CHECKS AND DISPLAY
# =============================================================================

required_outputs = [
    OUT / "experiment_10_final_summary.json",
    OUT / "final_metrics.json",
    OUT / "provenance.json",
    OUT / "JFAMP_final_research_summary.md",
    TABLES / "verified_fault_recovery_comparison.csv",
    TABLES / "verified_storage_protection_comparison.csv",
    TABLES / "experiment_evidence_matrix.csv",
    PLOTS / "fault_recovery_single_bit.png",
    PLOTS / "fault_recovery_double_bit.png",
    PLOTS / "storage_protection_tradeoff.png"
]

failed_outputs = [p for p in required_outputs if not p.exists()]

if failed_outputs:
    raise RuntimeError(
        "Some required Experiment 10 outputs were not created:\n"
        + "\n".join(map(str, failed_outputs))
    )

print("\n" + "=" * 82)
print("EXPERIMENT 10 COMPLETED")
print("=" * 82)

print("\nFINAL CANONICAL METRICS")
for key, value in final_metrics.items():
    print(f"{key:48s}: {value}")

print("\nEXPERIMENT EVIDENCE MATRIX")
print(evidence_df[["experiment", "name", "status"]].to_string(index=False))

print("\nGENERATED PLOTS")
for p in sorted(PLOTS.glob("*.png")):
    print(" -", p.name)

print("\nGENERATED TABLES")
for p in sorted(TABLES.glob("*.csv")):
    print(" -", p.name)

print("\nFINAL REPORT:")
print(OUT / "JFAMP_final_research_summary.md")

print("\nFINAL PACKAGE:")
print(ZIP_PATH)
print(f"ZIP size: {ZIP_PATH.stat().st_size / (1024**2):.2f} MiB")

print("\nAll Experiment 10 checks passed.")


JFAMP EXPERIMENT 10
FINAL CONSOLIDATED EVALUATION

Experiment artifact discovery:
e4m11_json                   FOUND
precision_csv                FOUND
precision_json               FOUND
layerwise_csv                FOUND
layer_inventory              FOUND
fault_layer_summary          FOUND
fault_mechanism_summary      FOUND
protection_layer_summary     FOUND
protection_mechanism_summary FOUND
original_optimizer           FOUND
refined_optimizer            FOUND
refined_config               FOUND
end_to_end_json              FOUND
end_to_end_config            FOUND
e9a_json                     FOUND
e9a_layers                   FOUND
e9b_json                     FOUND
e9b_network                  FOUND
e9b_mechanism                FOUND
e9b_storage                  FOUND

Canonical verification:
FP32 accuracy: 77.78
Clean mixed-precision accuracy: 77.88
Checkpoint SHA256: b133efde858e1c99ac82909ad17d2b519da59d04c5e78c944576a3792c709f5d
Experiment 8 / 9B baseline consistency: PASS

Glob